# Day 19 / Notebook 23 — Retrospective phase + geometry audit, family screen

## Entry context

Day 18B (`527a159`) closed the geometry-corrected matrix audit on 18 charge-first
pairs (Chen2020 / OKane2022 / ORegan2022 × 6 protocols): 0/18 non_trivial,
0/18 mixed_sign, all near_zero in Δt_resid. The 2026-05-06 alignment session
locked the decomposition `raw Δt(Q) = Δt_geom(Q) + Δt_resid(Q)` and identified
the historical PyBaMM DCAC waveform spanning nb04–nb20 as discharge-first —
opposite in phase to the NGU201 ARB experimental convention
(`dc + ac·sin` under PyBaMM `I>0 = discharge`).

Day 19 operates entirely within layer (1) of the five-layer full-protocol
decomposition: prescribed-current CC, AC-on, before V_max trigger. Layers
(2)–(5) are Day 20+.

## Primary finding driving Day 19

Raw Δt(Q) sign in historical DC-vs-DCAC simulations is jointly determined
by phase choice and current-waveform geometry, before any cell-internal
mechanism enters. Discharge-first generates a Δt_geom of opposite sign to
charge-first at low Q; that alone can flip raw Δt(Q) sign without invoking
any non-geometric acceleration.

19A therefore reclassifies historical DC-vs-DCAC evidence
(nb08 / nb09 / nb10 / nb20) under the phase-aware + geometry-corrected
framework. DCAC-vs-DCAC ablation verdicts (nb14 / nb15 / nb16, plus
nb18 / nb19 from Day 14) survive because Δt_geom cancels within same-phase
ablation; only their cross-notebook absolute-Δt comparability is forfeited.

## Scope and 19C deferral

Three execution branches today:

- **19A — Retrospective audit** (work-plan v3 §2): inventory + sign audit
  against raw `.npz`, analytical Δt_geom implementation, DC-vs-DCAC
  residual re-audit on nb08 / nb09 / nb10 / nb20, ablation phase audit
  on nb14–19, verdict aggregation.
- **19B — PyBaMM parameter-family availability** (§3): enumerate PyBaMM
  26.3.1 sets, 4-axis smoke (runnable / DCAC-compatible / native-physics /
  V_min-feasible), select 1× NMC, 1× NCA if available, 1× LFP-or-strong-
  OCP-plateau for Day 20.
- **19D — Day 20 design inputs** (§4): boundary-coupling Q-interval
  inventory, OCP shape briefs, five-layer segmented-audit interface document.

19C (chemistry-shift smoke) is **deferred to Day 20**. Under the
post-alignment framework an all-null CC-only outcome no longer falsifies
the chemistry-shift hypothesis; it only indicates that the prescribed-I CC
region is not the chemistry-shift exit window.

## Locked methodology

```text
Q_net(t)       = -∫₀ᵗ I(τ) dτ / 3600                  (PyBaMM I>0 = discharge)
first_passage  = smallest t with raw strict-net Q(t) ≥ Q_target
                 NO cummax — κ>1 generates local non-monotonicity by design
Δt_model(Q)   = t_DC(Q) − t_DCAC(Q)                   (+ve ⇒ DCAC reaches Q earlier)
Δt_geom(Q)    = same first-passage applied to prescribed I(t), pure numpy
Δt_resid(Q)   = Δt_model − Δt_geom                    (verdict variable)
verdict_bin   = max|Δt_resid| > 60 s   → supported_candidate
                max|Δt_resid| > floor  → near_zero    (floor = max(5·dt_eval, 1.0))
                otherwise               → null
```

Ground truth for sign / first-passage / window verification comes from raw
`data/day18B_smoke/*.npz` trajectories. Derived CSVs (including nb22 Cell 5C
v2 output, whose docstring carries a documented `Δt_model = t_DCAC − t_DC`
sign defect) cannot serve as reference — they may already carry the defects
under audit.

**Sanity anchor**: MJ1 0.3C+0.7C 10τ on shared Q-window [0.163, 2.4251] Ah,
Δt_geom mean ≈ +335.51 s. Δt_geom implementation must reproduce this before
§2.3 mass-apply.

## Notebook flow

1. **Cell 1** — this markdown.
2. **Cell 2** — boilerplate: imports, paths, helper carry-over from nb22 Cell 5C v2.
3. **Cell 3** — §2.1.a curves_long CSV schema scout.
   → `data/day19A_step1_curves_long_inventory.csv`.
4. **Cell 4** — §2.1.b per-notebook phase classification (nb04–nb22), source-truthed from ipynb code cells.
   → `data/day19A_phase_convention_inventory_04_22.csv`.
5. **Cell 5** — §2.1.c nb18 v1/v2 production CSV pair pin-down (records both Day-14 labels and post-Day-18B reframed labels).
   → `data/day19A_nb18_phase_pair_inventory.csv`.
6. **Cell 6** — §2.1.d sign-audit ground truth from raw `.npz` (one Day 18B charge-first anchor pair, Chen2020).
   → `data/day19A_step1_sign_audit.csv`.
7. **Cell 7** — §2.1.e decision-gate aggregator: routes each historical CSV among {§2.3 / §2.4 / §2.5 / regenerate / exclude} based on the four step-1 inventories.
   → `data/day19A_step1_decision_gate.csv`.
8. **Cell 8** — §2.2 analytical Δt_geom implementation + MJ1 0.3C+0.7C 10τ sanity anchor (PASS/FAIL gate before §2.3 mass-apply).
9. **Cells 9–11** — §2.3 DC-vs-DCAC residual audit on nb08 / nb09 / nb10, one cell per notebook for git-blame clarity.
   → `data/day19A_step3_day8to10_dt_resid_audit.csv`.
10. **Cells 12–13** — §2.4 nb20 Q-window compatibility sub-gate, then §2.3 residual audit on `compatible` rows only.
    → `data/day19A_step4_day16_window_audit.csv`,
      `data/day19A_step4_day16_dt_resid_audit.csv`.
11. **Cell 14** — §2.5 nb14 / nb15 / nb16 / nb18 / nb19 ablation phase audit (static inspection, no sim); findings dispatched to §2.6 write-up.
12. **Cell 15** — §2.6 verdict aggregation across §2.3 / §2.4 outputs.
    → `data/day19A_step6_verdict_summary.csv`.
13. **Cells 16–17** — §3 PyBaMM parameter-family enumeration + 4-axis smoke + selection.
    → `data/day19B_parameter_family_availability.csv`,
      `data/day19B_parameter_family_selection.csv`.
14. **Cell 18** — §4.1 boundary-coupling Q-interval inventory across all audited pairs.
    → `data/day19D_boundary_coupling_inventory.csv`.
15. **Cell 19** — close dispatch: prose targets for `docs/day19A_retrospective_audit.md`, `docs/day19_close.md`, `docs/day20_ocp_shape_briefs.md`, `docs/day20_segmented_audit_interface.md`.

## Authority hierarchy

1. Committed files (`ROADMAP.md` v0.3.0, `docs/day18B_close.md`,
   `docs/day18AB_handoff_for_day19.md`).
2. `day19_workplan_v3.md` §0 framework anchor + §2 revised scope.
   v3 supersedes v2 in §2 (nb20 enters §2.3 main flow with §2.4 as
   sub-gate); §0, §3, §4, §5 unchanged.
3. MJ1 0.3C+0.7C 10τ numerical anchor (Δt_geom ≈ +335.51 s, Δt_resid ≈ 0).
4. This notebook markdown — execution log only; not authoritative.

In [1]:
# Cell 2 — boilerplate: imports, paths, helper carry-over from nb22 Cell 5C v2

import json
import re
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.integrate import cumulative_trapezoid


def find_repo_root(start: Path = Path(".")) -> Path:
    """
    Locate repo root robustly from either repo root or notebooks/.
    Expected markers: data/ and notebooks/.
    """
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError(
        f"Could not locate repo root from {start}. "
        "Expected a directory containing both data/ and notebooks/."
    )


REPO = find_repo_root(Path("."))
DATA = REPO / "data"
NB_DIR = REPO / "notebooks"
SMOKE = DATA / "day18B_smoke"

assert DATA.exists(), f"data/ not found at {DATA}"
assert NB_DIR.exists(), f"notebooks/ not found at {NB_DIR}"
assert SMOKE.exists(), f"day18B_smoke/ not found at {SMOKE}"

npz_files = sorted(SMOKE.glob("*.npz"))
assert len(npz_files) > 0, f"No .npz files found in {SMOKE}"

print(f"[Day 19A audit bootstrap] {datetime.now().isoformat(timespec='seconds')}")
print(f"REPO  = {REPO}")
print(f"DATA  = {DATA}")
print(f"SMOKE = {SMOKE}")
print(f"pandas {pd.__version__}, numpy {np.__version__}")
print(f"NPZ files in day18B_smoke/: {len(npz_files)}")
print("First 5 NPZ files:")
for f in npz_files[:5]:
    print(f"  - {f.name}")

[Day 19A audit bootstrap] 2026-05-06T11:44:32
REPO  = /Users/louislu/pybamm-dcac-superimposed
DATA  = /Users/louislu/pybamm-dcac-superimposed/data
SMOKE = /Users/louislu/pybamm-dcac-superimposed/data/day18B_smoke
pandas 3.0.2, numpy 2.4.4
NPZ files in day18B_smoke/: 24
First 5 NPZ files:
  - Chen2020__DCAC_DC0p2C_AC0p4C_0p1tau.npz
  - Chen2020__DCAC_DC0p2C_AC0p4C_10tau.npz
  - Chen2020__DCAC_DC0p2C_AC0p4C_1tau.npz
  - Chen2020__DCAC_DC0p4C_AC0p6C_0p1tau.npz
  - Chen2020__DCAC_DC0p4C_AC0p6C_10tau.npz


In [12]:
# Cell 3 (v3) — 19A.1.a: inventory of historical result / curve CSVs
# Outputs:
#   data/day19A_step1_result_curve_inventory.csv
#
# v3 vs v2:
#   - SIGNED_DT      += dtQ_s / dtQ_min                 (Day 16 nb20, Day 18 v3/v4)
#   - GEOM_COLS      += dtQ_geom_s / dtQ_geom           (Day 18 v4 charge-first)
#   - RESID_COLS     += dtQ_resid_s / dtQ_resid         (Day 18 v4 charge-first)
#   - TIME_PAIR_COLS += (t_DC_s, t_protocol_s),
#                       (t_DC_s, t_DCAC_s)              (Day 16 nb20, Day 18 v3/v4)
#   - Column matching switched to case-insensitive lookup. nb20 uses uppercase
#     `t_DC_s`; v2 lowercase patterns produced false negatives that mis-classified
#     three real DC-vs-DCAC files as `summary_or_auxiliary_only`.
#
# Purpose:
#   Broad inventory of historical result / curve CSVs.
#   Includes:
#     - dtQ curve-like files
#     - legacy Day 8/9 summary CSVs
#     - ablation / metric summary CSVs
#   Excludes Day 19 audit-generated files to avoid recursive contamination.

import json
from pathlib import Path
from datetime import datetime


def classify_file_family(name: str) -> str:
    n = name.lower()

    if n.startswith(("day19a_", "day19b_", "day19d_")):
        return "audit_generated_exclude"

    if (
        "curves_long" in n
        or "deltatq_curves" in n
        or "delta_tq_curves" in n
        or "dtq_resid_curves" in n
        or "dt_q_curves" in n
    ):
        return "dtQ_curve_like"

    if n.startswith(("results_day8", "results_day9", "results_day10")):
        return "legacy_day8_9_10_result_summary"

    if n.startswith(("results_day11", "results_day12", "results_day13", "results_day14")):
        return "ablation_or_metric_summary"

    if "hppc" in n or "relaxation" in n:
        return "hppc_relaxation"

    return "other_result_csv"


patterns = [
    "day*_curves_long*.csv*",
    "*_dt_Q_curves_long*.csv*",
    "*curves*long*.csv*",
    "*delta_tQ_curves*.csv*",
    "*dtQ*curves*.csv*",
    "results_day*.csv*",  # catches results_day8_*, results_day9_* legacy families
]

candidate_list = []
for p in patterns:
    candidate_list.extend(DATA.glob(p))

# de-duplicate by resolved path, sort, and exclude audit-generated files
seen = set()
candidates = []
for f in sorted(candidate_list):
    key = f.resolve()
    if key in seen:
        continue
    seen.add(key)

    if f.name.startswith(("day19A_", "day19B_", "day19D_")):
        continue

    candidates.append(f)

print(f"Found {len(candidates)} historical result / curve CSV candidates\n")


# ---- Column-name patterns (canonical case; matched case-insensitively) -------

SIGNED_DT = [
    "dt_min", "dt_s",
    "delta_t", "delta_t_s", "delta_t_min",
    "delta_t_q_s", "delta_t_q_min",
    "dt_model", "dt_model_s", "dt_model_min",
    "dtQ_s", "dtQ_min",                      # NEW v3: Day 16 nb20, Day 18 v3/v4
]

TIME_PAIR_COLS = [
    ("t_dc_q_s", "t_protocol_q_s"),
    ("t_dc_q_min", "t_protocol_q_min"),
    ("t_dc_s", "t_dcac_s"),
    ("t_dc_min", "t_dcac_min"),
    ("t_ref_s", "t_protocol_s"),
    ("t_ref_min", "t_protocol_min"),
    ("t_DC_s", "t_protocol_s"),              # NEW v3: Day 16 nb20
    ("t_DC_s", "t_DCAC_s"),                  # NEW v3: Day 18 v3/v4
]

GEOM_COLS = [
    "dt_geom_s", "dt_geom",
    "delta_t_geom_s", "delta_t_geom",
    "dtQ_geom_s", "dtQ_geom",                # NEW v3: Day 18 v4 charge-first
]

RESID_COLS = [
    "dt_resid_s", "dt_resid",
    "delta_t_resid_s", "delta_t_resid",
    "dtQ_resid_s", "dtQ_resid",              # NEW v3: Day 18 v4 charge-first
]

Q_COLS = [
    "Q", "Q_Ah", "q_Ah", "Q_mAh",
    "Q_target", "Q_target_Ah", "Q_grid_Ah",
    "Q_percent", "Q_label",
]

CURRENT_COLS = [
    "I", "I_A", "I_py", "current_A", "Current [A]", "I_KL", "I1[A]",
]

VOLTAGE_COLS = [
    "V", "V_V", "voltage_V", "Terminal voltage [V]",
    "U_KL[V]", "U1[V]",
]

PROTOCOL_HINTS = [
    "protocol", "condition", "case", "pair", "ablation", "param_set",
    "parameter_set", "model", "dc_group", "dc_c", "ac_c", "tau", "freq", "f_hz",
]


# ---- Case-insensitive lookup helpers -----------------------------------------

def _build_lower_lookup(cols):
    """Map {lowercase_col_name: canonical_col_name_from_csv}.

    If two CSV columns differ only in case, the later one wins. In practice
    this collision does not occur in this project, but logged here as known
    behaviour.
    """
    return {c.lower(): c for c in cols}


def _hit_cols(lower_lookup, candidates):
    """Return CSV-canonical names matching any candidate (case-insensitive equality)."""
    return [lower_lookup[c.lower()] for c in candidates if c.lower() in lower_lookup]


def _hit_pair_cols(lower_lookup, pair_candidates):
    """Return list of 'A - B' strings for matched (a, b) pairs (case-insensitive)."""
    return [
        f"{lower_lookup[a.lower()]} - {lower_lookup[b.lower()]}"
        for a, b in pair_candidates
        if a.lower() in lower_lookup and b.lower() in lower_lookup
    ]


# ---- Main scan ---------------------------------------------------------------

inv = []

for f in candidates:
    base = {
        "file": f.name,
        "path": str(f),
        "file_family": classify_file_family(f.name),
        "size_kb": round(f.stat().st_size / 1024, 1),
        "mtime": datetime.fromtimestamp(f.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
    }

    try:
        head = pd.read_csv(f, nrows=5, compression="infer")
        cols = list(head.columns)
        lower_lookup = _build_lower_lookup(cols)

        time_pairs_present = _hit_pair_cols(lower_lookup, TIME_PAIR_COLS)
        signed_dt_cols     = _hit_cols(lower_lookup, SIGNED_DT)
        dt_geom_cols       = _hit_cols(lower_lookup, GEOM_COLS)
        dt_resid_cols      = _hit_cols(lower_lookup, RESID_COLS)
        q_cols             = _hit_cols(lower_lookup, Q_COLS)
        current_cols       = _hit_cols(lower_lookup, CURRENT_COLS)
        voltage_cols       = _hit_cols(lower_lookup, VOLTAGE_COLS)
        protocol_cols      = [
            c for c in cols
            if any(h in c.lower() for h in PROTOCOL_HINTS)
        ]

        # coarse audit-readiness label
        if len(time_pairs_present) > 0:
            audit_readiness = "sign_auditable_from_time_pairs"
        elif len(current_cols) > 0 and len(q_cols) > 0:
            audit_readiness = "possibly_recomputable_from_csv"
        elif base["file_family"] == "dtQ_curve_like" and len(signed_dt_cols) > 0:
            audit_readiness = "curve_dt_present_but_sign_needs_external_ground_truth"
        elif base["file_family"] == "legacy_day8_9_10_result_summary":
            audit_readiness = "legacy_summary_needs_json_schema"
        else:
            audit_readiness = "summary_or_auxiliary_only"

        inv.append({
            **base,
            "n_cols": len(cols),
            "cols_str": "|".join(cols),
            "cols_json": json.dumps(cols, ensure_ascii=False),
            "has_dt_signed": len(signed_dt_cols) > 0,
            "signed_dt_cols": json.dumps(signed_dt_cols, ensure_ascii=False),
            "has_time_pair": len(time_pairs_present) > 0,
            "time_pair_cols": json.dumps(time_pairs_present, ensure_ascii=False),
            "has_dt_geom": len(dt_geom_cols) > 0,
            "dt_geom_cols": json.dumps(dt_geom_cols, ensure_ascii=False),
            "has_dt_resid": len(dt_resid_cols) > 0,
            "dt_resid_cols": json.dumps(dt_resid_cols, ensure_ascii=False),
            "has_Q": len(q_cols) > 0,
            "Q_cols": json.dumps(q_cols, ensure_ascii=False),
            "has_current": len(current_cols) > 0,
            "current_cols": json.dumps(current_cols, ensure_ascii=False),
            "has_voltage": len(voltage_cols) > 0,
            "voltage_cols": json.dumps(voltage_cols, ensure_ascii=False),
            "protocol_cols": json.dumps(protocol_cols, ensure_ascii=False),
            "audit_readiness": audit_readiness,
            "read_ok": True,
            "error": None,
        })

    except Exception as e:
        inv.append({
            **base,
            "n_cols": 0,
            "cols_str": "",
            "cols_json": "[]",
            "has_dt_signed": False,
            "signed_dt_cols": "[]",
            "has_time_pair": False,
            "time_pair_cols": "[]",
            "has_dt_geom": False,
            "dt_geom_cols": "[]",
            "has_dt_resid": False,
            "dt_resid_cols": "[]",
            "has_Q": False,
            "Q_cols": "[]",
            "has_current": False,
            "current_cols": "[]",
            "has_voltage": False,
            "voltage_cols": "[]",
            "protocol_cols": "[]",
            "audit_readiness": "read_failed",
            "read_ok": False,
            "error": str(e)[:200],
        })

inv_df = pd.DataFrame(inv).sort_values(["file_family", "file"]).reset_index(drop=True)

out = DATA / "day19A_step1_result_curve_inventory.csv"
inv_df.to_csv(out, index=False)

display_cols = [
    "file",
    "file_family",
    "mtime",
    "has_dt_signed",
    "has_time_pair",
    "has_dt_geom",
    "has_dt_resid",
    "has_Q",
    "has_current",
    "has_voltage",
    "audit_readiness",
]

print(inv_df[display_cols].to_string(index=False))
print(f"\nWrote: {out}")

print("\nFamily counts:")
print(inv_df["file_family"].value_counts().to_string())

print("\nAudit-readiness counts:")
print(inv_df["audit_readiness"].value_counts().to_string())

Found 45 historical result / curve CSV candidates

                                               file                     file_family            mtime  has_dt_signed  has_time_pair  has_dt_geom  has_dt_resid  has_Q  has_current  has_voltage                                       audit_readiness
                    results_day11_curve_metrics.csv      ablation_or_metric_summary 2026-04-29 15:35          False          False        False         False  False        False        False                             summary_or_auxiliary_only
            results_day11_metric_stability_scan.csv      ablation_or_metric_summary 2026-04-29 13:06          False          False        False         False  False        False        False                             summary_or_auxiliary_only
                   results_day11_plating_dt_Q80.csv      ablation_or_metric_summary 2026-04-29 12:48          False          False        False         False  False        False        False                        

In [3]:
# Cell 3.5 — diagnostic before continuing:
#   (a) Are v2-production and v2_aligned_phase identical content?
#   (b) Where are Day 8/9/10 outputs (if anywhere)?
# Outputs:
#   data/day19A_nb18_v2_alias_check.csv
#   data/day19A_step1_diagnostic_file_presence.csv
#   data/day19A_step1_deleted_outputs_git_probe.txt

import hashlib
import subprocess

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

# --- (a) v2 production vs v2_aligned_phase identity ---
f_prod  = DATA / "day14_step5_delta_tQ_curves.csv"
f_align = DATA / "day14_step5_delta_tQ_curves_v2_aligned_phase.csv"

alias_rows = []

print("=== F1: day14 v2 production vs _v2_aligned_phase ===")

if not f_prod.exists() or not f_align.exists():
    print("⚠ one or both files missing")
    alias_rows.append({
        "file_a": str(f_prod.relative_to(REPO)),
        "file_b": str(f_align.relative_to(REPO)),
        "exists_a": f_prod.exists(),
        "exists_b": f_align.exists(),
        "size_a": f_prod.stat().st_size if f_prod.exists() else None,
        "size_b": f_align.stat().st_size if f_align.exists() else None,
        "sha_a": None,
        "sha_b": None,
        "identical": False,
        "interpretation": "missing_file",
    })
else:
    h_prod  = sha256_file(f_prod)
    h_align = sha256_file(f_align)
    identical = (h_prod == h_align)

    print(f"  production : {f_prod.stat().st_size:>10} bytes  sha={h_prod[:16]}")
    print(f"  aligned    : {f_align.stat().st_size:>10} bytes  sha={h_align[:16]}")
    print(f"  identical  : {identical}")

    interpretation = (
        "same_content_alias_for_discharge_first_phase"
        if identical else
        "different_content_treat_as_separate_records"
    )

    if identical:
        print("  → same content; record both names as aliases for the discharge-first phase")
    else:
        print("  ⚠ files differ — treat them as two separate phase records")

    alias_rows.append({
        "file_a": str(f_prod.relative_to(REPO)),
        "file_b": str(f_align.relative_to(REPO)),
        "exists_a": True,
        "exists_b": True,
        "size_a": f_prod.stat().st_size,
        "size_b": f_align.stat().st_size,
        "sha_a": h_prod,
        "sha_b": h_align,
        "identical": identical,
        "interpretation": interpretation,
    })

alias_df = pd.DataFrame(alias_rows)
alias_out = DATA / "day19A_nb18_v2_alias_check.csv"
alias_df.to_csv(alias_out, index=False)
print(f"Wrote: {alias_out}")


# --- (b) Day 8/9/10 output search across whole repo ---
print("\n=== F2: search for Day 8/9/10 outputs anywhere in repo ===")

patterns = [
    "day8*", "day_8*", "day08*",
    "day9*", "day_9*", "day09*",
    "day10*", "day_10*",
    "x2_*", "x4_*", "x5_*",
    "X2_*", "X4_*", "X5_*",
    "*mj1freq*", "*chenfreq*", "*corrected_protocol*",
    "*MJ1freq*", "*Chenfreq*",
]

hits = set()
for pat in patterns:
    hits.update(REPO.rglob(pat))

filtered_hits = []
for p in sorted(hits):
    if not p.is_file():
        continue

    rel = p.relative_to(REPO)

    if any(part.startswith(".") for part in rel.parts):
        continue
    if ".ipynb_checkpoints" in rel.parts:
        continue
    if "__pycache__" in rel.parts:
        continue

    filtered_hits.append(p)

presence_rows = []
for h in filtered_hits:
    rel = h.relative_to(REPO)
    presence_rows.append({
        "path": str(rel),
        "name": h.name,
        "suffix": h.suffix,
        "size_kb": round(h.stat().st_size / 1024, 1),
        "mtime": datetime.fromtimestamp(h.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
        "is_notebook": h.suffix == ".ipynb",
        "is_data_csv": h.suffix in [".csv", ".gz"],
        "is_npz": h.suffix == ".npz",
    })

presence_df = pd.DataFrame(presence_rows).sort_values("path").reset_index(drop=True)
presence_out = DATA / "day19A_step1_diagnostic_file_presence.csv"
presence_df.to_csv(presence_out, index=False)

print(f"Found {len(presence_df)} candidate files (excluding hidden / checkpoints / __pycache__):")
for _, row in presence_df.head(60).iterrows():
    print(f"  {row['path']:<80} {row['size_kb']:>8.1f} KB")
if len(presence_df) > 60:
    print(f"  ... and {len(presence_df)-60} more")

print(f"Wrote: {presence_out}")


# Bonus: git log to see if Day 8/9/10 outputs were ever committed and later deleted
print("\n=== F2b: git log for any deleted Day 8/9/10 files ===")

git_probe_out = DATA / "day19A_step1_deleted_outputs_git_probe.txt"

try:
    r = subprocess.run(
        [
            "git", "-C", str(REPO),
            "log", "--all", "--diff-filter=D",
            "--name-only",
            "--pretty=format:COMMIT %h %ad %s",
            "--date=short",
            "--", "data/",
        ],
        capture_output=True,
        text=True,
        timeout=20,
    )

    if r.returncode != 0:
        print("git log returned non-zero status")
        print(r.stderr[:1000])

    out = r.stdout
    tags = [
        "day8", "day9", "day10",
        "day_8", "day_9", "day_10",
        "day08", "day09",
        "x2", "x4", "x5",
        "mj1freq", "chenfreq", "corrected_protocol",
    ]

    lines_of_interest = []
    current_commit = ""

    for line in out.splitlines():
        if line.startswith("COMMIT "):
            current_commit = line
        elif any(tag in line.lower() for tag in tags):
            lines_of_interest.append(f"{current_commit}  ::  {line}")

    with open(git_probe_out, "w", encoding="utf-8") as fh:
        fh.write(out)
        fh.write("\n\n--- FILTERED LINES OF INTEREST ---\n")
        for l in lines_of_interest:
            fh.write(l + "\n")

    if lines_of_interest:
        print(f"Found {len(lines_of_interest)} deleted file events:")
        for l in lines_of_interest[:40]:
            print(f"  {l}")
        if len(lines_of_interest) > 40:
            print(f"  ... and {len(lines_of_interest)-40} more")
    else:
        print("No deleted Day 8/9/10 / X2/X4/X5 files in git history.")

    print(f"Wrote: {git_probe_out}")

except Exception as e:
    print(f"git log probe failed: {e}")

=== F1: day14 v2 production vs _v2_aligned_phase ===
  production :    1124399 bytes  sha=338f42e2b8f30644
  aligned    :    1124399 bytes  sha=338f42e2b8f30644
  identical  : True
  → same content; record both names as aliases for the discharge-first phase
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_nb18_v2_alias_check.csv

=== F2: search for Day 8/9/10 outputs anywhere in repo ===
Found 34 candidate files (excluding hidden / checkpoints / __pycache__):
  data/results_day8_x2_dfn_mj1freq.csv                                                  7.7 KB
  data/results_day8_x2_dfn_mj1freq_v2.csv                                               8.0 KB
  data/results_day8_x2_dfn_mj1freq_v2.json                                          25904.2 KB
  data/results_day8_x4_dfn_chenfreq.csv                                                17.3 KB
  data/results_day8_x4_dfn_chenfreq.json                                            26556.0 KB
  data/results_day8_x5A_dfn_corrected_mj1freq.csv  

In [6]:
# Cell 3.6 — JSON schema inspector for results_day*.json
# Determine whether the 16–75 MB JSON files alongside results_day*.csv are
# trajectory dumps (candidate ground truth for §2.3) or merely summary blobs.
# Probe schema on the smallest available file to bound load time.

import json as json_module

json_files = sorted(DATA.glob("results_day*.json"), key=lambda p: p.stat().st_size)
print(f"Found {len(json_files)} results_day*.json files")
assert len(json_files) > 0, f"No results_day*.json files found in {DATA}"

print(f"Size range: {json_files[0].stat().st_size/1024**2:.1f} MB "
      f"– {json_files[-1].stat().st_size/1024**2:.1f} MB")
print()

probe_path = json_files[0]
print(f"Probing smallest: {probe_path.name} "
      f"({probe_path.stat().st_size/1024**2:.1f} MB)\n")

with open(probe_path, encoding="utf-8") as fh:
    sample = json_module.load(fh)

print(f"Top-level type: {type(sample).__name__}")
if isinstance(sample, dict):
    print(f"Top-level keys ({len(sample)}):")
    for k, v in sample.items():
        if isinstance(v, list):
            elem_t = type(v[0]).__name__ if v else "empty"
            elem_len = len(v[0]) if v and hasattr(v[0], "__len__") else None
            print(
                f"  {k:<35} list  len={len(v):<8}  elem={elem_t}"
                + (f"  inner_len={elem_len}" if elem_len is not None else "")
            )
        elif isinstance(v, dict):
            sub_keys = list(v.keys())[:6]
            print(f"  {k:<35} dict  n_keys={len(v):<5}  sample_keys={sub_keys}")
        else:
            val_repr = repr(v)[:60]
            print(f"  {k:<35} {type(v).__name__:<8}  value={val_repr}")
elif isinstance(sample, list):
    print(f"List of {len(sample)} elements")
    if sample and isinstance(sample[0], dict):
        print(f"  first elem keys: {list(sample[0].keys())[:12]}")

# Heuristic: detect trajectory-like keys
print("\nTrajectory-key heuristic probe:")
flat_keys = set()

def walk(o, depth=0, max_depth=3):
    if depth > max_depth:
        return
    if isinstance(o, dict):
        for k, v in o.items():
            flat_keys.add(str(k))
            walk(v, depth + 1, max_depth)
    elif isinstance(o, list) and o and isinstance(o[0], dict):
        walk(o[0], depth + 1, max_depth)

walk(sample)

probe = {
    "time-like": [
        k for k in flat_keys
        if any(p in k.lower() for p in ["time", "t_s", "t_min", "timestamp"])
    ],
    "current-like": [
        k for k in flat_keys
        if any(p in k.lower() for p in ["current", "i_a", "i_py", "i_kl", "i1"])
    ],
    "voltage-like": [
        k for k in flat_keys
        if any(p in k.lower() for p in ["voltage", "v_v", "u_kl", "terminal"])
    ],
    "Q-like": [
        k for k in flat_keys
        if any(p in k.lower() for p in [
            "q_net", "q_ah", "q_mah", "q_target", "charge", "capacity"
        ])
    ],
    "protocol-meta": [
        k for k in flat_keys
        if any(p in k.lower() for p in [
            "protocol", "case", "condition", "dc_c", "ac_c", "f_hz"
        ])
    ],
}

for label, keys in probe.items():
    if keys:
        print(f"  {label:<15} {sorted(keys)[:10]}")
    else:
        print(f"  {label:<15} (none found at depth ≤ 3)")

print(f"\nTotal unique keys at depth ≤ 3: {len(flat_keys)}")

Found 13 results_day*.json files
Size range: 9.6 MB – 73.4 MB

Probing smallest: results_day6_batch_scan.json (9.6 MB)

Top-level type: list
List of 30 elements
  first elem keys: ['case_id', 'I_DC_Crate', 'A_Crate', 'f_Hz', 'kappa', 'wall_clock_s', 'CC_time_s', 'CV_time_s', 'total_time_s', 'CC_time_min', 'CV_time_min', 'total_time_min']

Trajectory-key heuristic probe:
  time-like       ['CC_time_min', 'CC_time_s', 'CV_time_min', 'CV_time_s', 'exp_total_time_min', 'total_time_min', 'total_time_s']
  current-like    (none found at depth ≤ 3)
  voltage-like    (none found at depth ≤ 3)
  Q-like          ['Q_net_final_mAh', 'Q_net_trajectory']
  protocol-meta   ['I_DC_Crate', 'case_id', 'condition_label', 'f_Hz']

Total unique keys at depth ≤ 3: 25


In [7]:
# Cell 3.6b — targeted schema probe for Day 8/9 legacy JSON trajectory payloads
# Purpose:
#   Determine whether Day 8/9 JSON files contain enough trajectory information
#   to reconstruct raw strict-net first-passage t(Q).
#
# Outputs:
#   data/day19A_step1_legacy_json_targeted_schema.csv

import json as json_module

target_jsons = [
    DATA / "results_day8_x2_dfn_mj1freq_v2.json",
    DATA / "results_day8_x4_dfn_chenfreq.json",
    DATA / "results_day8_x5A_dfn_corrected_mj1freq.json",
    DATA / "results_day8_x5BC_dfn_corrected_chenfreq.json",
    DATA / "results_day9_x5beta_v2_truephase0_mj1freq.json",
    DATA / "results_day9_x6alpha_composite_sigmoid_mj1freq.json",
    DATA / "results_day9_x6beta_v2_truephase0_mj1freq.json",
]

target_jsons = [p for p in target_jsons if p.exists()]
assert target_jsons, "No target Day 8/9 JSON files found."

def summarize_value(v, max_items=5):
    if isinstance(v, list):
        elem_type = type(v[0]).__name__ if v else "empty"
        out = {
            "type": "list",
            "len": len(v),
            "elem_type": elem_type,
        }
        if v and isinstance(v[0], (int, float, str, bool, type(None))):
            out["preview"] = v[:max_items]
        elif v and isinstance(v[0], list):
            out["inner_len_first"] = len(v[0])
            out["inner_preview_first"] = v[0][:max_items]
        elif v and isinstance(v[0], dict):
            out["first_keys"] = list(v[0].keys())[:20]
        return out

    if isinstance(v, dict):
        return {
            "type": "dict",
            "n_keys": len(v),
            "keys_preview": list(v.keys())[:20],
        }

    return {
        "type": type(v).__name__,
        "preview": repr(v)[:120],
    }

schema_rows = []

for p in target_jsons:
    print("\n" + "=" * 120)
    print(f"[JSON] {p.relative_to(REPO)}")
    print(f"size = {p.stat().st_size / 1024**2:.2f} MB")

    with open(p, encoding="utf-8") as fh:
        obj = json_module.load(fh)

    top_type = type(obj).__name__
    print(f"top-level type = {top_type}")

    # Normalize to records
    if isinstance(obj, list):
        records = obj
        print(f"records = list len {len(records)}")
    elif isinstance(obj, dict):
        # If dict values are records, inspect values; otherwise wrap dict
        if all(isinstance(v, dict) for v in obj.values()):
            records = list(obj.values())
            print(f"records = dict values len {len(records)}")
        else:
            records = [obj]
            print("records = single dict")
    else:
        records = []

    if not records:
        schema_rows.append({
            "file": str(p.relative_to(REPO)),
            "top_type": top_type,
            "n_records": 0,
            "record_keys": "[]",
            "has_Q_net_trajectory": False,
            "has_time_trajectory": False,
            "has_current_trajectory": False,
            "has_voltage_trajectory": False,
            "Q_net_trajectory_summary": "{}",
            "time_like_keys": "[]",
            "current_like_keys": "[]",
            "voltage_like_keys": "[]",
            "metadata_like_keys": "[]",
            "audit_use": "not_usable_empty_or_unknown",
        })
        continue

    first = records[0]
    if not isinstance(first, dict):
        print(f"first record type = {type(first).__name__}; cannot inspect keys")
        continue

    keys = list(first.keys())
    print(f"first record keys ({len(keys)}):")
    print(keys[:80])

    lower_map = {k.lower(): k for k in keys}

    time_like = [
        k for k in keys
        if any(s in k.lower() for s in [
            "time", "t_s", "t_min", "timestamp", "trajectory_t", "t_eval"
        ])
    ]
    current_like = [
        k for k in keys
        if any(s in k.lower() for s in [
            "current", "i_a", "i_py", "i_kl", "i1"
        ])
    ]
    voltage_like = [
        k for k in keys
        if any(s in k.lower() for s in [
            "voltage", "terminal", "v_v", "u_kl", "u1"
        ])
    ]
    q_like = [
        k for k in keys
        if any(s in k.lower() for s in [
            "q_net", "q_ah", "q_mah", "q_target", "charge", "capacity"
        ])
    ]
    metadata_like = [
        k for k in keys
        if any(s in k.lower() for s in [
            "case", "condition", "protocol", "dc", "ac", "f_hz", "tau", "kappa",
            "param", "model", "status"
        ])
    ]

    print("\nKey groups:")
    print(f"  time_like    : {time_like}")
    print(f"  current_like : {current_like}")
    print(f"  voltage_like : {voltage_like}")
    print(f"  q_like       : {q_like}")
    print(f"  metadata_like: {metadata_like}")

    # Deep inspect selected trajectory-like fields in first record
    print("\nSelected field summaries from first record:")
    for k in sorted(set(time_like + current_like + voltage_like + q_like + metadata_like)):
        summary = summarize_value(first.get(k))
        print(f"  {k:<35} {summary}")

    qtraj_key = None
    for candidate in ["Q_net_trajectory", "q_net_trajectory", "Q_trajectory", "Q_Ah_trajectory"]:
        if candidate in first:
            qtraj_key = candidate
            break

    q_summary = summarize_value(first[qtraj_key]) if qtraj_key else {}

    has_time_trajectory = any(
        isinstance(first.get(k), list) and len(first.get(k)) > 10
        for k in time_like
    )
    has_current_trajectory = any(
        isinstance(first.get(k), list) and len(first.get(k)) > 10
        for k in current_like
    )
    has_voltage_trajectory = any(
        isinstance(first.get(k), list) and len(first.get(k)) > 10
        for k in voltage_like
    )

    # Basic audit usability decision
    if qtraj_key and has_time_trajectory:
        audit_use = "candidate_tQ_ground_truth"
    elif qtraj_key:
        audit_use = "has_Q_trajectory_but_time_axis_unclear"
    elif has_current_trajectory and has_time_trajectory:
        audit_use = "candidate_recompute_Q_from_current"
    else:
        audit_use = "summary_or_insufficient_for_tQ_ground_truth"

    schema_rows.append({
        "file": str(p.relative_to(REPO)),
        "size_mb": round(p.stat().st_size / 1024**2, 2),
        "top_type": top_type,
        "n_records": len(records),
        "record_keys": json_module.dumps(keys, ensure_ascii=False),
        "has_Q_net_trajectory": qtraj_key is not None,
        "Q_net_trajectory_key": qtraj_key,
        "has_time_trajectory": has_time_trajectory,
        "has_current_trajectory": has_current_trajectory,
        "has_voltage_trajectory": has_voltage_trajectory,
        "Q_net_trajectory_summary": json_module.dumps(q_summary, ensure_ascii=False),
        "time_like_keys": json_module.dumps(time_like, ensure_ascii=False),
        "current_like_keys": json_module.dumps(current_like, ensure_ascii=False),
        "voltage_like_keys": json_module.dumps(voltage_like, ensure_ascii=False),
        "q_like_keys": json_module.dumps(q_like, ensure_ascii=False),
        "metadata_like_keys": json_module.dumps(metadata_like, ensure_ascii=False),
        "audit_use": audit_use,
    })

schema_target_df = pd.DataFrame(schema_rows)
out = DATA / "day19A_step1_legacy_json_targeted_schema.csv"
schema_target_df.to_csv(out, index=False)

print("\n" + "=" * 120)
print(f"Wrote: {out}")
display(schema_target_df[[
    "file",
    "n_records",
    "has_Q_net_trajectory",
    "has_time_trajectory",
    "has_current_trajectory",
    "has_voltage_trajectory",
    "audit_use",
]])


[JSON] data/results_day8_x2_dfn_mj1freq_v2.json
size = 25.30 MB
top-level type = list
records = list len 31
first record keys (22):
['case_id', 'I_DC_Crate', 'A_Crate', 'f_Hz', 'kappa', 'wall_clock_s', 'CC_time_s', 'CV_time_s', 'total_time_s', 'CC_time_min', 'CV_time_min', 'total_time_min', 'Q_net_final_mAh', 'Q_net_trajectory', 't_chg', 'I_chg', 'T_max_charging_C', 'n_AC_reversals_steps', 'pct_AC_reversals', 'status', 'condition', 'tau_label']

Key groups:
  time_like    : ['CC_time_s', 'CV_time_s', 'total_time_s', 'CC_time_min', 'CV_time_min', 'total_time_min']
  current_like : []
  voltage_like : []
  q_like       : ['Q_net_final_mAh', 'Q_net_trajectory']
  metadata_like: ['case_id', 'I_DC_Crate', 'f_Hz', 'kappa', 'n_AC_reversals_steps', 'pct_AC_reversals', 'status', 'condition', 'tau_label']

Selected field summaries from first record:
  CC_time_min                         {'type': 'float', 'preview': '499.72275983672927'}
  CC_time_s                           {'type': 'float', 'p

,file,n_records,has_Q_net_trajectory,has_time_trajectory,has_current_trajectory,has_voltage_trajectory,audit_use
0,data/results_day8_x2_dfn_mj1freq_v2.json,31,True,False,False,False,has_Q_trajectory_but_time_axis_unclear
1,data/results_day8_x4_dfn_chenfreq.json,62,True,False,False,False,has_Q_trajectory_but_time_axis_unclear
2,data/results_day8_x5A_dfn_corrected_mj1freq.json,31,True,False,False,False,has_Q_trajectory_but_time_axis_unclear
3,data/results_day8_x5BC_dfn_corrected_chenfreq....,62,True,False,False,False,has_Q_trajectory_but_time_axis_unclear
4,data/results_day9_x5beta_v2_truephase0_mj1freq...,31,True,False,False,False,has_Q_trajectory_but_time_axis_unclear
5,data/results_day9_x6alpha_composite_sigmoid_mj...,31,True,False,False,False,has_Q_trajectory_but_time_axis_unclear
6,data/results_day9_x6beta_v2_truephase0_mj1freq...,31,False,False,False,False,summary_or_insufficient_for_tQ_ground_truth


In [8]:
# Cell 3.6c — validate legacy JSON trajectories:
#   - t_chg / I_chg / Q_net_trajectory length consistency
#   - Q_net_trajectory consistency with strict-net integration from I_chg
#   - identify usable records for §2.3 residual audit
#
# Outputs:
#   data/day19A_step1_legacy_json_trajectory_validation.csv

from scipy.integrate import cumulative_trapezoid
import json as json_module
import numpy as np
import pandas as pd

target_jsons = [
    DATA / "results_day8_x2_dfn_mj1freq_v2.json",
    DATA / "results_day8_x4_dfn_chenfreq.json",
    DATA / "results_day8_x5A_dfn_corrected_mj1freq.json",
    DATA / "results_day8_x5BC_dfn_corrected_chenfreq.json",
    DATA / "results_day9_x5beta_v2_truephase0_mj1freq.json",
    DATA / "results_day9_x6alpha_composite_sigmoid_mj1freq.json",
    DATA / "results_day9_x6beta_v2_truephase0_mj1freq.json",
]
target_jsons = [p for p in target_jsons if p.exists()]

def as_float_array(x):
    """Robust conversion for list-like trajectory fields."""
    if x is None:
        return None
    arr = np.asarray(x, dtype=float)
    if arr.ndim != 1:
        arr = arr.reshape(-1)
    return arr

def q_from_current_mAh(t_s, I_A, mode):
    """
    Recompute charge in mAh from current.

    PyBaMM sign convention:
        I > 0 = discharge
        I < 0 = charge

    strict-net charging Q:
        Q_mAh = -∫ I(t) dt / 3600 * 1000

    Additional modes are diagnostic only.
    """
    if mode == "strict_net_minus_I":
        return -cumulative_trapezoid(I_A, t_s, initial=0.0) / 3600.0 * 1000.0
    if mode == "plus_I":
        return cumulative_trapezoid(I_A, t_s, initial=0.0) / 3600.0 * 1000.0
    if mode == "abs_I":
        return cumulative_trapezoid(np.abs(I_A), t_s, initial=0.0) / 3600.0 * 1000.0
    raise ValueError(mode)

def max_abs_diff(a, b):
    n = min(len(a), len(b))
    if n == 0:
        return np.nan
    return float(np.nanmax(np.abs(a[:n] - b[:n])))

def rmse(a, b):
    n = min(len(a), len(b))
    if n == 0:
        return np.nan
    d = a[:n] - b[:n]
    return float(np.sqrt(np.nanmean(d * d)))

rows = []

for p in target_jsons:
    print("\n" + "=" * 120)
    print(f"[VALIDATE] {p.relative_to(REPO)}")

    with open(p, encoding="utf-8") as fh:
        records = json_module.load(fh)

    assert isinstance(records, list), f"Expected list records in {p.name}"

    for idx, rec in enumerate(records):
        status = rec.get("status", None)

        t = as_float_array(rec.get("t_chg"))
        I = as_float_array(rec.get("I_chg"))
        Q_stored = as_float_array(rec.get("Q_net_trajectory"))

        has_t = t is not None
        has_I = I is not None
        has_Q = Q_stored is not None

        n_t = len(t) if has_t else 0
        n_I = len(I) if has_I else 0
        n_Q = len(Q_stored) if has_Q else 0

        lengths_equal = (n_t == n_I == n_Q) if (has_t and has_I and has_Q) else False
        t_monotonic = bool(np.all(np.diff(t) > 0)) if has_t and n_t > 1 else False

        # Default diagnostics
        q_match_mode = None
        q_match_max_abs_mAh = np.nan
        q_match_rmse_mAh = np.nan
        final_err_mAh = np.nan
        q_final_stored_mAh = float(Q_stored[-1]) if has_Q and n_Q > 0 else np.nan
        q_final_reported_mAh = rec.get("Q_net_final_mAh", np.nan)

        I_min = float(np.nanmin(I)) if has_I and n_I > 0 else np.nan
        I_max = float(np.nanmax(I)) if has_I and n_I > 0 else np.nan
        has_discharge_intervals = bool(np.nanmax(I) > 0) if has_I and n_I > 0 else False
        has_charge_intervals = bool(np.nanmin(I) < 0) if has_I and n_I > 0 else False

        if has_t and has_I and has_Q and lengths_equal and t_monotonic:
            candidates = {}
            for mode in ["strict_net_minus_I", "plus_I", "abs_I"]:
                q_calc = q_from_current_mAh(t, I, mode)
                candidates[mode] = {
                    "max_abs": max_abs_diff(q_calc, Q_stored),
                    "rmse": rmse(q_calc, Q_stored),
                    "final_err": float(q_calc[-1] - Q_stored[-1]),
                }

            # choose best alignment
            q_match_mode = min(candidates, key=lambda m: candidates[m]["rmse"])
            q_match_max_abs_mAh = candidates[q_match_mode]["max_abs"]
            q_match_rmse_mAh = candidates[q_match_mode]["rmse"]
            final_err_mAh = candidates[q_match_mode]["final_err"]

        # time consistency diagnostics
        t_last_s = float(t[-1]) if has_t and n_t > 0 else np.nan
        total_time_s = rec.get("total_time_s", np.nan)
        cc_time_s = rec.get("CC_time_s", np.nan)
        cv_time_s = rec.get("CV_time_s", np.nan)

        if np.isfinite(total_time_s) and np.isfinite(t_last_s):
            total_time_err_s = float(t_last_s - total_time_s)
        else:
            total_time_err_s = np.nan

        # Audit usability
        if status != "ok":
            audit_use = "exclude_non_ok"
        elif has_t and has_I and lengths_equal and t_monotonic:
            audit_use = "usable_recompute_strict_net_from_t_I"
        elif has_t and has_Q and n_t == n_Q and t_monotonic:
            audit_use = "usable_tQ_only_no_current"
        elif has_Q:
            audit_use = "has_Q_but_time_axis_invalid_or_missing"
        else:
            audit_use = "not_usable_no_trajectory"

        rows.append({
            "file": str(p.relative_to(REPO)),
            "record_idx": idx,
            "case_id": rec.get("case_id"),
            "condition": rec.get("condition"),
            "tau_label": rec.get("tau_label"),
            "I_DC_Crate": rec.get("I_DC_Crate"),
            "A_Crate": rec.get("A_Crate"),
            "f_Hz": rec.get("f_Hz"),
            "kappa": rec.get("kappa"),
            "status": status,
            "has_t_chg": has_t,
            "has_I_chg": has_I,
            "has_Q_net_trajectory": has_Q,
            "n_t": n_t,
            "n_I": n_I,
            "n_Q": n_Q,
            "lengths_equal": lengths_equal,
            "t_monotonic": t_monotonic,
            "t_last_s": t_last_s,
            "total_time_s": total_time_s,
            "total_time_err_s": total_time_err_s,
            "CC_time_s": cc_time_s,
            "CV_time_s": cv_time_s,
            "I_min_A": I_min,
            "I_max_A": I_max,
            "has_charge_intervals_Ineg": has_charge_intervals,
            "has_discharge_intervals_Ipos": has_discharge_intervals,
            "Q_final_stored_mAh": q_final_stored_mAh,
            "Q_net_final_mAh_reported": q_final_reported_mAh,
            "Q_report_vs_stored_final_err_mAh": (
                float(q_final_reported_mAh - q_final_stored_mAh)
                if np.isfinite(q_final_reported_mAh) and np.isfinite(q_final_stored_mAh)
                else np.nan
            ),
            "q_match_mode": q_match_mode,
            "q_match_max_abs_mAh": q_match_max_abs_mAh,
            "q_match_rmse_mAh": q_match_rmse_mAh,
            "q_match_final_err_mAh": final_err_mAh,
            "audit_use": audit_use,
        })

val_df = pd.DataFrame(rows)

out = DATA / "day19A_step1_legacy_json_trajectory_validation.csv"
val_df.to_csv(out, index=False)

print(f"\nWrote: {out}")

summary_cols = [
    "file", "record_idx", "case_id", "status",
    "n_t", "n_I", "n_Q", "lengths_equal", "t_monotonic",
    "I_min_A", "I_max_A", "q_match_mode",
    "q_match_rmse_mAh", "q_match_max_abs_mAh", "audit_use",
]

display(val_df[summary_cols].head(30))

print("\nAudit-use counts:")
print(val_df["audit_use"].value_counts(dropna=False).to_string())

print("\nQ match mode counts for usable records:")
usable = val_df[val_df["audit_use"].eq("usable_recompute_strict_net_from_t_I")]
if len(usable):
    print(usable["q_match_mode"].value_counts(dropna=False).to_string())
    print("\nRMSE statistics by file:")
    display(
        usable.groupby("file")["q_match_rmse_mAh"]
        .agg(["count", "median", "max"])
        .reset_index()
    )
else:
    print("No usable records found.")


[VALIDATE] data/results_day8_x2_dfn_mj1freq_v2.json

[VALIDATE] data/results_day8_x4_dfn_chenfreq.json

[VALIDATE] data/results_day8_x5A_dfn_corrected_mj1freq.json

[VALIDATE] data/results_day8_x5BC_dfn_corrected_chenfreq.json

[VALIDATE] data/results_day9_x5beta_v2_truephase0_mj1freq.json

[VALIDATE] data/results_day9_x6alpha_composite_sigmoid_mj1freq.json

[VALIDATE] data/results_day9_x6beta_v2_truephase0_mj1freq.json

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step1_legacy_json_trajectory_validation.csv


,file,record_idx,case_id,status,n_t,n_I,n_Q,lengths_equal,t_monotonic,I_min_A,I_max_A,q_match_mode,q_match_rmse_mAh,q_match_max_abs_mAh,audit_use
0,data/results_day8_x2_dfn_mj1freq_v2.json,0,DC0.10C+AC0.90C_f0.01430Hz,ok,31332,31332,31332,True,True,-5.000000,4.000000e+00,strict_net_minus_I,0.0,0.0,usable_recompute_strict_net_from_t_I
1,data/results_day8_x2_dfn_mj1freq_v2.json,1,DC0.10C+AC0.20C_f0.01430Hz,ok,19401,19401,19401,True,True,-1.500000,5.000000e-01,strict_net_minus_I,0.0,0.0,usable_recompute_strict_net_from_t_I
2,data/results_day8_x2_dfn_mj1freq_v2.json,2,DC0.20C+AC0.30C_f0.01430Hz,ok,10987,10987,10987,True,True,-2.500000,4.999999e-01,strict_net_minus_I,0.0,0.0,usable_recompute_strict_net_from_t_I
3,data/results_day8_x2_dfn_mj1freq_v2.json,3,DC0.20C+AC0.30C_f0.00143Hz,ok,1246,1246,1246,True,True,-2.500000,4.999929e-01,strict_net_minus_I,0.0,0.0,usable_recompute_strict_net_from_t_I
4,data/results_day8_x2_dfn_mj1freq_v2.json,4,DC0.20C+AC0.30C_f0.00041Hz,ok,444,444,444,True,False,-2.499987,4.999798e-01,NaN,NaN,NaN,has_Q_but_time_axis_invalid_or_missing
5,data/results_day8_x2_dfn_mj1freq_v2.json,5,DC0.20C+AC0.80C_f0.14300Hz,ok,154246,154246,154246,True,True,-5.000000,3.000000e+00,strict_net_minus_I,0.0,0.0,usable_recompute_strict_net_from_t_I
6,data/results_day8_x2_dfn_mj1freq_v2.json,6,DC0.20C+AC0.80C_f0.01430Hz,ok,14509,14509,14509,True,True,-5.000000,3.000000e+00,strict_net_minus_I,0.0,0.0,usable_recompute_strict_net_from_t_I
7,data/results_day8_x2_dfn_mj1freq_v2.json,7,DC0.20C+AC0.80C_f0.00143Hz,INFEASIBLE_MIN_V: V_min=1.500V in Phase 2,0,0,0,False,False,NaN,NaN,NaN,NaN,NaN,exclude_non_ok
8,data/results_day8_x2_dfn_mj1freq_v2.json,8,DC0.30C+AC0.70C_f0.14300Hz,ok,94015,94015,94015,True,True,-5.000000,2.000000e+00,strict_net_minus_I,0.0,0.0,usable_recompute_strict_net_from_t_I
9,data/results_day8_x2_dfn_mj1freq_v2.json,9,DC0.30C+AC0.70C_f0.01430Hz,ok,8916,8916,8916,True,True,-4.999999,2.000000e+00,strict_net_minus_I,0.0,0.0,usable_recompute_strict_net_from_t_I



Audit-use counts:
audit_use
usable_recompute_strict_net_from_t_I      247
exclude_non_ok                             27
has_Q_but_time_axis_invalid_or_missing      5

Q match mode counts for usable records:
q_match_mode
strict_net_minus_I    247

RMSE statistics by file:


,file,count,median,max
0,data/results_day8_x2_dfn_mj1freq_v2.json,27,0.0,0.0
1,data/results_day8_x4_dfn_chenfreq.json,53,0.0,0.0
2,data/results_day8_x5A_dfn_corrected_mj1freq.json,31,0.0,0.0
3,data/results_day8_x5BC_dfn_corrected_chenfreq....,58,0.0,0.0
4,data/results_day9_x5beta_v2_truephase0_mj1freq...,29,0.0,0.0
5,data/results_day9_x6alpha_composite_sigmoid_mj...,28,0.0,0.0
6,data/results_day9_x6beta_v2_truephase0_mj1freq...,21,0.0,0.0


In [11]:
inv_df = pd.read_csv(DATA / "day19A_step1_result_curve_inventory.csv")
suspect = inv_df[inv_df["file"].isin([
    "day16_step3a_dtQ_curves_long.csv.gz",
    "day18_step2_dt_Q_curves_long_v3.csv.gz",
    "day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz",
])]
for _, r in suspect.iterrows():
    cols = json.loads(r["cols_json"])
    dt_like = [c for c in cols if any(s in c.lower() for s in ["dt", "delta_t"])]
    print(f"\n{r['file']}: {len(cols)} cols")
    print(f"  all cols    : {cols}")
    print(f"  dt-like cols: {dt_like}")


day16_step3a_dtQ_curves_long.csv.gz: 14 cols
  all cols    : ['param_set', 'chem_tag', 'condition', 'pair_role', 'DC_C', 'AC_C', 'tau_label', 'Q_Ah', 't_DC_s', 't_protocol_s', 'dtQ_s', 'Q_nom_Ah', 'Q_low_Ah', 'Q_hi_Ah']
  dt-like cols: ['dtQ_s']

day18_step2_dt_Q_curves_long_v3.csv.gz: 12 cols
  all cols    : ['param_set', 'anchor_label', 'Q_Ah', 'Q_frac_nom_added', 't_DC_s', 't_DCAC_s', 'dtQ_s', 'window_full', 'window_stable_added', 'protocol_DC', 'protocol_DCAC', 'status']
  dt-like cols: ['dtQ_s']

day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz: 15 cols
  all cols    : ['param_set', 'anchor_label', 'phase_label', 'Q_Ah', 'Q_frac_nom_added', 't_DC_s', 't_DCAC_s', 'dtQ_s', 'dtQ_geom_s', 'dtQ_resid_s', 'window_full', 'window_stable_added', 'protocol_DC', 'protocol_DCAC', 'status']
  dt-like cols: ['dtQ_s', 'dtQ_geom_s', 'dtQ_resid_s']


In [14]:
# Cell 3.8 — CSV time-pair sign audit + residual identity audit
# Inputs:
#   data/day19A_step1_result_curve_inventory.csv
#
# Outputs:
#   data/day19A_step1_csv_timepair_sign_self_audit.csv
#   data/day19A_step1_csv_residual_identity_audit.csv
#
# Purpose:
#   Part A:
#     Verify whether stored total Δt columns follow the official convention:
#         Δt = t_ref - t_protocol
#     where t_ref is usually t_DC and t_protocol is t_DCAC / t_protocol.
#
#   Part B:
#     Verify residual decomposition identity where available:
#         dtQ_s = dtQ_geom_s + dtQ_resid_s
#
# Important:
#   Do NOT compare dtQ_geom_s or dtQ_resid_s directly to t_DC - t_DCAC.
#   Only total Δt-like columns are sign-audited against time pairs.

import json
import numpy as np
import pandas as pd
from pathlib import Path

inv_path = DATA / "day19A_step1_result_curve_inventory.csv"
assert inv_path.exists(), f"Inventory not found: {inv_path}"

inv_df = pd.read_csv(inv_path)


def load_csv_auto(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, compression="infer")


def lower_lookup(cols):
    """Case-insensitive column lookup: lower-case name -> CSV-native name."""
    return {c.lower(): c for c in cols}


def get_col_case_insensitive(lookup, name):
    return lookup.get(name.lower())


# Only total Δt columns belong here.
# Do NOT include dtQ_geom_s / dtQ_resid_s.
TOTAL_DT_CANDIDATES = [
    "dtQ", "dtQ_s", "dtQ_min",
    "dt_s", "dt_min",
    "delta_t", "delta_t_s", "delta_t_min",
    "delta_t_q_s", "delta_t_q_min",
    "dt_model", "dt_model_s", "dt_model_min",
]

# Candidate time-pair patterns.
# Case-insensitive matching means some lowercase/uppercase variants may map
# to the same CSV-native columns; deduplication is applied below.
TIMEPAIR_CANDIDATES = [
    ("t_DC_s", "t_protocol_s"),      # Day16 nb20
    ("t_DC_s", "t_DCAC_s"),         # Day18 v3/v4
    ("t_DC_min", "t_DCAC_min"),     # Day13/14 style
    ("t_dc_q_s", "t_protocol_q_s"),
    ("t_dc_q_min", "t_protocol_q_min"),
    ("t_dc_s", "t_dcac_s"),
    ("t_dc_min", "t_dcac_min"),
    ("t_ref_s", "t_protocol_s"),
    ("t_ref_min", "t_protocol_min"),
]


# ---------------------------------------------------------------------
# Part A — time-pair sign audit
# ---------------------------------------------------------------------

sign_files = inv_df[inv_df["audit_readiness"].eq("sign_auditable_from_time_pairs")].copy()

print(f"Sign-auditable files from inventory: {len(sign_files)}")
print(
    sign_files[
        ["file", "file_family", "time_pair_cols", "signed_dt_cols"]
    ].to_string(index=False)
)

sign_rows = []

for _, inv_row in sign_files.iterrows():
    f = Path(inv_row["path"])

    if not f.exists():
        sign_rows.append({
            "file": f.name,
            "file_family": inv_row.get("file_family", ""),
            "n_rows": 0,
            "t_ref_col": "",
            "t_protocol_col": "",
            "dt_col": "",
            "n_valid": 0,
            "median_t_ref": np.nan,
            "median_t_protocol": np.nan,
            "median_dt_stored": np.nan,
            "max_err_official": np.nan,
            "max_err_reversed": np.nan,
            "best_err": np.nan,
            "inferred_sign": "missing_file",
            "audit_status": "missing_file",
        })
        continue

    df = load_csv_auto(f)
    cols = list(df.columns)
    lookup = lower_lookup(cols)

    # Find actual time pairs present.
    present_pairs = []
    for a, b in TIMEPAIR_CANDIDATES:
        aa = get_col_case_insensitive(lookup, a)
        bb = get_col_case_insensitive(lookup, b)
        if aa is not None and bb is not None:
            present_pairs.append((aa, bb))

    # De-duplicate pairs produced by case-insensitive matching.
    # Example: ("t_dc_s", "t_dcac_s") and ("t_DC_s", "t_DCAC_s")
    # can both map to the same CSV-native pair.
    present_pairs = list(dict.fromkeys(present_pairs))

    # Find actual total-Δt columns present.
    present_dt_cols = []
    for c in TOTAL_DT_CANDIDATES:
        cc = get_col_case_insensitive(lookup, c)
        if cc is not None:
            present_dt_cols.append(cc)

    # De-duplicate dt columns as well.
    present_dt_cols = list(dict.fromkeys(present_dt_cols))

    if not present_pairs or not present_dt_cols:
        sign_rows.append({
            "file": f.name,
            "file_family": inv_row.get("file_family", ""),
            "n_rows": len(df),
            "t_ref_col": "",
            "t_protocol_col": "",
            "dt_col": "",
            "n_valid": 0,
            "median_t_ref": np.nan,
            "median_t_protocol": np.nan,
            "median_dt_stored": np.nan,
            "max_err_official": np.nan,
            "max_err_reversed": np.nan,
            "best_err": np.nan,
            "inferred_sign": "not_auditable_after_load",
            "audit_status": "missing_present_pairs_or_total_dt_cols",
        })
        continue

    for t_ref_col, t_proto_col in present_pairs:
        for dt_col in present_dt_cols:
            # Unit guard:
            # Compare seconds with seconds, minutes with minutes.
            # If unit suffixes are absent or mixed, skip to avoid false audits.
            pair_is_min = t_ref_col.lower().endswith("_min") or t_proto_col.lower().endswith("_min")
            pair_is_s = t_ref_col.lower().endswith("_s") or t_proto_col.lower().endswith("_s")
            dt_is_min = dt_col.lower().endswith("_min")
            dt_is_s = dt_col.lower().endswith("_s")

            if pair_is_min and not dt_is_min:
                continue
            if pair_is_s and not dt_is_s:
                continue

            t_ref = pd.to_numeric(df[t_ref_col], errors="coerce").to_numpy()
            t_proto = pd.to_numeric(df[t_proto_col], errors="coerce").to_numpy()
            dt_stored = pd.to_numeric(df[dt_col], errors="coerce").to_numpy()

            official = t_ref - t_proto
            reversed_ = t_proto - t_ref

            valid = (
                np.isfinite(t_ref)
                & np.isfinite(t_proto)
                & np.isfinite(dt_stored)
            )

            if valid.sum() == 0:
                inferred = "unknown"
                max_err_official = np.nan
                max_err_reversed = np.nan
                best_err = np.nan
                status = "no_valid_rows"
            else:
                max_err_official = float(
                    np.nanmax(np.abs(dt_stored[valid] - official[valid]))
                )
                max_err_reversed = float(
                    np.nanmax(np.abs(dt_stored[valid] - reversed_[valid]))
                )

                if max_err_official <= max_err_reversed:
                    inferred = "official_tRef_minus_tProtocol"
                    best_err = max_err_official
                else:
                    inferred = "reversed_tProtocol_minus_tRef"
                    best_err = max_err_reversed

                status = "ok" if best_err < 1e-6 else "nonzero_alignment_error"

            sign_rows.append({
                "file": f.name,
                "file_family": inv_row.get("file_family", ""),
                "n_rows": len(df),
                "t_ref_col": t_ref_col,
                "t_protocol_col": t_proto_col,
                "dt_col": dt_col,
                "n_valid": int(valid.sum()),
                "median_t_ref": float(np.nanmedian(t_ref)) if len(t_ref) else np.nan,
                "median_t_protocol": float(np.nanmedian(t_proto)) if len(t_proto) else np.nan,
                "median_dt_stored": float(np.nanmedian(dt_stored)) if len(dt_stored) else np.nan,
                "max_err_official": max_err_official,
                "max_err_reversed": max_err_reversed,
                "best_err": best_err,
                "inferred_sign": inferred,
                "audit_status": status,
            })


sign_self_df = pd.DataFrame(sign_rows)

out_sign = DATA / "day19A_step1_csv_timepair_sign_self_audit.csv"
sign_self_df.to_csv(out_sign, index=False)

print(f"\nWrote: {out_sign}")
display(sign_self_df)


# ---------------------------------------------------------------------
# Part B — residual identity audit
# ---------------------------------------------------------------------
# Only valid for files containing:
#   dtQ_s, dtQ_geom_s, dtQ_resid_s
#
# Check:
#   dtQ_s = dtQ_geom_s + dtQ_resid_s
# ---------------------------------------------------------------------

identity_rows = []

for _, inv_row in inv_df.iterrows():
    f = Path(inv_row["path"])

    if not f.exists():
        continue

    try:
        df = load_csv_auto(f)
    except Exception as e:
        identity_rows.append({
            "file": f.name,
            "file_family": inv_row.get("file_family", ""),
            "n_rows": 0,
            "dt_col": "",
            "geom_col": "",
            "resid_col": "",
            "n_valid": 0,
            "max_abs_identity_err_s": np.nan,
            "median_identity_err_s": np.nan,
            "identity_status": f"read_failed: {str(e)[:120]}",
        })
        continue

    lookup = lower_lookup(df.columns)

    dt_col = get_col_case_insensitive(lookup, "dtQ_s")
    geom_col = get_col_case_insensitive(lookup, "dtQ_geom_s")
    resid_col = get_col_case_insensitive(lookup, "dtQ_resid_s")

    if not (dt_col and geom_col and resid_col):
        continue

    dt = pd.to_numeric(df[dt_col], errors="coerce").to_numpy()
    geom = pd.to_numeric(df[geom_col], errors="coerce").to_numpy()
    resid = pd.to_numeric(df[resid_col], errors="coerce").to_numpy()

    valid = np.isfinite(dt) & np.isfinite(geom) & np.isfinite(resid)

    if valid.sum() == 0:
        max_abs_err = np.nan
        median_err = np.nan
        status = "no_valid_rows"
    else:
        err = dt[valid] - (geom[valid] + resid[valid])
        max_abs_err = float(np.nanmax(np.abs(err)))
        median_err = float(np.nanmedian(err))
        status = "ok" if max_abs_err < 1e-6 else "identity_mismatch"

    identity_rows.append({
        "file": f.name,
        "file_family": inv_row.get("file_family", ""),
        "n_rows": len(df),
        "dt_col": dt_col,
        "geom_col": geom_col,
        "resid_col": resid_col,
        "n_valid": int(valid.sum()),
        "max_abs_identity_err_s": max_abs_err,
        "median_identity_err_s": median_err,
        "identity_status": status,
    })


identity_df = pd.DataFrame(identity_rows)

out_identity = DATA / "day19A_step1_csv_residual_identity_audit.csv"
identity_df.to_csv(out_identity, index=False)

print(f"\nWrote: {out_identity}")
display(identity_df)


# ---------------------------------------------------------------------
# Compact summary
# ---------------------------------------------------------------------

print("\nSign audit counts:")
if len(sign_self_df):
    print(sign_self_df.groupby(["inferred_sign", "audit_status"]).size().to_string())
else:
    print("No sign-auditable rows.")

print("\nResidual identity counts:")
if len(identity_df):
    print(identity_df.groupby(["identity_status"]).size().to_string())
else:
    print("No residual identity files found.")

Sign-auditable files from inventory: 7
                                               file    file_family                             time_pair_cols  signed_dt_cols
                    day14_step5_delta_tQ_curves.csv dtQ_curve_like                  ["t_DC_min - t_DCAC_min"] ["delta_t_min"]
     day14_step5_delta_tQ_curves_v1_wrong_phase.csv dtQ_curve_like                  ["t_DC_min - t_DCAC_min"] ["delta_t_min"]
   day14_step5_delta_tQ_curves_v2_aligned_phase.csv dtQ_curve_like                  ["t_DC_min - t_DCAC_min"] ["delta_t_min"]
                day16_step3a_dtQ_curves_long.csv.gz dtQ_curve_like                  ["t_DC_s - t_protocol_s"]       ["dtQ_s"]
             day18_step2_dt_Q_curves_long_v3.csv.gz dtQ_curve_like ["t_DC_s - t_DCAC_s", "t_DC_s - t_DCAC_s"]       ["dtQ_s"]
day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz dtQ_curve_like ["t_DC_s - t_DCAC_s", "t_DC_s - t_DCAC_s"]       ["dtQ_s"]
                  results_day13_delta_tQ_curves.csv dtQ_curve_like             

,file,file_family,n_rows,t_ref_col,t_protocol_col,dt_col,n_valid,median_t_ref,median_t_protocol,median_dt_stored,max_err_official,max_err_reversed,best_err,inferred_sign,audit_status
0,day14_step5_delta_tQ_curves.csv,dtQ_curve_like,5760,t_DC_min,t_DCAC_min,delta_t_min,5755,86.484399,88.471224,-0.364243,5.695444e-14,38.135573,5.695444e-14,official_tRef_minus_tProtocol,ok
1,day14_step5_delta_tQ_curves_v1_wrong_phase.csv,dtQ_curve_like,5760,t_DC_min,t_DCAC_min,delta_t_min,5760,86.484399,84.976683,0.371012,1.137423e-13,38.597107,1.137423e-13,official_tRef_minus_tProtocol,ok
2,day14_step5_delta_tQ_curves_v2_aligned_phase.csv,dtQ_curve_like,5760,t_DC_min,t_DCAC_min,delta_t_min,5755,86.484399,88.471224,-0.364243,5.695444e-14,38.135573,5.695444e-14,official_tRef_minus_tProtocol,ok
3,day16_step3a_dtQ_curves_long.csv.gz,dtQ_curve_like,3600,t_DC_s,t_protocol_s,dtQ_s,3600,8671.740769,8838.165414,-29.305930,3.652190e-12,1056.021876,3.652190e-12,official_tRef_minus_tProtocol,ok
4,day18_step2_dt_Q_curves_long_v3.csv.gz,dtQ_curve_like,560,t_DC_s,t_DCAC_s,dtQ_s,560,8442.642938,9130.312924,-517.203199,3.751666e-12,3214.792228,3.751666e-12,official_tRef_minus_tProtocol,ok
5,day18_step2_dt_Q_curves_long_v4_charge_first.c...,dtQ_curve_like,240,t_DC_s,t_DCAC_s,dtQ_s,240,6679.671418,5574.791221,1343.200582,3.637979e-12,4772.153024,3.637979e-12,official_tRef_minus_tProtocol,ok
6,results_day13_delta_tQ_curves.csv,dtQ_curve_like,9520,t_DC_min,t_DCAC_min,delta_t_min,9520,88.342342,90.182628,-0.355150,1.099121e-13,38.278404,1.099121e-13,official_tRef_minus_tProtocol,ok



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step1_csv_residual_identity_audit.csv


,file,file_family,n_rows,dt_col,geom_col,resid_col,n_valid,max_abs_identity_err_s,median_identity_err_s,identity_status
0,day18_step2_dt_Q_curves_long_v4_charge_first.c...,dtQ_curve_like,240,dtQ_s,dtQ_geom_s,dtQ_resid_s,240,4.547474e-13,0.0,ok



Sign audit counts:
inferred_sign                  audit_status
official_tRef_minus_tProtocol  ok              7

Residual identity counts:
identity_status
ok    1


In [15]:
import pandas as pd
f = DATA / "day18B_smoke_dtQ_resid_curves_long.csv.gz"
df_full = pd.read_csv(f, compression="gzip")
print(f"Cols: {list(df_full.columns)}")
print(f"n_rows: {len(df_full)}")

if "dtQ_geom_s" in df_full.columns and "dtQ_resid_s" in df_full.columns:
    if "dtQ_s" not in df_full.columns:
        # Reconstruct total dt and report stats
        total_dt = df_full["dtQ_geom_s"] + df_full["dtQ_resid_s"]
        print("\nNo total dtQ_s column — total dt is reader-reconstructed.")
        print(f"Reconstructed median dtQ_s (= dtQ_geom + dtQ_resid): {total_dt.median():.4f} s")
        print(f"Median dtQ_geom_s: {df_full['dtQ_geom_s'].median():.4f} s")
        print(f"Median dtQ_resid_s: {df_full['dtQ_resid_s'].median():.4f} s")
    else:
        err = df_full["dtQ_s"] - df_full["dtQ_geom_s"] - df_full["dtQ_resid_s"]
        print(f"\nReconstruct identity max_abs_err: {err.abs().max():.3e}")

Cols: ['pair_id', 'param_set', 'protocol_label', 'DC_ref_protocol', 'n_tau', 'Q_Ah', 'dt_model_s', 'dt_geom_s', 'dt_resid_s']
n_rows: 1032


In [16]:
df = pd.read_csv(DATA / "day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz", compression="gzip")
print(f"Total rows: {len(df)}")
print(f"\nphase_label values:")
print(df["phase_label"].value_counts().to_string())

print("\n--- Group-by phase_label, dtQ_s (raw) ---")
print(df.groupby("phase_label")["dtQ_s"].agg(["count", "mean", "median", "std"]).round(2).to_string())

print("\n--- Group-by phase_label, dtQ_geom_s (geometry term) ---")
print(df.groupby("phase_label")["dtQ_geom_s"].agg(["count", "mean", "median", "std"]).round(2).to_string())

print("\n--- Group-by phase_label, dtQ_resid_s (state-layer residual) ---")
print(df.groupby("phase_label")["dtQ_resid_s"].agg(["count", "mean", "median", "std"]).round(2).to_string())

Total rows: 240

phase_label values:
phase_label
charge_first    240

--- Group-by phase_label, dtQ_s (raw) ---
              count     mean  median     std
phase_label                                 
charge_first    240  1332.47  1343.2  528.07

--- Group-by phase_label, dtQ_geom_s (geometry term) ---
              count     mean  median     std
phase_label                                 
charge_first    240  1332.47  1343.2  528.07

--- Group-by phase_label, dtQ_resid_s (state-layer residual) ---
              count  mean  median  std
phase_label                           
charge_first    240  -0.0    -0.0  0.0


In [17]:
# Cell 3.8b — Day18B smoke decomposition identity + residual magnitude audit
# Checks:
#   Algebra:
#       dt_model_s = dt_geom_s + dt_resid_s
#
#   Physical / methodological null-baseline observable:
#       |dt_resid_s| magnitude
#       |dt_resid_s| / |dt_geom_s|
#
# Output:
#   data/day19A_step1_day18B_smoke_identity_resid_magnitude_audit.csv
#
# Interpretation:
#   Algebra identity is necessary but partly vacuous if dt_resid_s was computed
#   as dt_model_s - dt_geom_s. The non-trivial audit target is residual magnitude.

import numpy as np
import pandas as pd

f = DATA / "day18B_smoke_dtQ_resid_curves_long.csv.gz"
assert f.exists(), f"Missing: {f}"

df = pd.read_csv(f, compression="gzip")

required = {"dt_model_s", "dt_geom_s", "dt_resid_s"}
missing = required - set(df.columns)
assert not missing, f"Missing columns in {f.name}: {missing}"

valid = (
    np.isfinite(df["dt_model_s"])
    & np.isfinite(df["dt_geom_s"])
    & np.isfinite(df["dt_resid_s"])
)

dfv = df.loc[valid].copy()

err = dfv["dt_model_s"] - (dfv["dt_geom_s"] + dfv["dt_resid_s"])

resid_abs = dfv["dt_resid_s"].abs()
geom_abs = dfv["dt_geom_s"].abs().replace(0, np.nan)
model_abs = dfv["dt_model_s"].abs().replace(0, np.nan)

resid_over_geom = (resid_abs / geom_abs).replace([np.inf, -np.inf], np.nan).dropna()
resid_over_model = (resid_abs / model_abs).replace([np.inf, -np.inf], np.nan).dropna()

summary = pd.DataFrame([{
    "file": f.name,
    "n_rows": len(df),
    "n_valid": int(valid.sum()),

    # Algebra identity
    "max_abs_identity_err_s": float(err.abs().max()) if len(err) else np.nan,
    "median_identity_err_s": float(err.median()) if len(err) else np.nan,
    "identity_status": "ok" if len(err) and err.abs().max() < 1e-6 else "identity_mismatch",

    # Non-trivial residual magnitude audit
    "dt_resid_max_abs_s": float(resid_abs.max()) if len(resid_abs) else np.nan,
    "dt_resid_p95_abs_s": float(resid_abs.quantile(0.95)) if len(resid_abs) else np.nan,
    "dt_resid_median_abs_s": float(resid_abs.median()) if len(resid_abs) else np.nan,

    "resid_over_geom_median": float(resid_over_geom.median()) if len(resid_over_geom) else np.nan,
    "resid_over_geom_p95": float(resid_over_geom.quantile(0.95)) if len(resid_over_geom) else np.nan,
    "resid_over_geom_max": float(resid_over_geom.max()) if len(resid_over_geom) else np.nan,

    "resid_over_model_median": float(resid_over_model.median()) if len(resid_over_model) else np.nan,
    "resid_over_model_p95": float(resid_over_model.quantile(0.95)) if len(resid_over_model) else np.nan,
    "resid_over_model_max": float(resid_over_model.max()) if len(resid_over_model) else np.nan,

    # Basic central tendency
    "dt_model_mean_s": float(dfv["dt_model_s"].mean()) if len(dfv) else np.nan,
    "dt_geom_mean_s": float(dfv["dt_geom_s"].mean()) if len(dfv) else np.nan,
    "dt_resid_mean_s": float(dfv["dt_resid_s"].mean()) if len(dfv) else np.nan,
    "dt_model_median_s": float(dfv["dt_model_s"].median()) if len(dfv) else np.nan,
    "dt_geom_median_s": float(dfv["dt_geom_s"].median()) if len(dfv) else np.nan,
    "dt_resid_median_s": float(dfv["dt_resid_s"].median()) if len(dfv) else np.nan,

    # Null-baseline verdict
    "residual_magnitude_status": (
        "near_zero_null_baseline"
        if len(resid_abs) and resid_abs.max() < 0.1
        else "nonzero_residual_requires_inspection"
    ),
}])

out = DATA / "day19A_step1_day18B_smoke_identity_resid_magnitude_audit.csv"
summary.to_csv(out, index=False)

print(f"Wrote: {out}")
display(summary)


# Group-level residual magnitude summary
group_cols = ["param_set", "protocol_label"]
available_group_cols = [c for c in group_cols if c in dfv.columns]

if available_group_cols:
    group_summary = (
        dfv
        .assign(
            abs_dt_resid_s=dfv["dt_resid_s"].abs(),
            abs_dt_geom_s=dfv["dt_geom_s"].abs(),
            resid_over_geom=(
                dfv["dt_resid_s"].abs()
                / dfv["dt_geom_s"].abs().replace(0, np.nan)
            ),
        )
        .groupby(available_group_cols)
        .agg(
            n=("dt_resid_s", "size"),
            dt_model_median_s=("dt_model_s", "median"),
            dt_geom_median_s=("dt_geom_s", "median"),
            dt_resid_median_s=("dt_resid_s", "median"),
            dt_resid_max_abs_s=("abs_dt_resid_s", "max"),
            dt_resid_p95_abs_s=("abs_dt_resid_s", lambda x: x.quantile(0.95)),
            resid_over_geom_median=("resid_over_geom", "median"),
            resid_over_geom_max=("resid_over_geom", "max"),
        )
        .reset_index()
    )

    out_group = DATA / "day19A_step1_day18B_smoke_resid_magnitude_by_pair.csv"
    group_summary.to_csv(out_group, index=False)

    print(f"\nWrote: {out_group}")
    display(group_summary.round(8))
else:
    print("\nNo param_set/protocol_label columns available for group summary.")

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step1_day18B_smoke_identity_resid_magnitude_audit.csv


,file,n_rows,n_valid,max_abs_identity_err_s,median_identity_err_s,identity_status,dt_resid_max_abs_s,dt_resid_p95_abs_s,dt_resid_median_abs_s,resid_over_geom_median,...,resid_over_model_median,resid_over_model_p95,resid_over_model_max,dt_model_mean_s,dt_geom_mean_s,dt_resid_mean_s,dt_model_median_s,dt_geom_median_s,dt_resid_median_s,residual_magnitude_status
0,day18B_smoke_dtQ_resid_curves_long.csv.gz,1032,1032,2.273737e-13,0.0,ok,0.044075,0.009089,0.001425,0.00004,...,0.00004,0.000523,0.012104,274.840859,274.843535,-0.002676,57.64851,57.652099,-0.00128,near_zero_null_baseline



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step1_day18B_smoke_resid_magnitude_by_pair.csv


,param_set,protocol_label,n,dt_model_median_s,dt_geom_median_s,dt_resid_median_s,dt_resid_max_abs_s,dt_resid_p95_abs_s,resid_over_geom_median,resid_over_geom_max
0,Chen2020,DCAC_DC0p2C_AC0p4C_0p1tau,79,9.580588,9.582854,-0.002485,0.011351,0.005354,0.000258,0.001595
1,Chen2020,DCAC_DC0p2C_AC0p4C_10tau,78,956.980821,956.981681,-0.001009,0.008537,0.004498,0.000001,0.000005
2,Chen2020,DCAC_DC0p2C_AC0p4C_1tau,75,94.396458,94.397649,-0.006896,0.015618,0.015388,0.000078,0.000217
3,Chen2020,DCAC_DC0p4C_AC0p6C_0p1tau,67,6.263684,6.264639,-0.001129,0.003779,0.002565,0.000198,0.011959
4,Chen2020,DCAC_DC0p4C_AC0p6C_10tau,60,553.482762,553.483956,-0.001194,0.010788,0.005713,0.000003,0.000022
5,Chen2020,DCAC_DC0p4C_AC0p6C_1tau,64,67.575548,67.579857,-0.005453,0.044075,0.013226,0.000085,0.003127
6,Ecker2015,DCAC_DC0p2C_AC0p4C_0p1tau,29,4.285321,4.285865,-0.000360,0.009043,0.001555,0.000098,0.001432
7,Ecker2015,DCAC_DC0p2C_AC0p4C_10tau,29,406.805769,406.811286,-0.005198,0.027433,0.019991,0.000014,0.000059
8,Ecker2015,DCAC_DC0p2C_AC0p4C_1tau,29,38.972053,38.972688,-0.000333,0.004169,0.002745,0.000014,0.000158
9,Ecker2015,DCAC_DC0p4C_AC0p6C_0p1tau,28,2.879534,2.879663,-0.000151,0.004625,0.000970,0.000047,0.002423


In [19]:
# Cell 3.9 — t-axis anomaly diagnostic for has_Q_but_time_axis_invalid records
# Inputs:
#   data/day19A_step1_legacy_json_trajectory_validation.csv (cell 3.6c)
#   results_day8_*.json / results_day9_*.json (originals)
# Output:
#   data/day19A_step1_t_axis_anomaly_diagnostic.csv
#
# Design choices (4 patches over draft, all merged):
#   1. CV-phase = tail_only (index) AND after_CC_time (physical time)
#      Index-tail alone is semantically ambiguous when t is non-monotonic.
#   2. dt-classification uses EPS_T = 1e-12 tolerance for JSON round-trip safety.
#   3. min/max via np.nanmin/np.nanmax with all-NaN guard.
#   4. Verdict tiers hierarchical: dedup_usable > cv_tail_fallback > must_exclude.

import json
import numpy as np
import pandas as pd

EPS_T = 1e-12

val_df = pd.read_csv(DATA / "day19A_step1_legacy_json_trajectory_validation.csv")
invalid = val_df[val_df["audit_use"] == "has_Q_but_time_axis_invalid_or_missing"].copy()

print(f"Records flagged for t-axis anomaly: {len(invalid)}\n")
print(invalid[["file", "record_idx", "case_id",
               "I_DC_Crate", "A_Crate", "f_Hz", "kappa",
               "n_t", "t_last_s", "total_time_s",
               "Q_final_stored_mAh"]].to_string(index=False))


def classify_anomaly(dt_arr, t_arr):
    """Return (anomaly_class, recoverable_via_dedup)."""
    n_nan_t = int(np.sum(~np.isfinite(t_arr)))
    n_dup = int(np.sum(np.abs(dt_arr) <= EPS_T))
    n_neg = int(np.sum(dt_arr < -EPS_T))

    if n_nan_t > 0:
        return "nan_time_values", False
    if n_neg > 0 and n_dup > 0:
        return "mixed_negative_and_duplicates", False
    if n_neg > 0:
        return "negative_time_jumps", False
    if n_dup > 0:
        return "duplicate_timestamps_only", True
    return "unclassified", False


def safe_nan_minmax(arr):
    """np.nanmin/nanmax with all-NaN guard to avoid RuntimeWarning."""
    if not np.isfinite(arr).any():
        return np.nan, np.nan
    return float(np.nanmin(arr)), float(np.nanmax(arr))


diag_rows = []

for fname in invalid["file"].unique():
    full_path = REPO / fname
    if not full_path.exists():
        print(f"[WARN] Missing source JSON: {fname}")
        continue

    with open(full_path, encoding="utf-8") as fh:
        records = json.load(fh)

    sub = invalid[invalid["file"] == fname]
    for _, ir in sub.iterrows():
        idx = int(ir["record_idx"])
        rec = records[idx]

        t = np.asarray(rec.get("t_chg", []), dtype=float)
        if len(t) < 2:
            diag_rows.append({
                "file": fname,
                "record_idx": idx,
                "case_id": rec.get("case_id"),
                "anomaly_class": "trivially_short",
                "n_total": len(t),
                "recoverable_via_dedup": False,
                "cv_phase_candidate": False,
            })
            continue

        dt = np.diff(t)
        anomaly_class, recoverable = classify_anomaly(dt, t)

        # Locate anomalies — tolerance-aware
        anomaly_mask = (dt <= EPS_T) | ~np.isfinite(dt)
        anomaly_pos = np.where(anomaly_mask)[0]

        if len(anomaly_pos) > 0:
            first_idx = int(anomaly_pos[0])
            last_idx = int(anomaly_pos[-1])
            first_frac = first_idx / len(t)
            last_frac = last_idx / len(t)
            first_anomaly_t_s = float(t[first_idx])
            last_anomaly_t_s = float(t[last_idx])
            tail_only = bool(first_frac > 0.85)
        else:
            first_idx = last_idx = -1
            first_frac = last_frac = np.nan
            first_anomaly_t_s = last_anomaly_t_s = np.nan
            tail_only = False

        CC_time_s = rec.get("CC_time_s", np.nan)
        total_time_s = rec.get("total_time_s", np.nan)

        after_CC_time = bool(
            np.isfinite(CC_time_s)
            and np.isfinite(first_anomaly_t_s)
            and first_anomaly_t_s >= CC_time_s
        )
        cv_phase_candidate = bool(tail_only and after_CC_time)

        # Q-grid coverage of clean prefix (from start up to first anomaly)
        Q = np.asarray(rec.get("Q_net_trajectory", []), dtype=float)
        if len(Q) == len(t) and first_idx > 0 and len(Q) > 0:
            Q_at_first_anomaly = float(Q[first_idx])
            Q_final = float(Q[-1])
            Q_coverage_clean_pct = (
                (Q_at_first_anomaly / Q_final * 100.0)
                if Q_final != 0 else np.nan
            )
        else:
            Q_at_first_anomaly = np.nan
            Q_coverage_clean_pct = np.nan

        min_dt_s, max_dt_s = safe_nan_minmax(dt)
        median_pos_dt_s = (
            float(np.median(dt[dt > EPS_T])) if (dt > EPS_T).any() else np.nan
        )

        diag_rows.append({
            "file": fname,
            "record_idx": idx,
            "case_id": rec.get("case_id"),
            "I_DC_Crate": rec.get("I_DC_Crate"),
            "A_Crate": rec.get("A_Crate"),
            "f_Hz": rec.get("f_Hz"),
            "tau_label": rec.get("tau_label"),
            "kappa": rec.get("kappa"),
            "n_total": len(t),
            "n_dup_dt_zero": int(np.sum(np.abs(dt) <= EPS_T)),
            "n_negative_dt": int(np.sum(dt < -EPS_T)),
            "n_nan_in_t": int(np.sum(~np.isfinite(t))),
            "min_dt_s": min_dt_s,
            "max_dt_s": max_dt_s,
            "median_pos_dt_s": median_pos_dt_s,
            "first_anomaly_idx": first_idx,
            "first_anomaly_frac_of_traj": first_frac,
            "first_anomaly_t_s": first_anomaly_t_s,
            "last_anomaly_idx": last_idx,
            "last_anomaly_frac_of_traj": last_frac,
            "last_anomaly_t_s": last_anomaly_t_s,
            "CC_time_s": CC_time_s,
            "total_time_s": total_time_s,
            "after_CC_time": after_CC_time,
            "tail_only_above_0p85": tail_only,
            "cv_phase_candidate": cv_phase_candidate,
            "Q_at_first_anomaly_mAh": Q_at_first_anomaly,
            "Q_coverage_clean_pct": Q_coverage_clean_pct,
            "anomaly_class": anomaly_class,
            "recoverable_via_dedup": recoverable,
        })

diag_df = pd.DataFrame(diag_rows)

out = DATA / "day19A_step1_t_axis_anomaly_diagnostic.csv"
diag_df.to_csv(out, index=False)
print(f"\nWrote: {out}\n")

display_cols = [
    "file", "record_idx", "case_id",
    "I_DC_Crate", "A_Crate", "f_Hz", "tau_label", "kappa",
    "n_total", "n_dup_dt_zero", "n_negative_dt", "n_nan_in_t",
    "first_anomaly_frac_of_traj", "first_anomaly_t_s", "CC_time_s",
    "after_CC_time", "tail_only_above_0p85", "cv_phase_candidate",
    "Q_coverage_clean_pct",
    "anomaly_class", "recoverable_via_dedup",
]
display(diag_df[display_cols])

print("\nAnomaly class distribution:")
print(diag_df["anomaly_class"].value_counts().to_string())

# Hierarchical verdict (recoverable_via_dedup takes precedence)
n_dedup_usable = int(diag_df["recoverable_via_dedup"].sum())
n_cv_tail_fallback = int(
    ((~diag_df["recoverable_via_dedup"]) & diag_df["cv_phase_candidate"]).sum()
)
n_must_exclude = int(
    ((~diag_df["recoverable_via_dedup"]) & (~diag_df["cv_phase_candidate"])).sum()
)

print("\n--- Verdict for §2.3 implementation (hierarchical) ---")
print(f"  · {n_dedup_usable} records: dup_only → keep-last dedup → fully usable")
print(f"  · {n_cv_tail_fallback} records: not-dedup-recoverable but CV-phase tail → truncate at t < first_anomaly_t_s, CC region usable")
print(f"  · {n_must_exclude} records: must EXCLUDE from §2.3 (anomaly in CC phase or NaN)")

Records flagged for t-axis anomaly: 5

                                    file  record_idx                    case_id  I_DC_Crate  A_Crate     f_Hz  kappa  n_t     t_last_s  total_time_s  Q_final_stored_mAh
data/results_day8_x2_dfn_mj1freq_v2.json           4 DC0.20C+AC0.30C_f0.00041Hz         0.2      0.3 0.000410    1.5  444 20381.305183  20381.305183         5127.593850
data/results_day8_x2_dfn_mj1freq_v2.json          25           DC0.20C_baseline         0.2      0.0 0.000000    0.0  236 20384.965532  20384.965532         5126.330542
  data/results_day8_x4_dfn_chenfreq.json          25           DC0.20C_baseline         0.2      0.0 0.000000    0.0  236 20384.965532  20384.965532         5126.330542
  data/results_day8_x4_dfn_chenfreq.json          34 DC0.20C+AC0.30C_f0.00080Hz         0.2      0.3 0.000805    1.5  814 20565.594760  20565.594760         5126.884544
  data/results_day8_x4_dfn_chenfreq.json          56           DC0.20C_baseline         0.2      0.0 0.000000    0.0

,file,record_idx,case_id,I_DC_Crate,A_Crate,f_Hz,tau_label,kappa,n_total,n_dup_dt_zero,...,n_nan_in_t,first_anomaly_frac_of_traj,first_anomaly_t_s,CC_time_s,after_CC_time,tail_only_above_0p85,cv_phase_candidate,Q_coverage_clean_pct,anomaly_class,recoverable_via_dedup
0,data/results_day8_x2_dfn_mj1freq_v2.json,4,DC0.20C+AC0.30C_f0.00041Hz,0.2,0.3,0.000410,34.8τ,1.5,444,1,...,0,0.880631,16439.149737,16439.149737,True,True,True,85.720807,duplicate_timestamps_only,True
1,data/results_day8_x2_dfn_mj1freq_v2.json,25,DC0.20C_baseline,0.2,0.0,0.000000,baseline,0.0,236,1,...,0,0.817797,17512.350032,17512.350032,True,False,False,94.893250,duplicate_timestamps_only,True
2,data/results_day8_x4_dfn_chenfreq.json,25,DC0.20C_baseline,0.2,0.0,0.000000,baseline,0.0,236,1,...,0,0.817797,17512.350032,17512.350032,True,False,False,94.893250,duplicate_timestamps_only,True
3,data/results_day8_x4_dfn_chenfreq.json,34,DC0.20C+AC0.30C_f0.00080Hz,0.2,0.3,0.000805,10τ,1.5,814,1,...,0,0.928747,16958.846911,16958.846911,True,True,True,89.344344,duplicate_timestamps_only,True
4,data/results_day8_x4_dfn_chenfreq.json,56,DC0.20C_baseline,0.2,0.0,0.000000,baseline,0.0,236,1,...,0,0.817797,17512.350032,17512.350032,True,False,False,94.893250,duplicate_timestamps_only,True



Anomaly class distribution:
anomaly_class
duplicate_timestamps_only    5

--- Verdict for §2.3 implementation (hierarchical) ---
  · 5 records: dup_only → keep-last dedup → fully usable
  · 0 records: not-dedup-recoverable but CV-phase tail → truncate at t < first_anomaly_t_s, CC region usable
  · 0 records: must EXCLUDE from §2.3 (anomaly in CC phase or NaN)


In [20]:
# Cell 4A — Waveform-only geometry helpers
#
# Defines the canonical helper functions used throughout Day 19A audit and
# the JES2 §4 framework. All helpers operate on prescribed currents only —
# no PyBaMM cell model is invoked.
#
# Convention LOCKS:
#   - PyBaMM current sign:    I > 0 = discharge,  I < 0 = charge
#   - Q_net accumulation:     Q_mAh(t) = -∫_0^t I(τ) dτ × (1000 / 3600)
#   - Δt_geom sign:           Δt > 0 → DCAC reaches Q* faster than DC arm
#   - First-passage default:  raw (no cummax). raw and cummax give identical
#                              first-crossing times mathematically; cummax
#                              retained only to reproduce / diagnose historical
#                              envelope-based implementations and interpolation
#                              behavior on coarse, non-monotone Q(t).
#   - Phase: 'charge_first', 'discharge_first', 'dc_only'
#
# Helpers enforce strict-monotone t precondition; callers handling raw legacy
# trajectories must dedup first via clean_time_series_keep_last (Cell 4C).
#
# Dependencies: numpy, scipy.integrate.cumulative_trapezoid

import numpy as np
from scipy.integrate import cumulative_trapezoid

PHASE_CHARGE_FIRST = "charge_first"
PHASE_DISCHARGE_FIRST = "discharge_first"
PHASE_DC_ONLY = "dc_only"
ALL_PHASES = (PHASE_CHARGE_FIRST, PHASE_DISCHARGE_FIRST, PHASE_DC_ONLY)

FP_MODE_RAW = "raw"
FP_MODE_CUMMAX = "cummax"
ALL_FP_MODES = (FP_MODE_RAW, FP_MODE_CUMMAX)


# -----------------------------------------------------------------------------
# 1. prescribed_current_py
# -----------------------------------------------------------------------------
def prescribed_current_py(t, dc_A, ac_A, f_Hz, phase=PHASE_CHARGE_FIRST):
    """
    Generate PyBaMM-convention current waveform I_py(t).

    PyBaMM convention: I > 0 = discharge, I < 0 = charge.
    `dc_A` and `ac_A` are positive magnitudes; the sign is encoded in `phase`.

    Phase formulas:
        charge_first    : I_py(t) = -dc_A - ac_A * sin(2π f_Hz t)
        discharge_first : I_py(t) = -dc_A + ac_A * sin(2π f_Hz t)
        dc_only         : I_py(t) = -dc_A

    Parameters
    ----------
    t : array-like
        Time grid in seconds.
    dc_A : float, > 0
        DC charging magnitude in Amperes.
    ac_A : float, >= 0
        AC peak amplitude in Amperes.
    f_Hz : float
        AC frequency in Hz. Must be > 0 for AC phases. Ignored for `dc_only`.
    phase : str
        One of ALL_PHASES.

    Returns
    -------
    I_py : np.ndarray
        Current in PyBaMM convention, same shape as `t`.
    """
    t = np.asarray(t, dtype=float)

    if dc_A <= 0:
        raise ValueError(f"dc_A must be > 0 (charging magnitude); got {dc_A}")
    if ac_A < 0:
        raise ValueError(f"ac_A must be >= 0; got {ac_A}")
    if phase not in ALL_PHASES:
        raise ValueError(f"phase must be in {ALL_PHASES}; got {phase!r}")
    if phase != PHASE_DC_ONLY and f_Hz <= 0:
        raise ValueError(
            f"f_Hz must be > 0 for AC phases ({phase!r}); got {f_Hz}. "
            "Use phase='dc_only' explicitly for DC reference arms."
        )

    if phase == PHASE_DC_ONLY:
        return -dc_A * np.ones_like(t)

    if phase == PHASE_CHARGE_FIRST:
        return -dc_A - ac_A * np.sin(2 * np.pi * f_Hz * t)

    if phase == PHASE_DISCHARGE_FIRST:
        return -dc_A + ac_A * np.sin(2 * np.pi * f_Hz * t)

    raise RuntimeError(f"Unhandled phase: {phase!r}")


# -----------------------------------------------------------------------------
# 2. compute_Q_net_mAh
# -----------------------------------------------------------------------------
def compute_Q_net_mAh(t, I_A):
    """
    Compute strict-net charge accumulated as a function of time, in mAh.

        Q_mAh(t) = -∫_0^t I(τ) dτ × (1000 / 3600)

    Validated against PyBaMM legacy JSON trajectories: Cell 3.6c shows
    247/247 records reproduce stored Q_net_trajectory with RMSE = 0.0.

    Parameters
    ----------
    t : array-like
        Time grid in seconds; must be strictly increasing and finite.
    I_A : array-like
        Current in Amperes, PyBaMM convention; must be finite.

    Returns
    -------
    Q_mAh : np.ndarray
        Cumulative net charge in mAh, same length as `t`, Q_mAh[0] = 0.
    """
    t = np.asarray(t, dtype=float)
    I_A = np.asarray(I_A, dtype=float)

    if t.shape != I_A.shape:
        raise ValueError(
            f"t and I_A must have same shape; got {t.shape} vs {I_A.shape}"
        )
    if len(t) > 1 and not np.all(np.diff(t) > 0):
        raise ValueError(
            "t must be strictly increasing before integration; "
            "clean duplicate timestamps first."
        )
    if not np.all(np.isfinite(t)) or not np.all(np.isfinite(I_A)):
        raise ValueError("t and I_A must be finite (no NaN / inf).")

    return -cumulative_trapezoid(I_A, t, initial=0.0) * 1000.0 / 3600.0


# -----------------------------------------------------------------------------
# 3. first_passage_time_raw
# -----------------------------------------------------------------------------
def first_passage_time_raw(t, Q_mAh, Q_target_mAh, mode=FP_MODE_RAW):
    """
    First-passage time at which Q(t) reaches Q_target.

    Two modes:
        raw     : scan raw Q for first index i with Q[i] >= Q_target;
                  linearly interpolate between (Q[i-1], t[i-1]) and (Q[i], t[i]).

        cummax  : Q is replaced by its running maximum before the same
                  first-crossing search. For exact first-passage queries,
                  raw and cummax crossing times are mathematically equivalent.
                  This option is retained to reproduce / diagnose historical
                  envelope-based implementations and interpolation behavior
                  on coarse, non-monotone trajectories. The two modes can
                  differ by sub-grid-cell time when Q drops before the crossing
                  segment, because cummax replaces a trough-then-rise ascending
                  segment with a plateau-then-jump segment for interpolation.

    Preconditions:
        - t and Q_mAh have the same shape
        - t is strictly increasing
        - all values are finite

    Parameters
    ----------
    t : array-like
        Time grid in seconds.
    Q_mAh : array-like
        Charge trajectory in mAh.
    Q_target_mAh : float
        Target Q value.
    mode : str
        One of ALL_FP_MODES.

    Returns
    -------
    t_first : float
        Interpolated first-passage time in seconds, or NaN if never crossed.
    """
    t = np.asarray(t, dtype=float)
    Q = np.asarray(Q_mAh, dtype=float)

    if mode not in ALL_FP_MODES:
        raise ValueError(f"mode must be in {ALL_FP_MODES}; got {mode!r}")
    if t.shape != Q.shape:
        raise ValueError(
            f"t and Q_mAh must have same shape; got {t.shape} vs {Q.shape}"
        )
    if len(t) > 1 and not np.all(np.diff(t) > 0):
        raise ValueError(
            "t must be strictly increasing; clean duplicate timestamps first."
        )
    if not np.all(np.isfinite(t)) or not np.all(np.isfinite(Q)):
        raise ValueError("t and Q_mAh must be finite (no NaN / inf).")
    if not np.isfinite(Q_target_mAh):
        raise ValueError(f"Q_target_mAh must be finite; got {Q_target_mAh}")

    if mode == FP_MODE_CUMMAX:
        Q = np.maximum.accumulate(Q)

    crossed = Q >= Q_target_mAh
    if not crossed.any():
        return np.nan

    idx = int(np.argmax(crossed))  # first True index
    if idx == 0:
        return float(t[0])

    Q_lo, Q_hi = Q[idx - 1], Q[idx]
    t_lo, t_hi = t[idx - 1], t[idx]

    if Q_hi == Q_lo:
        return float(t_hi)

    frac = (Q_target_mAh - Q_lo) / (Q_hi - Q_lo)
    return float(t_lo + frac * (t_hi - t_lo))


# -----------------------------------------------------------------------------
# 4. times_on_Q_grid
# -----------------------------------------------------------------------------
def times_on_Q_grid(t, Q_mAh, Q_grid_mAh, mode=FP_MODE_RAW):
    """
    Vectorize first_passage_time_raw over a Q grid.

    Linear scan implementation per Q-target. Acceptable for
    N_grid ≈ O(10²), N_traj ≈ O(10⁵).

    Parameters
    ----------
    t : array-like
        Time grid in seconds.
    Q_mAh : array-like
        Charge trajectory.
    Q_grid_mAh : array-like
        Target Q values.
    mode : str
        One of ALL_FP_MODES.

    Returns
    -------
    t_array : np.ndarray
        First-passage time for each Q in grid; NaN if never reached.
    """
    Q_grid_mAh = np.asarray(Q_grid_mAh, dtype=float)

    if Q_grid_mAh.size == 0:
        raise ValueError("Q_grid_mAh must not be empty.")
    if not np.all(np.isfinite(Q_grid_mAh)):
        raise ValueError("Q_grid_mAh must be finite.")

    if mode == FP_MODE_CUMMAX:
        Q_eff = np.maximum.accumulate(np.asarray(Q_mAh, dtype=float))
        scan_mode = FP_MODE_RAW
    elif mode == FP_MODE_RAW:
        Q_eff = np.asarray(Q_mAh, dtype=float)
        scan_mode = FP_MODE_RAW
    else:
        raise ValueError(f"mode must be in {ALL_FP_MODES}; got {mode!r}")

    out = np.empty_like(Q_grid_mAh, dtype=float)
    for i, Q_t in enumerate(Q_grid_mAh):
        out[i] = first_passage_time_raw(t, Q_eff, Q_t, mode=scan_mode)

    return out


# -----------------------------------------------------------------------------
# 5. dt_from_trajectories
# -----------------------------------------------------------------------------
def dt_from_trajectories(
    t_DC,
    I_DC_A,
    t_DCAC,
    I_DCAC_A,
    Q_grid_mAh,
    mode=FP_MODE_RAW,
):
    """
    Pair-wise Δt(Q) from two raw current trajectories on a shared Q grid.

    For each Q*:
        Δt(Q*) = t_first(DC arm, Q*) - t_first(DCAC arm, Q*)

    Sign convention:
        Δt > 0 → DCAC reaches Q* faster than DC arm.

    Caller is responsible for time-axis cleanup. Pre-clean both trajectories
    via clean_time_series_keep_last (Cell 4C) when working with raw legacy
    PyBaMM JSON / .npz outputs that may contain CC→CV stitching duplicates.

    Parameters
    ----------
    t_DC, I_DC_A : array-like
        DC reference arm trajectory: time grid (s), current (A, PyBaMM convention).
    t_DCAC, I_DCAC_A : array-like
        DCAC arm trajectory.
    Q_grid_mAh : array-like
        Target Q values.
    mode : str
        First-passage mode for both arms.

    Returns
    -------
    out : dict
        Q_grid_mAh, t_DC_s, t_DCAC_s, dt_s, Q_DC_final_mAh, Q_DCAC_final_mAh
    """
    t_DC = np.asarray(t_DC, dtype=float)
    I_DC_A = np.asarray(I_DC_A, dtype=float)
    t_DCAC = np.asarray(t_DCAC, dtype=float)
    I_DCAC_A = np.asarray(I_DCAC_A, dtype=float)
    Q_grid_mAh = np.asarray(Q_grid_mAh, dtype=float)

    if t_DC.shape != I_DC_A.shape:
        raise ValueError(
            f"t_DC and I_DC_A shape mismatch: {t_DC.shape} vs {I_DC_A.shape}"
        )
    if t_DCAC.shape != I_DCAC_A.shape:
        raise ValueError(
            f"t_DCAC and I_DCAC_A shape mismatch: {t_DCAC.shape} vs {I_DCAC_A.shape}"
        )

    for name, t_arr in [("t_DC", t_DC), ("t_DCAC", t_DCAC)]:
        if len(t_arr) > 1 and not np.all(np.diff(t_arr) > 0):
            raise ValueError(
                f"{name} must be strictly increasing; "
                "use clean_time_series_keep_last (Cell 4C) to dedup first."
            )
        if not np.all(np.isfinite(t_arr)):
            raise ValueError(f"{name} must be finite.")

    if not np.all(np.isfinite(I_DC_A)) or not np.all(np.isfinite(I_DCAC_A)):
        raise ValueError("Current arrays must be finite.")

    Q_DC = compute_Q_net_mAh(t_DC, I_DC_A)
    Q_DCAC = compute_Q_net_mAh(t_DCAC, I_DCAC_A)

    t_first_DC = times_on_Q_grid(t_DC, Q_DC, Q_grid_mAh, mode=mode)
    t_first_DCAC = times_on_Q_grid(t_DCAC, Q_DCAC, Q_grid_mAh, mode=mode)

    return {
        "Q_grid_mAh": Q_grid_mAh,
        "t_DC_s": t_first_DC,
        "t_DCAC_s": t_first_DCAC,
        "dt_s": t_first_DC - t_first_DCAC,
        "Q_DC_final_mAh": float(Q_DC[-1]) if len(Q_DC) > 0 else np.nan,
        "Q_DCAC_final_mAh": float(Q_DCAC[-1]) if len(Q_DCAC) > 0 else np.nan,
    }


# -----------------------------------------------------------------------------
# 6. dt_geom_from_protocol
# -----------------------------------------------------------------------------
def dt_geom_from_protocol(
    I_DC_A,
    I_AC_A,
    f_Hz,
    phase,
    Q_grid_mAh,
    dt_eval_s=0.1,
    t_max_s=None,
    t_max_safety_factor=1.5,
    fp_mode=FP_MODE_RAW,
):
    """
    Waveform-only geometry baseline Δt_geom(Q) for a prescribed-current
    protocol pair: DC reference arm vs DC+AC arm.

    No PyBaMM cell model is invoked. Q(t) is determined purely by the imposed
    analytical current waveform and integrated by strict-net trapezoidal rule.

    Parameters
    ----------
    I_DC_A : float, > 0
        DC charging magnitude in Amperes.
    I_AC_A : float, >= 0
        AC peak amplitude in Amperes.
    f_Hz : float, > 0
        AC frequency.
    phase : str
        DCAC arm phase. Must be 'charge_first' or 'discharge_first'.
        DC reference arm always uses 'dc_only'.
    Q_grid_mAh : array-like
        Target Q values for first-passage evaluation.
    dt_eval_s : float
        Time grid resolution. Default 0.1 s; can be tightened if
        interpolation error is suspected.
    t_max_s : float or None
        Time horizon. Default: t_max_safety_factor × Q_max_mAh × 3.6 / I_DC_A.
    t_max_safety_factor : float
        Multiplier on DC-only completion time at Q_max.
    fp_mode : str
        First-passage mode. Default: 'raw'.

    Returns
    -------
    out : dict
        Q_grid_mAh, t_DC_s, t_DCAC_s, dt_geom_s, metadata.

    Sanity anchor, verified in Cell 4B:
        MJ1 0.3C+0.7C 10τ, charge_first, 80-point grid [0.163, 2.425] Ah,
        I_DC=1.02 A, I_AC=2.38 A, f≈1.4339 mHz:
            mean(dt_geom_s) ≈ +335.51 s
    """
    if phase not in (PHASE_CHARGE_FIRST, PHASE_DISCHARGE_FIRST):
        raise ValueError(
            f"phase must be 'charge_first' or 'discharge_first' "
            f"for dt_geom_from_protocol; got {phase!r}"
        )
    if I_DC_A <= 0:
        raise ValueError(f"I_DC_A must be > 0; got {I_DC_A}")
    if I_AC_A < 0:
        raise ValueError(f"I_AC_A must be >= 0; got {I_AC_A}")
    if f_Hz <= 0:
        raise ValueError(f"f_Hz must be > 0; got {f_Hz}")
    if dt_eval_s <= 0:
        raise ValueError(f"dt_eval_s must be > 0; got {dt_eval_s}")
    if t_max_safety_factor <= 0:
        raise ValueError(
            f"t_max_safety_factor must be > 0; got {t_max_safety_factor}"
        )

    Q_grid_mAh = np.asarray(Q_grid_mAh, dtype=float)
    if Q_grid_mAh.size == 0:
        raise ValueError("Q_grid_mAh must not be empty.")
    if not np.all(np.isfinite(Q_grid_mAh)):
        raise ValueError("Q_grid_mAh must be finite.")

    Q_max = float(np.nanmax(Q_grid_mAh))

    if t_max_s is None:
        # t [s] = Q [mAh] × 3.6 / I [A]
        t_max_s = t_max_safety_factor * Q_max * 3.6 / I_DC_A
    elif t_max_s <= 0:
        raise ValueError(f"t_max_s must be > 0 if provided; got {t_max_s}")

    t = np.arange(0.0, t_max_s + dt_eval_s, dt_eval_s)

    # DC reference arm
    I_dc_arm = prescribed_current_py(t, I_DC_A, 0.0, f_Hz, phase=PHASE_DC_ONLY)
    Q_dc_arm = compute_Q_net_mAh(t, I_dc_arm)
    t_DC = times_on_Q_grid(t, Q_dc_arm, Q_grid_mAh, mode=fp_mode)

    # DCAC arm
    I_dcac_arm = prescribed_current_py(t, I_DC_A, I_AC_A, f_Hz, phase=phase)
    Q_dcac_arm = compute_Q_net_mAh(t, I_dcac_arm)
    t_DCAC = times_on_Q_grid(t, Q_dcac_arm, Q_grid_mAh, mode=fp_mode)

    return {
        "Q_grid_mAh": Q_grid_mAh,
        "t_DC_s": t_DC,
        "t_DCAC_s": t_DCAC,
        "dt_geom_s": t_DC - t_DCAC,
        "metadata": {
            "I_DC_A": float(I_DC_A),
            "I_AC_A": float(I_AC_A),
            "f_Hz": float(f_Hz),
            "phase": phase,
            "dt_eval_s": float(dt_eval_s),
            "t_max_s": float(t_max_s),
            "t_max_safety_factor": float(t_max_safety_factor),
            "fp_mode": fp_mode,
            "n_t_points": int(len(t)),
            "Q_grid_min_mAh": float(np.nanmin(Q_grid_mAh)),
            "Q_grid_max_mAh": Q_max,
            "Q_DC_final_mAh": float(Q_dc_arm[-1]),
            "Q_DCAC_final_mAh": float(Q_dcac_arm[-1]),
            "kappa": float(I_AC_A / I_DC_A),
        },
    }


print("Geometry helper library loaded.")
print("Functions defined:")
for fn in (
    prescribed_current_py,
    compute_Q_net_mAh,
    first_passage_time_raw,
    times_on_Q_grid,
    dt_from_trajectories,
    dt_geom_from_protocol,
):
    print(f"  · {fn.__name__}")

print(f"Phase constants: {ALL_PHASES}")
print(f"FP-mode constants: {ALL_FP_MODES}")

Geometry helper library loaded.
Functions defined:
  · prescribed_current_py
  · compute_Q_net_mAh
  · first_passage_time_raw
  · times_on_Q_grid
  · dt_from_trajectories
  · dt_geom_from_protocol
Phase constants: ('charge_first', 'discharge_first', 'dc_only')
FP-mode constants: ('raw', 'cummax')


In [22]:
# Cell 4B — MJ1 0.3+0.7C 10τ waveform-geometry sanity anchor
#                      (Option B + persist cummax branch)
#
# Validates Cell 4A geometry helpers against the frozen MJ1 charge-first
# anchor under raw and cummax first-passage modes × charge_first /
# discharge_first phases.
#
# Geometry-only setup (no PyBaMM cell model):
#   PyBaMM sign:      I > 0 = discharge, I < 0 = charge
#   charge_first    : I_py = -I_DC - I_AC sin(ωt)
#   discharge_first : I_py = -I_DC + I_AC sin(ωt)
#
# Frozen hard anchor:
#   charge_first × raw : mean(dt_geom_s) ≈ +335.51 s ± 2.0 s
#
# Diagnostic checks:
#   charge_first × cummax : should reproduce charge_first × raw
#   discharge_first       : expected negative geometry mean; magnitude is diagnostic
#   phase separation      : reported, not symmetry-asserted
#   all dt_geom finite    : True
#   Q reachability        : both arms reach Q_grid_max
#
# Outputs:
#   data/day19A_step2_MJ1_0p3_0p7_10tau_geom_anchor_curves.csv   (320 rows)
#   data/day19A_step2_MJ1_0p3_0p7_10tau_geom_anchor_summary.csv  (4 rows)

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Frozen MJ1 anchor parameters
# ---------------------------------------------------------------------

I_NOM_MJ1_A = 3.4
DC_C = 0.3
AC_C = 0.7

I_DC_A = DC_C * I_NOM_MJ1_A    # 1.02 A
I_AC_A = AC_C * I_NOM_MJ1_A    # 2.38 A

tau_label_s = 11.1
n_tau = 10.0
T_AC_s = 2 * np.pi * n_tau * tau_label_s
f_Hz = 1.0 / T_AC_s

Q_ref_Ah = 3.2593
Q_4p2V_DC_Ah = 2.9628
Q_4p2V_DCAC_10tau_Ah = 2.4902

q_lo_Ah = 0.05 * Q_ref_Ah
q_hi_Ah = min(Q_4p2V_DC_Ah, Q_4p2V_DCAC_10tau_Ah) - 0.02 * Q_ref_Ah

Q_grid_Ah = np.linspace(q_lo_Ah, q_hi_Ah, 80)
Q_grid_mAh = Q_grid_Ah * 1000.0

EXPECTED_MEAN_CHARGE_S = 335.51
TOL_CHARGE_ANCHOR_S = 2.0
TOL_RAW_CUMMAX_PER_Q_S = 1.0

print("=" * 72)
print("Cell 4B — MJ1 0.3+0.7C 10τ waveform-geometry sanity anchor")
print("=" * 72)
print(f"I_nom           : {I_NOM_MJ1_A:.4f} A")
print(f"I_DC            : {I_DC_A:.4f} A  ({DC_C:.1f}C)")
print(f"I_AC            : {I_AC_A:.4f} A  ({AC_C:.1f}C)")
print(f"kappa           : {I_AC_A / I_DC_A:.6f}")
print(f"T_AC            : {T_AC_s:.6f} s")
print(f"f_Hz            : {f_Hz:.9f} Hz  ({f_Hz * 1000:.6f} mHz)")
print(f"Q_ref           : {Q_ref_Ah:.4f} Ah")
print(f"Q window        : [{q_lo_Ah:.6f}, {q_hi_Ah:.6f}] Ah")
print(f"Q grid          : [{Q_grid_mAh[0]:.3f}, {Q_grid_mAh[-1]:.3f}] mAh × {len(Q_grid_mAh)}")
print()

# ---------------------------------------------------------------------
# Run four configurations
# ---------------------------------------------------------------------

configurations = [
    (PHASE_CHARGE_FIRST,    FP_MODE_RAW),
    (PHASE_CHARGE_FIRST,    FP_MODE_CUMMAX),
    (PHASE_DISCHARGE_FIRST, FP_MODE_RAW),
    (PHASE_DISCHARGE_FIRST, FP_MODE_CUMMAX),
]

results = {}

for phase, fp_mode in configurations:
    print(f"Running {phase} × {fp_mode} ...")
    results[(phase, fp_mode)] = dt_geom_from_protocol(
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        f_Hz=f_Hz,
        phase=phase,
        Q_grid_mAh=Q_grid_mAh,
        dt_eval_s=0.1,
        fp_mode=fp_mode,
    )

print("All geometry runs completed.\n")

# ---------------------------------------------------------------------
# Build curves DataFrame
# ---------------------------------------------------------------------

curve_rows = []

for (phase, fp_mode), out in results.items():
    md = out["metadata"]

    for q_mAh, t_dc, t_dcac, dt in zip(
        Q_grid_mAh,
        out["t_DC_s"],
        out["t_DCAC_s"],
        out["dt_geom_s"],
    ):
        curve_rows.append({
            "phase": phase,
            "fp_mode": fp_mode,
            "Q_mAh": float(q_mAh),
            "Q_Ah": float(q_mAh / 1000.0),
            "t_DC_s": float(t_dc),
            "t_DCAC_s": float(t_dcac),
            "dt_geom_s": float(dt),

            # Self-describing protocol metadata
            "I_nom_A": I_NOM_MJ1_A,
            "I_DC_A": I_DC_A,
            "I_AC_A": I_AC_A,
            "DC_C": DC_C,
            "AC_C": AC_C,
            "kappa": md["kappa"],
            "f_Hz": f_Hz,
            "T_AC_s": T_AC_s,
            "tau_label_s": tau_label_s,
            "n_tau": n_tau,
            "dt_eval_s": md["dt_eval_s"],
            "t_max_s": md["t_max_s"],
            "Q_ref_Ah": Q_ref_Ah,
            "Q_4p2V_DC_Ah": Q_4p2V_DC_Ah,
            "Q_4p2V_DCAC_10tau_Ah": Q_4p2V_DCAC_10tau_Ah,
            "q_lo_Ah": q_lo_Ah,
            "q_hi_Ah": q_hi_Ah,
            "Q_DC_final_mAh": md["Q_DC_final_mAh"],
            "Q_DCAC_final_mAh": md["Q_DCAC_final_mAh"],
            "Q_grid_max_mAh": md["Q_grid_max_mAh"],
            "n_t_points": md["n_t_points"],
        })

curves_df = pd.DataFrame(curve_rows)

# ---------------------------------------------------------------------
# Cross-configuration diagnostics
# ---------------------------------------------------------------------

raw_vs_cummax_per_phase = {}

for phase in (PHASE_CHARGE_FIRST, PHASE_DISCHARGE_FIRST):
    diff = (
        results[(phase, FP_MODE_RAW)]["dt_geom_s"]
        - results[(phase, FP_MODE_CUMMAX)]["dt_geom_s"]
    )
    raw_vs_cummax_per_phase[phase] = float(np.nanmax(np.abs(diff)))

mean_cf_raw = float(np.nanmean(results[(PHASE_CHARGE_FIRST, FP_MODE_RAW)]["dt_geom_s"]))
mean_cf_cummax = float(np.nanmean(results[(PHASE_CHARGE_FIRST, FP_MODE_CUMMAX)]["dt_geom_s"]))
mean_df_raw = float(np.nanmean(results[(PHASE_DISCHARGE_FIRST, FP_MODE_RAW)]["dt_geom_s"]))
mean_df_cummax = float(np.nanmean(results[(PHASE_DISCHARGE_FIRST, FP_MODE_CUMMAX)]["dt_geom_s"]))

phase_separation_s = mean_cf_raw - mean_df_raw
phase_sum_diagnostic_s = mean_cf_raw + mean_df_raw

# ---------------------------------------------------------------------
# Build summary DataFrame
# ---------------------------------------------------------------------

# anchor_status:
#   - charge_first rows are checked against the frozen MJ1 +335.51 s mean.
#   - discharge_first rows are diagnostic; expected to be negative for this anchor.
#     They are NOT constrained to be the exact negative mirror of charge_first.

summary_rows = []

for (phase, fp_mode), out in results.items():
    dt = np.asarray(out["dt_geom_s"], dtype=float)
    md = out["metadata"]

    all_finite = bool(np.all(np.isfinite(dt)))

    mean_dt = float(np.nanmean(dt))
    median_dt = float(np.nanmedian(dt))
    min_dt = float(np.nanmin(dt))
    max_dt = float(np.nanmax(dt))

    if phase == PHASE_CHARGE_FIRST:
        expected_mean_s = EXPECTED_MEAN_CHARGE_S
        mean_error_s = mean_dt - expected_mean_s
        anchor_status = (
            "ok"
            if abs(mean_error_s) <= TOL_CHARGE_ANCHOR_S
            else "outside_tolerance"
        )
    else:
        expected_mean_s = np.nan
        mean_error_s = np.nan
        anchor_status = (
            "diagnostic_negative_geometry"
            if mean_dt < 0
            else "diagnostic_unexpected_positive"
        )

    summary_rows.append({
        "phase": phase,
        "fp_mode": fp_mode,
        "n_Q": len(Q_grid_mAh),
        "mean_dt_geom_s": mean_dt,
        "median_dt_geom_s": median_dt,
        "min_dt_geom_s": min_dt,
        "max_dt_geom_s": max_dt,
        "expected_mean_s": expected_mean_s,
        "mean_error_s": mean_error_s,
        "tolerance_s": TOL_CHARGE_ANCHOR_S if phase == PHASE_CHARGE_FIRST else np.nan,
        "anchor_status": anchor_status,
        "all_finite": all_finite,
        "raw_vs_cummax_max_per_Q_diff_s": raw_vs_cummax_per_phase[phase],

        # Diagnostics only, not hard symmetry constraints
        "phase_separation_s": phase_separation_s,
        "phase_sum_diagnostic_s": phase_sum_diagnostic_s,
        "mean_charge_first_raw_s": mean_cf_raw,
        "mean_charge_first_cummax_s": mean_cf_cummax,
        "mean_discharge_first_raw_s": mean_df_raw,
        "mean_discharge_first_cummax_s": mean_df_cummax,

        # Reachability / t_max diagnostics
        "Q_DC_final_mAh": md["Q_DC_final_mAh"],
        "Q_DCAC_final_mAh": md["Q_DCAC_final_mAh"],
        "Q_grid_max_mAh": md["Q_grid_max_mAh"],
        "n_t_points": md["n_t_points"],

        # Self-describing protocol metadata
        "I_nom_A": I_NOM_MJ1_A,
        "I_DC_A": I_DC_A,
        "I_AC_A": I_AC_A,
        "DC_C": DC_C,
        "AC_C": AC_C,
        "kappa": md["kappa"],
        "tau_label_s": tau_label_s,
        "n_tau": n_tau,
        "T_AC_s": T_AC_s,
        "f_Hz": f_Hz,
        "Q_ref_Ah": Q_ref_Ah,
        "Q_4p2V_DC_Ah": Q_4p2V_DC_Ah,
        "Q_4p2V_DCAC_10tau_Ah": Q_4p2V_DCAC_10tau_Ah,
        "q_lo_Ah": q_lo_Ah,
        "q_hi_Ah": q_hi_Ah,
        "q_lo_mAh": q_lo_Ah * 1000.0,
        "q_hi_mAh": q_hi_Ah * 1000.0,
        "dt_eval_s": md["dt_eval_s"],
        "t_max_s": md["t_max_s"],
    })

summary_df = pd.DataFrame(summary_rows)

summary_df["q_reached"] = (
    (summary_df["Q_DC_final_mAh"] >= summary_df["Q_grid_max_mAh"])
    & (summary_df["Q_DCAC_final_mAh"] >= summary_df["Q_grid_max_mAh"])
)

# ---------------------------------------------------------------------
# Persist outputs
# ---------------------------------------------------------------------

out_curves = DATA / "day19A_step2_MJ1_0p3_0p7_10tau_geom_anchor_curves.csv"
out_summary = DATA / "day19A_step2_MJ1_0p3_0p7_10tau_geom_anchor_summary.csv"

curves_df.to_csv(out_curves, index=False)
summary_df.to_csv(out_summary, index=False)

print(f"Wrote: {out_curves}  ({len(curves_df)} rows)")
print(f"Wrote: {out_summary} ({len(summary_df)} rows)")

# ---------------------------------------------------------------------
# Display summary
# ---------------------------------------------------------------------

display_cols = [
    "phase",
    "fp_mode",
    "n_Q",
    "mean_dt_geom_s",
    "median_dt_geom_s",
    "expected_mean_s",
    "mean_error_s",
    "tolerance_s",
    "anchor_status",
    "all_finite",
    "q_reached",
    "raw_vs_cummax_max_per_Q_diff_s",
    "phase_separation_s",
    "phase_sum_diagnostic_s",
]

display(summary_df[display_cols])

print("\nSanity checks:")
for _, r in summary_df.iterrows():
    print(
        f"  {r['phase']:<16} × {r['fp_mode']:<6}  "
        f"mean = {r['mean_dt_geom_s']:+9.4f} s, "
        f"err = {r['mean_error_s'] if np.isfinite(r['mean_error_s']) else np.nan:+8.4f} s, "
        f"raw_vs_cummax max = {r['raw_vs_cummax_max_per_Q_diff_s']:.6f} s, "
        f"all_finite = {r['all_finite']}, "
        f"q_reached = {r['q_reached']}, "
        f"status = {r['anchor_status']}"
    )

print(f"\nphase_separation_s = mean(cf_raw) - mean(df_raw) = {phase_separation_s:+.6f} s")
print(f"phase_sum_diagnostic_s = mean(cf_raw) + mean(df_raw) = {phase_sum_diagnostic_s:+.6f} s")
print(
    "Note: phase_sum_diagnostic_s is reported only. "
    "Discharge-first is not expected to be the exact negative mirror of charge-first."
)

# ---------------------------------------------------------------------
# Hard-fail assertions
# ---------------------------------------------------------------------

# Charge-first is the frozen MJ1 experimental-faithful anchor.
cf_rows = summary_df[summary_df["phase"].eq(PHASE_CHARGE_FIRST)]
assert (cf_rows["anchor_status"] == "ok").all(), (
    "Charge-first anchor failed tolerance:\n"
    f"{cf_rows[['phase', 'fp_mode', 'mean_dt_geom_s', 'mean_error_s', 'anchor_status']]}"
)

# Discharge-first is diagnostic: require negative geometry mean, not mirrored magnitude.
df_rows = summary_df[summary_df["phase"].eq(PHASE_DISCHARGE_FIRST)]
assert (df_rows["mean_dt_geom_s"] < 0).all(), (
    "Discharge-first branch should produce negative geometry mean for this anchor:\n"
    f"{df_rows[['phase', 'fp_mode', 'mean_dt_geom_s', 'anchor_status']]}"
)

max_raw_cummax = float(summary_df["raw_vs_cummax_max_per_Q_diff_s"].max())
assert max_raw_cummax < TOL_RAW_CUMMAX_PER_Q_S, (
    f"raw vs cummax disagree by {max_raw_cummax:.6f} s, "
    f"exceeds {TOL_RAW_CUMMAX_PER_Q_S} s"
)

# q_reached gives the root-cause diagnostic for insufficient t_max.
# It is intentionally checked before all_finite.
assert summary_df["q_reached"].all(), (
    "At least one configuration does not reach Q_grid_max — t_max insufficient:\n"
    f"{summary_df[['phase', 'fp_mode', 'Q_DC_final_mAh', 'Q_DCAC_final_mAh', 'Q_grid_max_mAh', 'q_reached']]}"
)

assert summary_df["all_finite"].all(), (
    "Some configurations have non-finite dt_geom:\n"
    f"{summary_df[['phase', 'fp_mode', 'all_finite']]}"
)

# Session object for downstream cells
mj1_anchor_results = {
    "results": results,
    "curves_df": curves_df,
    "summary_df": summary_df,
    "cross_checks": {
        "mean_charge_first_raw_s": mean_cf_raw,
        "mean_charge_first_cummax_s": mean_cf_cummax,
        "mean_discharge_first_raw_s": mean_df_raw,
        "mean_discharge_first_cummax_s": mean_df_cummax,
        "phase_separation_s": phase_separation_s,
        "phase_sum_diagnostic_s": phase_sum_diagnostic_s,
        "raw_vs_cummax_max_per_Q_diff_s": max_raw_cummax,
    },
}

print("\n" + "=" * 72)
print("Cell 4B PASSED — Cell 4A geometry helpers verified vs MJ1 charge-first anchor")
print("=" * 72)

Cell 4B — MJ1 0.3+0.7C 10τ waveform-geometry sanity anchor
I_nom           : 3.4000 A
I_DC            : 1.0200 A  (0.3C)
I_AC            : 2.3800 A  (0.7C)
kappa           : 2.333333
T_AC            : 697.433569 s
f_Hz            : 0.001433828 Hz  (1.433828 mHz)
Q_ref           : 3.2593 Ah
Q window        : [0.162965, 2.425014] Ah
Q grid          : [162.965, 2425.014] mAh × 80

Running charge_first × raw ...
Running charge_first × cummax ...
Running discharge_first × raw ...
Running discharge_first × cummax ...
All geometry runs completed.

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step2_MJ1_0p3_0p7_10tau_geom_anchor_curves.csv  (320 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step2_MJ1_0p3_0p7_10tau_geom_anchor_summary.csv (4 rows)


,phase,fp_mode,n_Q,mean_dt_geom_s,median_dt_geom_s,expected_mean_s,mean_error_s,tolerance_s,anchor_status,all_finite,q_reached,raw_vs_cummax_max_per_Q_diff_s,phase_separation_s,phase_sum_diagnostic_s
0,charge_first,raw,80,337.503105,353.247007,335.51,1.993105,2.0,ok,True,True,0.0,526.500741,148.505469
1,charge_first,cummax,80,337.503105,353.247007,335.51,1.993105,2.0,ok,True,True,0.0,526.500741,148.505469
2,discharge_first,raw,80,-188.997636,-180.461769,NaN,NaN,NaN,diagnostic_negative_geometry,True,True,0.0,526.500741,148.505469
3,discharge_first,cummax,80,-188.997636,-180.461769,NaN,NaN,NaN,diagnostic_negative_geometry,True,True,0.0,526.500741,148.505469



Sanity checks:
  charge_first     × raw     mean = +337.5031 s, err =  +1.9931 s, raw_vs_cummax max = 0.000000 s, all_finite = True, q_reached = True, status = ok
  charge_first     × cummax  mean = +337.5031 s, err =  +1.9931 s, raw_vs_cummax max = 0.000000 s, all_finite = True, q_reached = True, status = ok
  discharge_first  × raw     mean = -188.9976 s, err =     +nan s, raw_vs_cummax max = 0.000000 s, all_finite = True, q_reached = True, status = diagnostic_negative_geometry
  discharge_first  × cummax  mean = -188.9976 s, err =     +nan s, raw_vs_cummax max = 0.000000 s, all_finite = True, q_reached = True, status = diagnostic_negative_geometry

phase_separation_s = mean(cf_raw) - mean(df_raw) = +526.500741 s
phase_sum_diagnostic_s = mean(cf_raw) + mean(df_raw) = +148.505469 s
Note: phase_sum_diagnostic_s is reported only. Discharge-first is not expected to be the exact negative mirror of charge-first.

Cell 4B PASSED — Cell 4A geometry helpers verified vs MJ1 charge-first ancho

In [23]:
# Cell 4C — Trajectory first-passage helpers with keep-last de-duplication
#
# Purpose:
#   Provide trajectory-level helpers for §2.3 retro audit. These functions
#   operate on stored PyBaMM JSON / NPZ trajectories and enforce the Day 19A
#   raw strict-net convention:
#
#       Q_mAh(t) = -∫ I(t) dt × 1000/3600
#       Δt(Q*)   = t_DC(Q*) − t_DCAC(Q*)
#
#   No cummax is applied by default. Duplicate timestamps are repaired by
#   keep-last de-duplication, as justified by Cell 3.9:
#       5/5 time-axis anomalies = duplicate_timestamps_only
#       0 negative jumps
#       0 NaN time values
#
# Depends on Cell 4A:
#   compute_Q_net_mAh()
#   dt_from_trajectories()
#   FP_MODE_RAW / FP_MODE_CUMMAX

import json as json_module
import numpy as np
import pandas as pd
from pathlib import Path


# -----------------------------------------------------------------------------
# 1. clean_time_series_keep_last
# -----------------------------------------------------------------------------
def clean_time_series_keep_last(t, *arrays, eps_t=1e-12, allow_sort=False):
    """
    Clean a trajectory time axis by keeping the last sample at duplicate timestamps.

    Intended use:
        Repair CC→CV stitching duplicates observed in legacy Day 8/9 JSON files.

    Rules:
        - t must be finite.
        - If negative time jumps exist, raise by default.
        - Consecutive duplicate timestamps within eps_t are collapsed.
        - For each duplicate group, keep the last sample.
        - All companion arrays are cleaned with the same retained indices.

    Parameters
    ----------
    t : array-like
        Time axis in seconds.
    *arrays : array-like
        Companion arrays with same length as t, e.g. I(t), Q(t), V(t).
    eps_t : float
        Tolerance for identifying duplicate timestamps.
    allow_sort : bool
        If False, negative jumps raise ValueError.
        If True, sort by t before de-duplication. Default False for audit safety.

    Returns
    -------
    result : dict
        {
            "t": cleaned time array,
            "arrays": list of cleaned companion arrays,
            "kept_idx": original retained indices,
            "n_original": int,
            "n_clean": int,
            "n_removed": int,
            "n_duplicate_groups": int,
            "had_negative_jumps": bool,
        }
    """
    t = np.asarray(t, dtype=float)

    if t.ndim != 1:
        raise ValueError(f"t must be 1D; got shape {t.shape}")
    if not np.all(np.isfinite(t)):
        raise ValueError("t contains NaN or inf.")

    companion = [np.asarray(a, dtype=float) for a in arrays]
    for j, a in enumerate(companion):
        if a.shape != t.shape:
            raise ValueError(f"array[{j}] shape mismatch: {a.shape} vs {t.shape}")
        if not np.all(np.isfinite(a)):
            raise ValueError(f"array[{j}] contains NaN or inf.")

    if len(t) <= 1:
        return {
            "t": t.copy(),
            "arrays": [a.copy() for a in companion],
            "kept_idx": np.arange(len(t), dtype=int),
            "n_original": len(t),
            "n_clean": len(t),
            "n_removed": 0,
            "n_duplicate_groups": 0,
            "had_negative_jumps": False,
        }

    dt = np.diff(t)
    had_negative_jumps = bool(np.any(dt < -eps_t))

    if had_negative_jumps and not allow_sort:
        raise ValueError(
            "Negative time jump detected. Do not silently sort audit trajectories; "
            "inspect owning notebook or exclude record."
        )

    if allow_sort:
        order = np.argsort(t, kind="stable")
        t_work = t[order]
        arrays_work = [a[order] for a in companion]
        original_idx_work = order
    else:
        t_work = t
        arrays_work = companion
        original_idx_work = np.arange(len(t), dtype=int)

    # keep last sample in each consecutive duplicate-time group
    kept = []
    duplicate_groups = 0
    group_start = 0

    for i in range(1, len(t_work)):
        if abs(t_work[i] - t_work[i - 1]) > eps_t:
            # group [group_start, i-1], keep last
            kept.append(i - 1)
            if i - group_start > 1:
                duplicate_groups += 1
            group_start = i

    # final group
    kept.append(len(t_work) - 1)
    if len(t_work) - group_start > 1:
        duplicate_groups += 1

    kept = np.asarray(kept, dtype=int)
    t_clean = t_work[kept]
    arrays_clean = [a[kept] for a in arrays_work]
    kept_original_idx = original_idx_work[kept]

    if len(t_clean) > 1 and not np.all(np.diff(t_clean) > 0):
        raise RuntimeError("Internal error: cleaned t is still not strictly increasing.")

    return {
        "t": t_clean,
        "arrays": arrays_clean,
        "kept_idx": kept_original_idx,
        "n_original": len(t),
        "n_clean": len(t_clean),
        "n_removed": int(len(t) - len(t_clean)),
        "n_duplicate_groups": int(duplicate_groups),
        "had_negative_jumps": had_negative_jumps,
    }


# -----------------------------------------------------------------------------
# 2. load_legacy_json_records
# -----------------------------------------------------------------------------
def load_legacy_json_records(path):
    """
    Load a legacy results_day*.json file.

    Expected schema:
        top-level list of dict records.
    """
    path = Path(path)
    with open(path, encoding="utf-8") as fh:
        records = json_module.load(fh)

    if not isinstance(records, list):
        raise ValueError(f"Expected top-level list in {path.name}; got {type(records).__name__}")

    return records


# -----------------------------------------------------------------------------
# 3. extract_legacy_record_trajectory
# -----------------------------------------------------------------------------
def extract_legacy_record_trajectory(
    record,
    clean=True,
    eps_t=1e-12,
    q_match_tol_mAh=1e-9,
):
    """
    Extract t_chg, I_chg, and Q_net_trajectory from one legacy JSON record.

    By default:
        - keep-last de-duplicates repeated timestamps
        - recomputes Q_mAh from I_chg
        - compares recomputed Q against stored Q_net_trajectory

    Parameters
    ----------
    record : dict
        One JSON record.
    clean : bool
        Whether to apply keep-last de-duplication.
    eps_t : float
        Timestamp duplicate tolerance.
    q_match_tol_mAh : float
        Warning threshold for stored-vs-recomputed Q mismatch.

    Returns
    -------
    out : dict
        {
            "t_s": cleaned time,
            "I_A": cleaned current,
            "Q_stored_mAh": cleaned stored Q trajectory,
            "Q_recomputed_mAh": recomputed strict-net Q trajectory,
            "q_match_max_abs_mAh": float,
            "q_match_rmse_mAh": float,
            "cleaning": {...},
            "metadata": {...},
        }
    """
    if record.get("status") != "ok":
        raise ValueError(f"Record status is not ok: {record.get('status')}")

    if "t_chg" not in record or "I_chg" not in record:
        raise ValueError("Record lacks t_chg or I_chg.")
    if "Q_net_trajectory" not in record:
        raise ValueError("Record lacks Q_net_trajectory.")

    t = np.asarray(record["t_chg"], dtype=float)
    I = np.asarray(record["I_chg"], dtype=float)
    Q_stored = np.asarray(record["Q_net_trajectory"], dtype=float)

    if not (t.shape == I.shape == Q_stored.shape):
        raise ValueError(
            f"t/I/Q shape mismatch: {t.shape}, {I.shape}, {Q_stored.shape}"
        )

    if clean:
        cleaned = clean_time_series_keep_last(t, I, Q_stored, eps_t=eps_t)
        t_clean = cleaned["t"]
        I_clean, Q_stored_clean = cleaned["arrays"]
    else:
        cleaned = {
            "n_original": len(t),
            "n_clean": len(t),
            "n_removed": 0,
            "n_duplicate_groups": 0,
            "had_negative_jumps": bool(np.any(np.diff(t) < -eps_t)) if len(t) > 1 else False,
        }
        t_clean = t
        I_clean = I
        Q_stored_clean = Q_stored

    Q_recomputed = compute_Q_net_mAh(t_clean, I_clean)

    # If duplicates were removed, stored Q and recomputed Q can still match exactly
    # because the kept samples are selected consistently.
    diff = Q_recomputed - Q_stored_clean
    q_match_max_abs = float(np.nanmax(np.abs(diff))) if len(diff) else np.nan
    q_match_rmse = float(np.sqrt(np.nanmean(diff * diff))) if len(diff) else np.nan

    if np.isfinite(q_match_max_abs) and q_match_max_abs > q_match_tol_mAh:
        # Do not raise. Stored Q is diagnostic; recomputed Q is authoritative for Day 19A.
        pass

    metadata = {
        "case_id": record.get("case_id"),
        "condition": record.get("condition"),
        "status": record.get("status"),
        "I_DC_Crate": record.get("I_DC_Crate"),
        "A_Crate": record.get("A_Crate"),
        "f_Hz": record.get("f_Hz"),
        "kappa": record.get("kappa"),
        "tau_label": record.get("tau_label"),
        "CC_time_s": record.get("CC_time_s"),
        "CV_time_s": record.get("CV_time_s"),
        "total_time_s": record.get("total_time_s"),
        "Q_net_final_mAh_reported": record.get("Q_net_final_mAh"),
    }

    return {
        "t_s": t_clean,
        "I_A": I_clean,
        "Q_stored_mAh": Q_stored_clean,
        "Q_recomputed_mAh": Q_recomputed,
        "q_match_max_abs_mAh": q_match_max_abs,
        "q_match_rmse_mAh": q_match_rmse,
        "cleaning": cleaned,
        "metadata": metadata,
    }


# -----------------------------------------------------------------------------
# 4. record helpers
# -----------------------------------------------------------------------------
def is_dc_baseline_record(record):
    """
    Identify DC baseline records in legacy JSON.
    """
    case_id = str(record.get("case_id", "")).lower()
    condition = str(record.get("condition", "")).lower()
    A = record.get("A_Crate", None)
    f = record.get("f_Hz", None)

    if "baseline" in case_id or "baseline" in condition or "_dc" in condition:
        return True

    try:
        if float(A) == 0.0:
            return True
    except Exception:
        pass

    try:
        if float(f) == 0.0 and float(A or 0.0) == 0.0:
            return True
    except Exception:
        pass

    return False


def find_dc_baseline_for_record(records, dc_c, prefer_status_ok=True):
    """
    Find a DC baseline record matching I_DC_Crate == dc_c.
    """
    candidates = []
    for idx, rec in enumerate(records):
        if prefer_status_ok and rec.get("status") != "ok":
            continue
        if not is_dc_baseline_record(rec):
            continue

        try:
            rec_dc = float(rec.get("I_DC_Crate"))
        except Exception:
            continue

        if np.isclose(rec_dc, float(dc_c), atol=1e-9):
            candidates.append((idx, rec))

    if not candidates:
        raise LookupError(f"No DC baseline record found for I_DC_Crate={dc_c}")

    # If duplicates exist, return first deterministic match.
    return candidates[0]


def find_record_by_case_id(records, case_id, status_ok=True):
    """
    Find a record by exact case_id.
    """
    matches = []
    for idx, rec in enumerate(records):
        if status_ok and rec.get("status") != "ok":
            continue
        if rec.get("case_id") == case_id:
            matches.append((idx, rec))

    if not matches:
        raise LookupError(f"No record found for case_id={case_id!r}")

    return matches[0]


# -----------------------------------------------------------------------------
# 5. dt_from_legacy_records
# -----------------------------------------------------------------------------
def dt_from_legacy_records(
    dc_record,
    dcac_record,
    Q_grid_mAh,
    fp_mode=FP_MODE_RAW,
    clean=True,
    eps_t=1e-12,
):
    """
    Compute Δt_model(Q) from two legacy JSON records.

    Uses recomputed strict-net Q from t_chg/I_chg, not stored Q, although
    stored Q is validated and returned diagnostically.

    Parameters
    ----------
    dc_record : dict
        DC baseline record.
    dcac_record : dict
        DCAC record.
    Q_grid_mAh : array-like
        Shared Q grid.
    fp_mode : str
        raw or cummax.
    clean : bool
        Apply keep-last de-duplication to both arms.

    Returns
    -------
    out : dict
        dt_from_trajectories output plus metadata.
    """
    dc = extract_legacy_record_trajectory(dc_record, clean=clean, eps_t=eps_t)
    dcac = extract_legacy_record_trajectory(dcac_record, clean=clean, eps_t=eps_t)

    out = dt_from_trajectories(
        t_DC=dc["t_s"],
        I_DC_A=dc["I_A"],
        t_DCAC=dcac["t_s"],
        I_DCAC_A=dcac["I_A"],
        Q_grid_mAh=Q_grid_mAh,
        mode=fp_mode,
    )

    out["metadata"] = {
        "fp_mode": fp_mode,
        "clean": clean,
        "eps_t": eps_t,
        "dc_case_id": dc["metadata"]["case_id"],
        "dcac_case_id": dcac["metadata"]["case_id"],
        "dc_condition": dc["metadata"]["condition"],
        "dcac_condition": dcac["metadata"]["condition"],
        "dc_I_DC_Crate": dc["metadata"]["I_DC_Crate"],
        "dcac_I_DC_Crate": dcac["metadata"]["I_DC_Crate"],
        "dcac_A_Crate": dcac["metadata"]["A_Crate"],
        "dcac_f_Hz": dcac["metadata"]["f_Hz"],
        "dcac_kappa": dcac["metadata"]["kappa"],
        "dcac_tau_label": dcac["metadata"]["tau_label"],
        "dc_cleaning": dc["cleaning"],
        "dcac_cleaning": dcac["cleaning"],
        "dc_q_match_max_abs_mAh": dc["q_match_max_abs_mAh"],
        "dcac_q_match_max_abs_mAh": dcac["q_match_max_abs_mAh"],
    }

    return out


# -----------------------------------------------------------------------------
# 6. self-test on known anomaly records from Cell 3.9
# -----------------------------------------------------------------------------
anom_path = DATA / "day19A_step1_t_axis_anomaly_diagnostic.csv"

selftest_rows = []

if anom_path.exists():
    anom_df = pd.read_csv(anom_path)

    for _, row in anom_df.iterrows():
        src = REPO / row["file"]
        records = load_legacy_json_records(src)
        rec = records[int(row["record_idx"])]

        tr = extract_legacy_record_trajectory(rec, clean=True)

        selftest_rows.append({
            "file": row["file"],
            "record_idx": int(row["record_idx"]),
            "case_id": rec.get("case_id"),
            "condition": rec.get("condition"),
            "n_original": tr["cleaning"]["n_original"],
            "n_clean": tr["cleaning"]["n_clean"],
            "n_removed": tr["cleaning"]["n_removed"],
            "n_duplicate_groups": tr["cleaning"]["n_duplicate_groups"],
            "had_negative_jumps": tr["cleaning"]["had_negative_jumps"],
            "q_match_max_abs_mAh": tr["q_match_max_abs_mAh"],
            "q_match_rmse_mAh": tr["q_match_rmse_mAh"],
            "t_strictly_increasing_after_clean": bool(np.all(np.diff(tr["t_s"]) > 0)),
        })

selftest_df = pd.DataFrame(selftest_rows)

out_selftest = DATA / "day19A_step2_trajectory_helper_dedup_selftest.csv"
selftest_df.to_csv(out_selftest, index=False)

print("Trajectory helper library loaded.")
print("Functions defined:")
for fn in (
    clean_time_series_keep_last,
    load_legacy_json_records,
    extract_legacy_record_trajectory,
    is_dc_baseline_record,
    find_dc_baseline_for_record,
    find_record_by_case_id,
    dt_from_legacy_records,
):
    print(f"  · {fn.__name__}")

print(f"\nWrote: {out_selftest}")

if len(selftest_df):
    display(selftest_df)
    assert (selftest_df["t_strictly_increasing_after_clean"]).all(), \
        "Dedup self-test failed: cleaned t is not strictly increasing."
    assert (selftest_df["n_removed"] == 1).all(), \
        "Expected one duplicate sample removed for each known anomaly record."
    assert (selftest_df["had_negative_jumps"] == False).all(), \
        "Unexpected negative time jump in known anomaly records."
    print("\nCell 4C self-test PASSED — all known duplicate-timestamp anomalies repairable.")
else:
    print("\nNo anomaly records found for self-test; helper library loaded only.")

Trajectory helper library loaded.
Functions defined:
  · clean_time_series_keep_last
  · load_legacy_json_records
  · extract_legacy_record_trajectory
  · is_dc_baseline_record
  · find_dc_baseline_for_record
  · find_record_by_case_id
  · dt_from_legacy_records

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step2_trajectory_helper_dedup_selftest.csv


,file,record_idx,case_id,condition,n_original,n_clean,n_removed,n_duplicate_groups,had_negative_jumps,q_match_max_abs_mAh,q_match_rmse_mAh,t_strictly_increasing_after_clean
0,data/results_day8_x2_dfn_mj1freq_v2.json,4,DC0.20C+AC0.30C_f0.00041Hz,0.2+0.3C 34.8τ,444,443,1,1,False,0.000002,7.533084e-07,True
1,data/results_day8_x2_dfn_mj1freq_v2.json,25,DC0.20C_baseline,0.2C_DC,236,235,1,1,False,0.000029,1.208726e-05,True
2,data/results_day8_x4_dfn_chenfreq.json,25,DC0.20C_baseline,0.2C_DC,236,235,1,1,False,0.000029,1.208726e-05,True
3,data/results_day8_x4_dfn_chenfreq.json,34,DC0.20C+AC0.30C_f0.00080Hz,0.2+0.3C 10τ,814,813,1,1,False,0.000038,9.965614e-06,True
4,data/results_day8_x4_dfn_chenfreq.json,56,DC0.20C_baseline,0.2C_DC,236,235,1,1,False,0.000029,1.208726e-05,True



Cell 4C self-test PASSED — all known duplicate-timestamp anomalies repairable.


In [24]:
# Cell 4D — Validate geometry helper against stored Day18B / Day18 v4 geometry columns
#
# Purpose:
#   Verify that Cell 4A's dt_geom_from_protocol() reproduces stored geometry
#   columns in independent historical outputs:
#
#   1) Day18B smoke:
#        data/day18B_smoke_dtQ_resid_curves_long.csv.gz
#        stored column: dt_geom_s
#
#   2) Day18 v4 charge-first:
#        data/day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz
#        stored column: dtQ_geom_s
#
# Outputs:
#   data/day19A_step2_geometry_helper_validation_long.csv
#   data/day19A_step2_geometry_helper_validation_summary.csv
#
# Critical conventions:
#   - Uses PyBaMM parameter-set nominal capacity for C-rate scaling.
#   - No hardcoded Q_nom fallback: capacity lookup must succeed.
#   - Uses phase_label if present; Day18B smoke is treated as charge_first.
#   - Uses f = 1 / (2π · n_tau · tau_label_s), tau_label_s = 11.1 s.
#   - Validates helper-recomputed geometry against stored geometry.

import re
import numpy as np
import pandas as pd
from pathlib import Path

try:
    import pybamm
except Exception as e:
    pybamm = None
    print(f"[WARN] pybamm import failed: {e}")


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

TAU_LABEL_DEFAULT_S = 11.1
DT_EVAL_GEOM_S = 0.1

# First-pass tolerance: initial smoke threshold.
# If global_max_abs_err_s is << 0.1 s, tighten after first run.
TOL_GROUP_MAX_ABS_S = 0.10
TOL_GLOBAL_MAX_ABS_S = 0.10

day18b_path = DATA / "day18B_smoke_dtQ_resid_curves_long.csv.gz"
day18v4_path = DATA / "day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz"

assert day18b_path.exists(), f"Missing: {day18b_path}"
assert day18v4_path.exists(), f"Missing: {day18v4_path}"


# ---------------------------------------------------------------------
# Helper: parse protocol labels and parameter capacities
# ---------------------------------------------------------------------

def _p_to_float(s):
    """Convert strings like '0p2', '10', '1p67', '0.1' to float."""
    return float(str(s).replace("p", "."))


def parse_dcac_protocol_label(label):
    """
    Parse labels such as:
        DCAC_DC0p2C_AC0p4C_10tau
        DCAC_DC0p2C_AC0p4C_0p1tau
        DC0p2C_AC0p4C_10tau
        DC0.2C_AC0.4C_10tau

    Returns
    -------
    dict with DC_C, AC_C, n_tau
    """
    s = str(label)

    # Strip optional 'DCAC_' prefix
    if s.upper().startswith("DCAC_"):
        s = s[5:]

    pat = re.compile(
        r"DC(?P<dc>[0-9p.]+)C[_+]?AC(?P<ac>[0-9p.]+)C[_-]?(?P<tau>[0-9p.]+)tau",
        re.IGNORECASE,
    )
    m = pat.search(s)
    if not m:
        raise ValueError(f"Cannot parse DCAC protocol label: {label!r}")

    return {
        "DC_C": _p_to_float(m.group("dc")),
        "AC_C": _p_to_float(m.group("ac")),
        "n_tau": _p_to_float(m.group("tau")),
    }


def get_nominal_capacity_Ah(param_set):
    """
    Get nominal cell capacity from PyBaMM parameter set.

    No hardcoded fallbacks are allowed here. Wrong Q_nom silently rescales
    I_DC/I_AC and can create systematic false validation failures.
    """
    if pybamm is None:
        raise RuntimeError(
            f"PyBaMM unavailable; cannot look up Q_nom for {param_set!r}. "
            "Hardcoded fallbacks are intentionally disabled."
        )

    pv = pybamm.ParameterValues(str(param_set))
    return float(pv["Nominal cell capacity [A.h]"])


def n_tau_to_f_Hz(n_tau, tau_label_s=TAU_LABEL_DEFAULT_S):
    return 1.0 / (2.0 * np.pi * float(n_tau) * float(tau_label_s))


def resolve_n_tau(group_df, param_set, protocol_label, n_tau_from_label):
    """
    Resolve n_tau from group column if present, but require consistency
    with protocol-label parsing.
    """
    if "n_tau" not in group_df.columns:
        return float(n_tau_from_label)

    n_tau_vals = (
        pd.to_numeric(group_df["n_tau"], errors="coerce")
        .dropna()
        .unique()
    )

    if len(n_tau_vals) > 1:
        raise ValueError(
            f"Inconsistent n_tau values in group ({param_set}, {protocol_label}): "
            f"{n_tau_vals.tolist()} — group key may be missing an axis."
        )

    if len(n_tau_vals) == 1:
        n_tau_col = float(n_tau_vals[0])

        if abs(n_tau_col - float(n_tau_from_label)) > 1e-9:
            raise ValueError(
                f"n_tau column ({n_tau_col}) != label parse ({n_tau_from_label}) "
                f"for group ({param_set}, {protocol_label})."
            )

        return n_tau_col

    return float(n_tau_from_label)


def recompute_geom_for_group(
    group_df,
    dataset_name,
    param_set,
    protocol_label,
    Q_col_Ah,
    stored_geom_col,
    phase,
    tau_label_s=TAU_LABEL_DEFAULT_S,
):
    """
    Recompute dt_geom_s for one group with shared protocol metadata.
    """
    parsed = parse_dcac_protocol_label(protocol_label)

    DC_C = parsed["DC_C"]
    AC_C = parsed["AC_C"]
    n_tau_from_label = parsed["n_tau"]
    n_tau = resolve_n_tau(group_df, param_set, protocol_label, n_tau_from_label)

    # Optional NaN guard with diagnostic row filtering.
    q_numeric = pd.to_numeric(group_df[Q_col_Ah], errors="coerce")
    stored_numeric = pd.to_numeric(group_df[stored_geom_col], errors="coerce")
    mask_valid = q_numeric.notna() & stored_numeric.notna()

    if not mask_valid.all():
        n_drop = len(group_df) - int(mask_valid.sum())
        print(
            f"[WARN] {dataset_name} {param_set} {protocol_label}: "
            f"dropping {n_drop}/{len(group_df)} rows with NaN in "
            f"{Q_col_Ah} or {stored_geom_col}"
        )
        group_df = group_df.loc[mask_valid].reset_index(drop=True)
        q_numeric = q_numeric.loc[mask_valid].reset_index(drop=True)
        stored_numeric = stored_numeric.loc[mask_valid].reset_index(drop=True)

    if len(group_df) == 0:
        raise ValueError(
            f"Group became empty after NaN filtering: "
            f"{dataset_name}, {param_set}, {protocol_label}"
        )

    Q_nom_Ah = get_nominal_capacity_Ah(param_set)
    I_DC_A = DC_C * Q_nom_Ah
    I_AC_A = AC_C * Q_nom_Ah
    f_Hz = n_tau_to_f_Hz(n_tau, tau_label_s=tau_label_s)

    q_grid_mAh = q_numeric.to_numpy(dtype=float) * 1000.0

    out = dt_geom_from_protocol(
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        f_Hz=f_Hz,
        phase=phase,
        Q_grid_mAh=q_grid_mAh,
        dt_eval_s=DT_EVAL_GEOM_S,
        fp_mode=FP_MODE_RAW,
    )

    stored = stored_numeric.to_numpy(dtype=float)
    recomputed = np.asarray(out["dt_geom_s"], dtype=float)
    err = recomputed - stored

    result = group_df.copy()
    result["dataset"] = dataset_name
    result["protocol_label_for_parse"] = protocol_label
    result["phase_used_for_recompute"] = phase
    result["stored_geom_col"] = stored_geom_col
    result["dt_geom_recomputed_s"] = recomputed
    result["dt_geom_stored_s"] = stored
    result["geom_recompute_err_s"] = err

    # Metadata
    result["parsed_DC_C"] = DC_C
    result["parsed_AC_C"] = AC_C
    result["parsed_n_tau"] = n_tau
    result["tau_label_s_used"] = tau_label_s
    result["f_Hz_recomputed"] = f_Hz
    result["Q_nom_Ah_used"] = Q_nom_Ah
    result["I_DC_A_used"] = I_DC_A
    result["I_AC_A_used"] = I_AC_A
    result["kappa_used"] = I_AC_A / I_DC_A
    result["dt_eval_s_used"] = DT_EVAL_GEOM_S
    result["t_max_s_used"] = out["metadata"]["t_max_s"]
    result["Q_DC_final_mAh_geom"] = out["metadata"]["Q_DC_final_mAh"]
    result["Q_DCAC_final_mAh_geom"] = out["metadata"]["Q_DCAC_final_mAh"]

    return result


# ---------------------------------------------------------------------
# Part A — Day18B smoke validation
# ---------------------------------------------------------------------

day18b = pd.read_csv(day18b_path, compression="gzip")
required18b = {"param_set", "protocol_label", "Q_Ah", "dt_geom_s"}
missing18b = required18b - set(day18b.columns)
assert not missing18b, f"Day18B missing columns: {missing18b}"

day18b_rows = []

for (param_set, protocol_label), g in day18b.groupby(
    ["param_set", "protocol_label"], sort=False
):
    print(f"[Day18B] recompute {param_set} | {protocol_label} | n={len(g)}")

    gg = recompute_geom_for_group(
        group_df=g,
        dataset_name="day18B_smoke",
        param_set=param_set,
        protocol_label=protocol_label,
        Q_col_Ah="Q_Ah",
        stored_geom_col="dt_geom_s",
        phase=PHASE_CHARGE_FIRST,
        tau_label_s=TAU_LABEL_DEFAULT_S,
    )
    day18b_rows.append(gg)

day18b_cmp = pd.concat(day18b_rows, ignore_index=True)


# ---------------------------------------------------------------------
# Part B — Day18 v4 charge-first validation
# ---------------------------------------------------------------------

day18v4 = pd.read_csv(day18v4_path, compression="gzip")
required18v4 = {"param_set", "protocol_DCAC", "Q_Ah", "dtQ_geom_s", "phase_label"}
missing18v4 = required18v4 - set(day18v4.columns)
assert not missing18v4, f"Day18 v4 missing columns: {missing18v4}"

day18v4_rows = []

for (param_set, protocol_label, phase_label), g in day18v4.groupby(
    ["param_set", "protocol_DCAC", "phase_label"], sort=False
):
    if str(phase_label) != PHASE_CHARGE_FIRST:
        raise ValueError(f"Unexpected Day18 v4 phase_label: {phase_label}")

    print(f"[Day18 v4] recompute {param_set} | {protocol_label} | {phase_label} | n={len(g)}")

    gg = recompute_geom_for_group(
        group_df=g,
        dataset_name="day18_v4_charge_first",
        param_set=param_set,
        protocol_label=protocol_label,
        Q_col_Ah="Q_Ah",
        stored_geom_col="dtQ_geom_s",
        phase=PHASE_CHARGE_FIRST,
        tau_label_s=TAU_LABEL_DEFAULT_S,
    )
    day18v4_rows.append(gg)

day18v4_cmp = pd.concat(day18v4_rows, ignore_index=True)


# ---------------------------------------------------------------------
# Combine + summarize
# ---------------------------------------------------------------------

cmp_df = pd.concat([day18b_cmp, day18v4_cmp], ignore_index=True)
cmp_df["abs_err_s"] = cmp_df["geom_recompute_err_s"].abs()

summary = (
    cmp_df
    .groupby(
        ["dataset", "param_set", "protocol_label_for_parse", "phase_used_for_recompute"],
        dropna=False,
    )
    .agg(
        n=("geom_recompute_err_s", "size"),
        max_abs_err_s=("abs_err_s", "max"),
        median_abs_err_s=("abs_err_s", "median"),
        mean_err_s=("geom_recompute_err_s", "mean"),
        stored_geom_median_s=("dt_geom_stored_s", "median"),
        recomputed_geom_median_s=("dt_geom_recomputed_s", "median"),
        Q_nom_Ah_used=("Q_nom_Ah_used", "first"),
        I_DC_A_used=("I_DC_A_used", "first"),
        I_AC_A_used=("I_AC_A_used", "first"),
        f_Hz_recomputed=("f_Hz_recomputed", "first"),
        parsed_n_tau=("parsed_n_tau", "first"),
        dt_eval_s_used=("dt_eval_s_used", "first"),
    )
    .reset_index()
)

summary["validation_status"] = np.where(
    summary["max_abs_err_s"] <= TOL_GROUP_MAX_ABS_S,
    "ok",
    "outside_tolerance",
)

global_summary = pd.DataFrame([{
    "n_rows": len(cmp_df),
    "n_groups": len(summary),
    "global_max_abs_err_s": float(cmp_df["abs_err_s"].max()),
    "global_median_abs_err_s": float(cmp_df["abs_err_s"].median()),
    "global_mean_err_s": float(cmp_df["geom_recompute_err_s"].mean()),
    "global_status": (
        "ok"
        if float(cmp_df["abs_err_s"].max()) <= TOL_GLOBAL_MAX_ABS_S
        else "outside_tolerance"
    ),
    "tol_global_max_abs_s": TOL_GLOBAL_MAX_ABS_S,
    "tol_group_max_abs_s": TOL_GROUP_MAX_ABS_S,
}])

# Attach global fields to each summary row for self-contained CSV
for c in global_summary.columns:
    summary[c] = global_summary.loc[0, c]


# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_long = DATA / "day19A_step2_geometry_helper_validation_long.csv"
out_summary = DATA / "day19A_step2_geometry_helper_validation_summary.csv"

cmp_df.to_csv(out_long, index=False)
summary.to_csv(out_summary, index=False)

print(f"\nWrote: {out_long} ({len(cmp_df)} rows)")
print(f"Wrote: {out_summary} ({len(summary)} groups)")

display(global_summary)
display(summary.sort_values("max_abs_err_s", ascending=False).head(20))


# ---------------------------------------------------------------------
# Hard-fail if helper fails to reproduce stored geometry within smoke tolerance
# ---------------------------------------------------------------------

bad = summary[summary["validation_status"] != "ok"]

assert bad.empty, (
    "Some geometry groups exceed tolerance:\n"
    f"{bad[['dataset', 'param_set', 'protocol_label_for_parse', 'max_abs_err_s', 'validation_status']]}"
)

assert global_summary.loc[0, "global_status"] == "ok", (
    "Global geometry validation failed:\n"
    f"{global_summary}"
)

print("\n" + "=" * 72)
print("Cell 4D PASSED — geometry helper reproduces stored Day18B / Day18 v4 geometry columns")
print("=" * 72)

[Day18B] recompute Ecker2015 | DCAC_DC0p2C_AC0p4C_0p1tau | n=29
[Day18B] recompute Ecker2015 | DCAC_DC0p2C_AC0p4C_1tau | n=29
[Day18B] recompute Ecker2015 | DCAC_DC0p2C_AC0p4C_10tau | n=29
[Day18B] recompute Ecker2015 | DCAC_DC0p4C_AC0p6C_0p1tau | n=28
[Day18B] recompute Ecker2015 | DCAC_DC0p4C_AC0p6C_1tau | n=28
[Day18B] recompute Ecker2015 | DCAC_DC0p4C_AC0p6C_10tau | n=28
[Day18B] recompute Chen2020 | DCAC_DC0p2C_AC0p4C_0p1tau | n=79
[Day18B] recompute Chen2020 | DCAC_DC0p2C_AC0p4C_1tau | n=75
[Day18B] recompute Chen2020 | DCAC_DC0p2C_AC0p4C_10tau | n=78
[Day18B] recompute Chen2020 | DCAC_DC0p4C_AC0p6C_0p1tau | n=67
[Day18B] recompute Chen2020 | DCAC_DC0p4C_AC0p6C_1tau | n=64
[Day18B] recompute Chen2020 | DCAC_DC0p4C_AC0p6C_10tau | n=60
[Day18B] recompute ORegan2022 | DCAC_DC0p2C_AC0p4C_0p1tau | n=87
[Day18B] recompute ORegan2022 | DCAC_DC0p2C_AC0p4C_1tau | n=87
[Day18B] recompute ORegan2022 | DCAC_DC0p2C_AC0p4C_10tau | n=85
[Day18B] recompute ORegan2022 | DCAC_DC0p4C_AC0p6C_0p1tau 

,n_rows,n_groups,global_max_abs_err_s,global_median_abs_err_s,global_mean_err_s,global_status,tol_global_max_abs_s,tol_group_max_abs_s
0,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1


,dataset,param_set,protocol_label_for_parse,phase_used_for_recompute,n,max_abs_err_s,median_abs_err_s,mean_err_s,stored_geom_median_s,recomputed_geom_median_s,...,dt_eval_s_used,validation_status,n_rows,n_groups,global_max_abs_err_s,global_median_abs_err_s,global_mean_err_s,global_status,tol_global_max_abs_s,tol_group_max_abs_s
20,day18_v4_charge_first,ORegan2022,DCAC_DC0p2C_AC0p5C_10tau,charge_first,80,2206.651287,1047.142025,-1089.138917,1378.839433,371.040445,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
13,day18B_smoke,ORegan2022,DCAC_DC0p2C_AC0p4C_10tau,charge_first,85,1856.837520,837.234128,-872.222502,1150.917956,264.551155,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
19,day18_v4_charge_first,OKane2022,DCAC_DC0p2C_AC0p5C_10tau,charge_first,80,1824.341929,945.917251,-924.356506,1302.003184,360.302348,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
18,day18_v4_charge_first,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,charge_first,80,1774.546218,881.993568,-902.464660,1275.747571,360.912661,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
1,day18B_smoke,Chen2020,DCAC_DC0p2C_AC0p4C_10tau,charge_first,78,1426.402564,688.691982,-674.239484,956.981681,269.635106,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
16,day18B_smoke,ORegan2022,DCAC_DC0p4C_AC0p6C_10tau,charge_first,42,1409.531737,299.709453,-489.354621,563.717709,197.722299,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
4,day18B_smoke,Chen2020,DCAC_DC0p4C_AC0p6C_10tau,charge_first,60,1139.493013,327.761339,-412.444748,553.483956,188.603675,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
7,day18B_smoke,Ecker2015,DCAC_DC0p2C_AC0p4C_10tau,charge_first,29,502.566766,186.586033,-129.980192,406.811286,269.585423,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
10,day18B_smoke,Ecker2015,DCAC_DC0p4C_AC0p6C_10tau,charge_first,28,436.297992,170.723287,-82.513438,263.193247,190.350307,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1
14,day18B_smoke,ORegan2022,DCAC_DC0p2C_AC0p4C_1tau,charge_first,87,184.296901,88.371805,-87.193526,115.500527,27.133783,...,0.1,outside_tolerance,1272,21,2206.651287,86.556667,-340.768583,outside_tolerance,0.1,0.1


AssertionError: Some geometry groups exceed tolerance:
                  dataset   param_set   protocol_label_for_parse  \
0            day18B_smoke    Chen2020  DCAC_DC0p2C_AC0p4C_0p1tau   
1            day18B_smoke    Chen2020   DCAC_DC0p2C_AC0p4C_10tau   
2            day18B_smoke    Chen2020    DCAC_DC0p2C_AC0p4C_1tau   
3            day18B_smoke    Chen2020  DCAC_DC0p4C_AC0p6C_0p1tau   
4            day18B_smoke    Chen2020   DCAC_DC0p4C_AC0p6C_10tau   
5            day18B_smoke    Chen2020    DCAC_DC0p4C_AC0p6C_1tau   
6            day18B_smoke   Ecker2015  DCAC_DC0p2C_AC0p4C_0p1tau   
7            day18B_smoke   Ecker2015   DCAC_DC0p2C_AC0p4C_10tau   
8            day18B_smoke   Ecker2015    DCAC_DC0p2C_AC0p4C_1tau   
9            day18B_smoke   Ecker2015  DCAC_DC0p4C_AC0p6C_0p1tau   
10           day18B_smoke   Ecker2015   DCAC_DC0p4C_AC0p6C_10tau   
11           day18B_smoke   Ecker2015    DCAC_DC0p4C_AC0p6C_1tau   
12           day18B_smoke  ORegan2022  DCAC_DC0p2C_AC0p4C_0p1

In [25]:
# Cell 4D.0 — Locate actual frequency metadata for Day18B / Day18 v4 geometry validation
#
# Purpose:
#   4D failed because recomputation used fixed MJ1 tau_label=11.1 s.
#   This audit locates the actual f_Hz / tau_ref source used by Day18B / Day18 v4.
#
# Outputs:
#   data/day19A_step2_frequency_metadata_source_audit.csv
#   data/day19A_step2_day18B_npz_schema_frequency_audit.csv

import json
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# A. Search data/ for frequency / rebasing / design / tau metadata files
# ---------------------------------------------------------------------

patterns = [
    "*day18*",
    "*Day18*",
    "*rebasing*",
    "*frequency*",
    "*freq*",
    "*tau_ref*",
    "*tau*design*",
    "*design*",
]

candidate_files = []
seen = set()

for pat in patterns:
    for p in DATA.glob(pat):
        if p.is_file() and p.resolve() not in seen:
            candidate_files.append(p)
            seen.add(p.resolve())

candidate_files = sorted(candidate_files)

rows = []

for p in candidate_files:
    row = {
        "path": str(p.relative_to(REPO)),
        "name": p.name,
        "suffix": "".join(p.suffixes),
        "size_kb": round(p.stat().st_size / 1024, 2),
        "read_ok": False,
        "n_rows": np.nan,
        "n_cols": np.nan,
        "columns": "",
        "freq_like_cols": "",
        "tau_like_cols": "",
        "protocol_like_cols": "",
        "param_like_cols": "",
        "error": "",
    }

    try:
        if p.suffix in [".csv"] or p.name.endswith(".csv.gz"):
            df_head = pd.read_csv(p, compression="infer", nrows=5)
            cols = list(df_head.columns)
            cols_l = [c.lower() for c in cols]

            freq_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["f_hz", "freq", "frequency"])
            ]
            tau_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["tau", "tau_ref", "tau_label", "n_tau"])
            ]
            protocol_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["protocol", "condition", "anchor", "label"])
            ]
            param_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["param", "chem", "set"])
            ]

            # Get real row count cheaply enough for these files
            try:
                n_rows = sum(1 for _ in pd.read_csv(p, compression="infer", chunksize=100000))
            except Exception:
                n_rows = np.nan

            row.update({
                "read_ok": True,
                "n_rows": n_rows,
                "n_cols": len(cols),
                "columns": json.dumps(cols, ensure_ascii=False),
                "freq_like_cols": json.dumps(freq_cols, ensure_ascii=False),
                "tau_like_cols": json.dumps(tau_cols, ensure_ascii=False),
                "protocol_like_cols": json.dumps(protocol_cols, ensure_ascii=False),
                "param_like_cols": json.dumps(param_cols, ensure_ascii=False),
            })

        elif p.suffix == ".json":
            with open(p, encoding="utf-8") as fh:
                obj = json.load(fh)

            if isinstance(obj, list) and obj and isinstance(obj[0], dict):
                cols = list(obj[0].keys())
            elif isinstance(obj, dict):
                cols = list(obj.keys())
            else:
                cols = []

            freq_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["f_hz", "freq", "frequency"])
            ]
            tau_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["tau", "tau_ref", "tau_label", "n_tau"])
            ]
            protocol_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["protocol", "condition", "anchor", "label"])
            ]
            param_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["param", "chem", "set"])
            ]

            row.update({
                "read_ok": True,
                "n_rows": len(obj) if isinstance(obj, list) else 1,
                "n_cols": len(cols),
                "columns": json.dumps(cols, ensure_ascii=False),
                "freq_like_cols": json.dumps(freq_cols, ensure_ascii=False),
                "tau_like_cols": json.dumps(tau_cols, ensure_ascii=False),
                "protocol_like_cols": json.dumps(protocol_cols, ensure_ascii=False),
                "param_like_cols": json.dumps(param_cols, ensure_ascii=False),
            })

    except Exception as e:
        row["error"] = str(e)[:300]

    rows.append(row)

source_audit_df = pd.DataFrame(rows)

out_source = DATA / "day19A_step2_frequency_metadata_source_audit.csv"
source_audit_df.to_csv(out_source, index=False)

print(f"Wrote: {out_source}")
display(
    source_audit_df[
        [
            "path", "size_kb", "read_ok",
            "freq_like_cols", "tau_like_cols",
            "protocol_like_cols", "param_like_cols",
            "error",
        ]
    ].sort_values("path").head(80)
)


# ---------------------------------------------------------------------
# B. Inspect day18B_smoke NPZ schema for frequency/current metadata
# ---------------------------------------------------------------------

npz_rows = []

npz_files = sorted(SMOKE.glob("*.npz"))
print(f"\nFound {len(npz_files)} NPZ files in {SMOKE}")

for p in npz_files:
    z = np.load(p, allow_pickle=True)
    keys = list(z.keys())

    key_l = [k.lower() for k in keys]
    freq_like = [
        k for k in keys
        if any(s in k.lower() for s in ["f_hz", "freq", "frequency"])
    ]
    tau_like = [
        k for k in keys
        if any(s in k.lower() for s in ["tau", "tau_ref", "tau_label", "n_tau"])
    ]
    current_like = [
        k for k in keys
        if any(s in k.lower() for s in ["current", "i_", "i_a", "i_py"])
    ]
    time_like = [
        k for k in keys
        if any(s in k.lower() for s in ["time", "t_s", "t"])
    ]
    voltage_like = [
        k for k in keys
        if any(s in k.lower() for s in ["voltage", "v_"])
    ]

    # preview scalar metadata
    scalar_preview = {}
    for k in keys:
        arr = z[k]
        try:
            if np.asarray(arr).shape == ():
                scalar_preview[k] = np.asarray(arr).item()
        except Exception:
            pass

    npz_rows.append({
        "file": p.name,
        "n_keys": len(keys),
        "keys": json.dumps(keys, ensure_ascii=False),
        "freq_like_keys": json.dumps(freq_like, ensure_ascii=False),
        "tau_like_keys": json.dumps(tau_like, ensure_ascii=False),
        "time_like_keys": json.dumps(time_like, ensure_ascii=False),
        "current_like_keys": json.dumps(current_like, ensure_ascii=False),
        "voltage_like_keys": json.dumps(voltage_like, ensure_ascii=False),
        "scalar_preview": json.dumps(scalar_preview, ensure_ascii=False, default=str),
    })

npz_schema_df = pd.DataFrame(npz_rows)

out_npz = DATA / "day19A_step2_day18B_npz_schema_frequency_audit.csv"
npz_schema_df.to_csv(out_npz, index=False)

print(f"\nWrote: {out_npz}")
display(
    npz_schema_df[
        [
            "file", "freq_like_keys", "tau_like_keys",
            "time_like_keys", "current_like_keys", "scalar_preview",
        ]
    ].head(30)
)

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step2_frequency_metadata_source_audit.csv


,path,size_kb,read_ok,freq_like_cols,tau_like_cols,protocol_like_cols,param_like_cols,error
0,data/day17_step1_tau_ref_audit.csv,7.82,True,[],"[""tau1_biexp"", ""tau2_biexp"", ""tau1_secondary_6...",[],"[""param_set"", ""chem_tag""]",
1,data/day17_step1_tau_ref_descriptor_matrix.csv,0.89,True,[],"[""tau2_biexp"", ""tau2_secondary_60s"", ""tau_FG_e...",[],"[""param_set"", ""chem_tag""]",
2,data/day18B_protocol_design_full.csv,20.65,True,[],"[""tau95_eq_s"", ""n_tau""]","[""protocol_label"", ""protocol_type"", ""phase_lab...","[""param_set""]",
3,data/day18B_protocol_design_smoke.csv,7.72,True,[],"[""tau95_eq_s"", ""n_tau""]","[""protocol_label"", ""protocol_type"", ""phase_lab...","[""param_set""]",
4,data/day18B_smoke_dtQ_resid_curves_long.csv.gz,40.77,True,[],"[""n_tau""]","[""protocol_label"", ""DC_ref_protocol""]","[""param_set""]",
5,data/day18B_smoke_dtQ_resid_summary.csv,7.30,True,[],"[""n_tau""]","[""protocol_label"", ""DC_ref_protocol""]","[""param_set""]",
6,data/day18B_smoke_metadata.csv,13.05,True,[],"[""n_tau"", ""tau95_eq_s""]","[""protocol_label"", ""protocol_type"", ""phase_lab...","[""param_set""]",
7,data/day18_step0_frequency_rebasing_design_lon...,5.01,True,[],"[""n_tau"", ""n_tau_tag"", ""tau_label_display"", ""t...","[""tau_label_display"", ""f_label_Hz"", ""ratio_nat...","[""param_set""]",
8,data/day18_step0_frequency_rebasing_design_tab...,2.77,True,[],"[""tau_FG_eff_s"", ""tau2_secondary_60s_s"", ""tau2...","[""ratio_native_over_label"", ""ratio_sensitivity...","[""param_set""]",
9,data/day18_step0_outlier_policy_status.json,0.45,True,[],"[""outlier_policy_tauFG_activated""]",[],"[""n_sets""]",



Found 24 NPZ files in /Users/louislu/pybamm-dcac-superimposed/data/day18B_smoke

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step2_day18B_npz_schema_frequency_audit.csv


,file,freq_like_keys,tau_like_keys,time_like_keys,current_like_keys,scalar_preview
0,Chen2020__DCAC_DC0p2C_AC0p4C_0p1tau.npz,[],[],"[""t"", ""Q_net""]",[],{}
1,Chen2020__DCAC_DC0p2C_AC0p4C_10tau.npz,[],[],"[""t"", ""Q_net""]",[],{}
2,Chen2020__DCAC_DC0p2C_AC0p4C_1tau.npz,[],[],"[""t"", ""Q_net""]",[],{}
3,Chen2020__DCAC_DC0p4C_AC0p6C_0p1tau.npz,[],[],"[""t"", ""Q_net""]",[],{}
4,Chen2020__DCAC_DC0p4C_AC0p6C_10tau.npz,[],[],"[""t"", ""Q_net""]",[],{}
5,Chen2020__DCAC_DC0p4C_AC0p6C_1tau.npz,[],[],"[""t"", ""Q_net""]",[],{}
6,Chen2020__DC_0p2C.npz,[],[],"[""t"", ""Q_net""]",[],{}
7,Chen2020__DC_0p4C.npz,[],[],"[""t"", ""Q_net""]",[],{}
8,Ecker2015__DCAC_DC0p2C_AC0p4C_0p1tau.npz,[],[],"[""t"", ""Q_net""]",[],{}
9,Ecker2015__DCAC_DC0p2C_AC0p4C_10tau.npz,[],[],"[""t"", ""Q_net""]",[],{}


In [26]:
# Cell 4D.1 — Inspect actual frequency metadata and join keys for 4D v2
#
# Purpose:
#   4D v1 failed because fixed tau_label=11.1 s was used.
#   This cell inspects the real Day18B / Day18 v4 frequency metadata sources
#   before rerunning geometry validation.
#
# Outputs:
#   data/day19A_step2_frequency_join_key_audit.csv

import json
import numpy as np
import pandas as pd

files = {
    "day18B_protocol_design_smoke": DATA / "day18B_protocol_design_smoke.csv",
    "day18B_smoke_metadata": DATA / "day18B_smoke_metadata.csv",
    "day18B_curves": DATA / "day18B_smoke_dtQ_resid_curves_long.csv.gz",
    "day18_v4_audit": DATA / "day18_step2_dt_Q_audit_v4_charge_first.csv",
    "day18_v4_curves": DATA / "day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz",
    "day18_phase_matrix_v4": DATA / "day18_step1_phase_audit_matrix_v4_charge_first.csv",
}

for name, path in files.items():
    print("\n" + "=" * 100)
    print(f"[{name}] {path.name}")
    assert path.exists(), f"Missing: {path}"

    df = pd.read_csv(path, compression="infer")
    print(f"shape = {df.shape}")
    print(f"columns = {list(df.columns)}")

    # print selected columns if present
    key_cols = [
        "param_set", "chem_tag",
        "protocol_label", "protocol_type", "anchor_label",
        "protocol", "protocol_DCAC", "protocol_DC",
        "phase_label",
        "n_tau", "tau95_eq_s", "tau_FG_eff_s",
        "f_anchor_Hz", "T_anchor_s",
        "DC_C", "AC_C",
    ]
    present = [c for c in key_cols if c in df.columns]

    if present:
        print("\nSelected preview:")
        display(df[present].head(10))

    # uniqueness diagnostics
    candidate_keys = [
        ["param_set", "protocol_label"],
        ["param_set", "protocol_label", "phase_label"],
        ["param_set", "anchor_label"],
        ["param_set", "anchor_label", "phase_label"],
        ["param_set", "protocol_DCAC"],
        ["param_set", "protocol_DCAC", "phase_label"],
    ]

    print("\nUniqueness diagnostics:")
    for keys in candidate_keys:
        if all(k in df.columns for k in keys):
            n_groups = df.groupby(keys, dropna=False).ngroups
            n_rows = len(df)
            dup_groups = (
                df.groupby(keys, dropna=False)
                .size()
                .reset_index(name="n")
                .query("n > 1")
            )
            print(
                f"  keys={keys}: groups={n_groups}, rows={n_rows}, "
                f"dup_groups={len(dup_groups)}"
            )
            if len(dup_groups):
                display(dup_groups.head(10))

# ---------------------------------------------------------------------
# Build compact join-key audit table
# ---------------------------------------------------------------------

audit_rows = []

# Day18B curves vs design / metadata
curves18b = pd.read_csv(files["day18B_curves"], compression="infer")
for meta_name in ["day18B_protocol_design_smoke", "day18B_smoke_metadata"]:
    meta = pd.read_csv(files[meta_name], compression="infer")

    if {"param_set", "protocol_label"}.issubset(meta.columns):
        curve_keys = curves18b[["param_set", "protocol_label"]].drop_duplicates()
        meta_keys = meta[["param_set", "protocol_label"]].drop_duplicates()

        merged = curve_keys.merge(
            meta_keys,
            on=["param_set", "protocol_label"],
            how="left",
            indicator=True,
        )

        audit_rows.append({
            "dataset": "day18B",
            "metadata_source": meta_name,
            "join_keys": "param_set, protocol_label",
            "n_curve_keys": len(curve_keys),
            "n_meta_keys": len(meta_keys),
            "n_matched": int((merged["_merge"] == "both").sum()),
            "n_missing": int((merged["_merge"] == "left_only").sum()),
            "missing_examples": json.dumps(
                merged[merged["_merge"] == "left_only"]
                [["param_set", "protocol_label"]]
                .head(10)
                .to_dict("records"),
                ensure_ascii=False,
            ),
        })

# Day18 v4 curves vs audit table
curves18v4 = pd.read_csv(files["day18_v4_curves"], compression="infer")
audit18v4 = pd.read_csv(files["day18_v4_audit"], compression="infer")

for keys in [
    ["param_set", "anchor_label", "phase_label"],
    ["param_set", "anchor_label"],
]:
    if all(k in curves18v4.columns for k in keys) and all(k in audit18v4.columns for k in keys):
        curve_keys = curves18v4[keys].drop_duplicates()
        meta_keys = audit18v4[keys].drop_duplicates()
        merged = curve_keys.merge(meta_keys, on=keys, how="left", indicator=True)

        audit_rows.append({
            "dataset": "day18_v4",
            "metadata_source": "day18_step2_dt_Q_audit_v4_charge_first",
            "join_keys": ", ".join(keys),
            "n_curve_keys": len(curve_keys),
            "n_meta_keys": len(meta_keys),
            "n_matched": int((merged["_merge"] == "both").sum()),
            "n_missing": int((merged["_merge"] == "left_only").sum()),
            "missing_examples": json.dumps(
                merged[merged["_merge"] == "left_only"][keys]
                .head(10)
                .to_dict("records"),
                ensure_ascii=False,
            ),
        })

join_audit_df = pd.DataFrame(audit_rows)

out = DATA / "day19A_step2_frequency_join_key_audit.csv"
join_audit_df.to_csv(out, index=False)

print("\n" + "=" * 100)
print(f"Wrote: {out}")
display(join_audit_df)


[day18B_protocol_design_smoke] day18B_protocol_design_smoke.csv
shape = (24, 32)
columns = ['param_set', 'Q_nom_Ah', 'tau95_eq_s', 'V_max_cutoff', 'V_min_cutoff', 'protocol_label', 'protocol_type', 'phase_label', 'current_formula', 'DC_C', 'AC_C', 'kappa', 'n_tau', 'T_period_s', 'f_anchor_Hz', 'I_DC_A', 'I_AC_A', 'I_peak_charge_A', 'I_peak_discharge_A', 'I_peak_charge_C', 'I_peak_discharge_C', 'peak_current_constraint_margin_C', 'dt_eval_max_s', 'est_t_total_s', 'est_output_points', 'est_wall_s', 'Q_sin_ripple_amp_Ah', 'Q_lobe_per_period_Ah', 'Q_lobe_Q_frac', 'smoke_member', 'feasibility_risk', 'compute_risk']

Selected preview:


,param_set,protocol_label,protocol_type,phase_label,n_tau,tau95_eq_s,f_anchor_Hz,DC_C,AC_C
0,Ecker2015,DC_0p2C,DC,charge_first,NaN,16.349926,NaN,0.2,0.0
1,Ecker2015,DC_0p4C,DC,charge_first,NaN,16.349926,NaN,0.4,0.0
2,Ecker2015,DCAC_DC0p2C_AC0p4C_0p1tau,DCAC,charge_first,0.1,16.349926,0.097343,0.2,0.4
3,Ecker2015,DCAC_DC0p2C_AC0p4C_1tau,DCAC,charge_first,1.0,16.349926,0.009734,0.2,0.4
4,Ecker2015,DCAC_DC0p2C_AC0p4C_10tau,DCAC,charge_first,10.0,16.349926,0.000973,0.2,0.4
5,Ecker2015,DCAC_DC0p4C_AC0p6C_0p1tau,DCAC,charge_first,0.1,16.349926,0.097343,0.4,0.6
6,Ecker2015,DCAC_DC0p4C_AC0p6C_1tau,DCAC,charge_first,1.0,16.349926,0.009734,0.4,0.6
7,Ecker2015,DCAC_DC0p4C_AC0p6C_10tau,DCAC,charge_first,10.0,16.349926,0.000973,0.4,0.6
8,Chen2020,DC_0p2C,DC,charge_first,NaN,39.048883,NaN,0.2,0.0
9,Chen2020,DC_0p4C,DC,charge_first,NaN,39.048883,NaN,0.4,0.0



Uniqueness diagnostics:
  keys=['param_set', 'protocol_label']: groups=24, rows=24, dup_groups=0
  keys=['param_set', 'protocol_label', 'phase_label']: groups=24, rows=24, dup_groups=0

[day18B_smoke_metadata] day18B_smoke_metadata.csv
shape = (24, 40)
columns = ['param_set', 'protocol_label', 'protocol_type', 'phase_label', 'current_formula', 'DC_C', 'AC_C', 'n_tau', 'tau95_eq_s', 'f_anchor_Hz', 'T_period_s', 'dt_eval_s', 'Q_nom_Ah', 'I_DC_A', 'I_AC_A', 't_end_s', 'termination_raw', 'termination_class', 'feasibility_verdict', 'V_init_V', 'V_min_obs', 'V_max_obs', 'V_min_cutoff', 'V_max_cutoff', 'frac_below_Vmin', 'frac_above_Vmax', 'Q_to_Vmax_Ah', 't_to_Vmax_s', 'Q_net_end_Ah', 'wall_s', 'trajectory_npz', 'error', 'I_initial_A', 'I_first_nonzero_A', 'I_min_first_period_A', 'I_max_first_period_A', 'I_initial_C', 'I_first_nonzero_C', 'I_min_first_period_C', 'I_max_first_period_C']

Selected preview:


,param_set,protocol_label,protocol_type,phase_label,n_tau,tau95_eq_s,f_anchor_Hz,DC_C,AC_C
0,Ecker2015,DC_0p2C,DC,charge_first,NaN,16.349926,NaN,0.2,0.0
1,Ecker2015,DC_0p4C,DC,charge_first,NaN,16.349926,NaN,0.4,0.0
2,Ecker2015,DCAC_DC0p2C_AC0p4C_0p1tau,DCAC,charge_first,0.1,16.349926,0.097343,0.2,0.4
3,Ecker2015,DCAC_DC0p2C_AC0p4C_1tau,DCAC,charge_first,1.0,16.349926,0.009734,0.2,0.4
4,Ecker2015,DCAC_DC0p2C_AC0p4C_10tau,DCAC,charge_first,10.0,16.349926,0.000973,0.2,0.4
5,Ecker2015,DCAC_DC0p4C_AC0p6C_0p1tau,DCAC,charge_first,0.1,16.349926,0.097343,0.4,0.6
6,Ecker2015,DCAC_DC0p4C_AC0p6C_1tau,DCAC,charge_first,1.0,16.349926,0.009734,0.4,0.6
7,Ecker2015,DCAC_DC0p4C_AC0p6C_10tau,DCAC,charge_first,10.0,16.349926,0.000973,0.4,0.6
8,Chen2020,DC_0p2C,DC,charge_first,NaN,39.048883,NaN,0.2,0.0
9,Chen2020,DC_0p4C,DC,charge_first,NaN,39.048883,NaN,0.4,0.0



Uniqueness diagnostics:
  keys=['param_set', 'protocol_label']: groups=24, rows=24, dup_groups=0
  keys=['param_set', 'protocol_label', 'phase_label']: groups=24, rows=24, dup_groups=0

[day18B_curves] day18B_smoke_dtQ_resid_curves_long.csv.gz
shape = (1032, 9)
columns = ['pair_id', 'param_set', 'protocol_label', 'DC_ref_protocol', 'n_tau', 'Q_Ah', 'dt_model_s', 'dt_geom_s', 'dt_resid_s']

Selected preview:


...

,param_set,anchor_label,n
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,2
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,2
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,2


  keys=['param_set', 'anchor_label', 'phase_label']: groups=3, rows=6, dup_groups=3


,param_set,anchor_label,phase_label,n
0,Chen2020,Route2_AC0p5C_10tau_native_charge_first,charge_first,2
1,OKane2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,2
2,ORegan2022,Route2_AC0p5C_10tau_native_charge_first,charge_first,2



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step2_frequency_join_key_audit.csv


,dataset,metadata_source,join_keys,n_curve_keys,n_meta_keys,n_matched,n_missing,missing_examples
0,day18B,day18B_protocol_design_smoke,"param_set, protocol_label",18,24,18,0,[]
1,day18B,day18B_smoke_metadata,"param_set, protocol_label",18,24,18,0,[]
2,day18_v4,day18_step2_dt_Q_audit_v4_charge_first,"param_set, anchor_label, phase_label",3,3,3,0,[]
3,day18_v4,day18_step2_dt_Q_audit_v4_charge_first,"param_set, anchor_label",3,3,3,0,[]


In [28]:
# Cell 4D v2 — Validate geometry helper against stored Day18B / Day18 v4 geometry columns
#
# Fix vs 4D v1:
#   - Do NOT use fixed tau_label_s = 11.1 s.
#   - Day18B uses actual metadata from day18B_protocol_design_smoke.csv:
#       f_anchor_Hz, I_DC_A, I_AC_A, Q_nom_Ah, phase_label
#   - Day18 v4 uses actual f_anchor_Hz and Q_nom_Ah from
#       day18_step2_dt_Q_audit_v4_charge_first.csv
#     and parses DC_C / AC_C from protocol_DCAC.
#
# Outputs:
#   data/day19A_step2_geometry_helper_validation_long.csv
#   data/day19A_step2_geometry_helper_validation_summary.csv

import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

DT_EVAL_GEOM_S = 0.1

# Start with smoke-level tolerance. If max error is much smaller, tighten later.
TOL_GROUP_MAX_ABS_S = 0.10
TOL_GLOBAL_MAX_ABS_S = 0.10

day18b_curves_path = DATA / "day18B_smoke_dtQ_resid_curves_long.csv.gz"
day18b_design_path = DATA / "day18B_protocol_design_smoke.csv"

day18v4_curves_path = DATA / "day18_step2_dt_Q_curves_long_v4_charge_first.csv.gz"
day18v4_audit_path = DATA / "day18_step2_dt_Q_audit_v4_charge_first.csv"

for p in [day18b_curves_path, day18b_design_path, day18v4_curves_path, day18v4_audit_path]:
    assert p.exists(), f"Missing: {p}"


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def _p_to_float(s):
    return float(str(s).replace("p", "."))


def parse_dcac_protocol_label(label):
    """
    Parse labels such as:
        DCAC_DC0p2C_AC0p5C_10tau
        DC0p2C_AC0p5C_10tau
    """
    s = str(label)

    if s.upper().startswith("DCAC_"):
        s = s[5:]

    pat = re.compile(
        r"DC(?P<dc>[0-9p.]+)C[_+]?AC(?P<ac>[0-9p.]+)C[_-]?(?P<tau>[0-9p.]+)tau",
        re.IGNORECASE,
    )

    m = pat.search(s)
    if not m:
        raise ValueError(f"Cannot parse DCAC protocol label: {label!r}")

    return {
        "DC_C": _p_to_float(m.group("dc")),
        "AC_C": _p_to_float(m.group("ac")),
        "n_tau": _p_to_float(m.group("tau")),
    }


def normalize_phase_label(phase):
    phase = str(phase)
    if phase == PHASE_CHARGE_FIRST:
        return PHASE_CHARGE_FIRST
    if phase == PHASE_DISCHARGE_FIRST:
        return PHASE_DISCHARGE_FIRST
    raise ValueError(f"Unsupported phase_label: {phase!r}")


def recompute_geom_from_metadata_group(
    group_df,
    dataset_name,
    Q_col_Ah,
    stored_geom_col,
    phase,
    I_DC_A,
    I_AC_A,
    f_Hz,
    metadata_extra,
):
    """
    Recompute geometry for one already-resolved group.
    """
    q_numeric = pd.to_numeric(group_df[Q_col_Ah], errors="coerce")
    stored_numeric = pd.to_numeric(group_df[stored_geom_col], errors="coerce")

    mask_valid = q_numeric.notna() & stored_numeric.notna()

    if not mask_valid.all():
        n_drop = len(group_df) - int(mask_valid.sum())
        print(
            f"[WARN] {dataset_name}: dropping {n_drop}/{len(group_df)} rows "
            f"with NaN in {Q_col_Ah} or {stored_geom_col}"
        )
        group_df = group_df.loc[mask_valid].reset_index(drop=True)
        q_numeric = q_numeric.loc[mask_valid].reset_index(drop=True)
        stored_numeric = stored_numeric.loc[mask_valid].reset_index(drop=True)

    if len(group_df) == 0:
        raise ValueError(f"{dataset_name}: group empty after NaN filtering")

    if I_DC_A <= 0:
        raise ValueError(f"I_DC_A must be > 0, got {I_DC_A}")
    if I_AC_A < 0:
        raise ValueError(f"I_AC_A must be >= 0, got {I_AC_A}")
    if f_Hz <= 0:
        raise ValueError(f"f_Hz must be > 0, got {f_Hz}")

    Q_grid_mAh = q_numeric.to_numpy(dtype=float) * 1000.0

    out = dt_geom_from_protocol(
        I_DC_A=float(I_DC_A),
        I_AC_A=float(I_AC_A),
        f_Hz=float(f_Hz),
        phase=phase,
        Q_grid_mAh=Q_grid_mAh,
        dt_eval_s=DT_EVAL_GEOM_S,
        fp_mode=FP_MODE_RAW,
    )

    stored = stored_numeric.to_numpy(dtype=float)
    recomputed = np.asarray(out["dt_geom_s"], dtype=float)
    err = recomputed - stored

    result = group_df.copy()

    result["dataset"] = dataset_name
    result["phase_used_for_recompute"] = phase
    result["stored_geom_col"] = stored_geom_col
    result["dt_geom_recomputed_s"] = recomputed
    result["dt_geom_stored_s"] = stored
    result["geom_recompute_err_s"] = err

    result["I_DC_A_used"] = float(I_DC_A)
    result["I_AC_A_used"] = float(I_AC_A)
    result["f_Hz_used"] = float(f_Hz)
    result["dt_eval_s_used"] = DT_EVAL_GEOM_S
    result["t_max_s_used"] = out["metadata"]["t_max_s"]
    result["Q_DC_final_mAh_geom"] = out["metadata"]["Q_DC_final_mAh"]
    result["Q_DCAC_final_mAh_geom"] = out["metadata"]["Q_DCAC_final_mAh"]
    result["kappa_used"] = float(I_AC_A / I_DC_A)

    for k, v in metadata_extra.items():
        result[k] = v

    return result


# ---------------------------------------------------------------------
# Part A — Day18B smoke validation using design metadata
# ---------------------------------------------------------------------

day18b_curves = pd.read_csv(day18b_curves_path, compression="gzip")
day18b_design = pd.read_csv(day18b_design_path)

required_curves_18b = {"param_set", "protocol_label", "Q_Ah", "dt_geom_s", "n_tau"}
required_design_18b = {
    "param_set", "protocol_label", "phase_label",
    "I_DC_A", "I_AC_A", "f_anchor_Hz", "Q_nom_Ah",
    "DC_C", "AC_C", "n_tau", "tau95_eq_s",
}

missing_curves_18b = required_curves_18b - set(day18b_curves.columns)
missing_design_18b = required_design_18b - set(day18b_design.columns)

assert not missing_curves_18b, f"Day18B curves missing: {missing_curves_18b}"
assert not missing_design_18b, f"Day18B design missing: {missing_design_18b}"

# Rename design-side metadata to avoid pandas merge suffix collisions.
design_18b_meta = (
    day18b_design[
        [
            "param_set", "protocol_label",
            "phase_label", "I_DC_A", "I_AC_A", "f_anchor_Hz", "Q_nom_Ah",
            "DC_C", "AC_C", "n_tau", "tau95_eq_s",
        ]
    ]
    .rename(columns={
        "phase_label": "phase_label_meta",
        "I_DC_A": "I_DC_A_meta",
        "I_AC_A": "I_AC_A_meta",
        "f_anchor_Hz": "f_anchor_Hz_meta",
        "Q_nom_Ah": "Q_nom_Ah_meta",
        "DC_C": "DC_C_meta",
        "AC_C": "AC_C_meta",
        "n_tau": "n_tau_meta",
        "tau95_eq_s": "tau95_eq_s_meta",
    })
    .drop_duplicates(["param_set", "protocol_label"])
)

# one metadata row per protocol expected
dup_meta = (
    design_18b_meta
    .groupby(["param_set", "protocol_label"])
    .size()
    .reset_index(name="n")
    .query("n > 1")
)
assert dup_meta.empty, f"Duplicate Day18B design metadata rows:\n{dup_meta}"

day18b_join = day18b_curves.merge(
    design_18b_meta,
    on=["param_set", "protocol_label"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

missing_join_18b = day18b_join[day18b_join["_merge"] != "both"]
assert missing_join_18b.empty, (
    "Day18B curves failed to join design metadata:\n"
    f"{missing_join_18b[['param_set', 'protocol_label']].drop_duplicates().head(20)}"
)

day18b_join = day18b_join.drop(columns=["_merge"])

# Cross-check curve-side n_tau and design-side n_tau_meta.
n_tau_curve = pd.to_numeric(day18b_join["n_tau"], errors="coerce")
n_tau_meta = pd.to_numeric(day18b_join["n_tau_meta"], errors="coerce")
n_tau_err = (n_tau_curve - n_tau_meta).abs()

assert np.nanmax(n_tau_err) < 1e-9, (
    "Day18B curve n_tau does not match design metadata n_tau_meta:\n"
    f"{day18b_join.loc[n_tau_err >= 1e-9, ['param_set', 'protocol_label', 'n_tau', 'n_tau_meta']].drop_duplicates().head(20)}"
)

day18b_rows = []

for keys, g in day18b_join.groupby(["param_set", "protocol_label"], sort=False):
    param_set, protocol_label = keys

    uniq_cols = [
        "phase_label_meta",
        "I_DC_A_meta",
        "I_AC_A_meta",
        "f_anchor_Hz_meta",
        "Q_nom_Ah_meta",
        "DC_C_meta",
        "AC_C_meta",
        "n_tau_meta",
        "tau95_eq_s_meta",
    ]

    meta = {}
    for c in uniq_cols:
        vals = pd.Series(g[c]).dropna().unique()
        if len(vals) != 1:
            raise ValueError(f"Day18B group {keys}: expected unique {c}, got {vals}")
        meta[c] = vals[0]

    phase = normalize_phase_label(meta["phase_label_meta"])

    print(
        f"[Day18B v2] {param_set} | {protocol_label} | "
        f"phase={phase} | f={float(meta['f_anchor_Hz_meta']):.9g} Hz | n={len(g)}"
    )

    gg = recompute_geom_from_metadata_group(
        group_df=g,
        dataset_name="day18B_smoke",
        Q_col_Ah="Q_Ah",
        stored_geom_col="dt_geom_s",
        phase=phase,
        I_DC_A=float(meta["I_DC_A_meta"]),
        I_AC_A=float(meta["I_AC_A_meta"]),
        f_Hz=float(meta["f_anchor_Hz_meta"]),
        metadata_extra={
            "metadata_source": "day18B_protocol_design_smoke.csv",
            "protocol_label_for_parse": protocol_label,
            "param_set_meta": param_set,
            "Q_nom_Ah_used": float(meta["Q_nom_Ah_meta"]),
            "DC_C_used": float(meta["DC_C_meta"]),
            "AC_C_used": float(meta["AC_C_meta"]),
            "n_tau_used": float(meta["n_tau_meta"]),
            "tau95_eq_s_used": float(meta["tau95_eq_s_meta"]),
        },
    )

    day18b_rows.append(gg)

day18b_cmp = pd.concat(day18b_rows, ignore_index=True)


# ---------------------------------------------------------------------
# Part B — Day18 v4 charge-first validation using Step 2 audit metadata
# ---------------------------------------------------------------------

day18v4_curves = pd.read_csv(day18v4_curves_path, compression="gzip")
day18v4_audit = pd.read_csv(day18v4_audit_path)

required_curves_v4 = {
    "param_set", "anchor_label", "phase_label",
    "protocol_DCAC", "Q_Ah", "dtQ_geom_s",
}
required_audit_v4 = {
    "param_set", "anchor_label", "phase_label",
    "Q_nom_Ah", "f_anchor_Hz",
}

missing_curves_v4 = required_curves_v4 - set(day18v4_curves.columns)
missing_audit_v4 = required_audit_v4 - set(day18v4_audit.columns)

assert not missing_curves_v4, f"Day18 v4 curves missing: {missing_curves_v4}"
assert not missing_audit_v4, f"Day18 v4 audit missing: {missing_audit_v4}"

audit_v4_dedup = day18v4_audit[list(required_audit_v4)].drop_duplicates(
    ["param_set", "anchor_label", "phase_label"]
)

day18v4_join = day18v4_curves.merge(
    audit_v4_dedup,
    on=["param_set", "anchor_label", "phase_label"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

missing_join_v4 = day18v4_join[day18v4_join["_merge"] != "both"]
assert missing_join_v4.empty, (
    "Day18 v4 curves failed to join audit metadata:\n"
    f"{missing_join_v4[['param_set', 'anchor_label', 'phase_label']].drop_duplicates().head(20)}"
)

day18v4_join = day18v4_join.drop(columns=["_merge"])

day18v4_rows = []

for keys, g in day18v4_join.groupby(
    ["param_set", "anchor_label", "phase_label", "protocol_DCAC"],
    sort=False,
):
    param_set, anchor_label, phase_label, protocol_DCAC = keys

    phase = normalize_phase_label(phase_label)
    parsed = parse_dcac_protocol_label(protocol_DCAC)

    uniq_cols = ["Q_nom_Ah", "f_anchor_Hz"]
    meta = {}
    for c in uniq_cols:
        vals = pd.Series(g[c]).dropna().unique()
        if len(vals) != 1:
            raise ValueError(f"Day18 v4 group {keys}: expected unique {c}, got {vals}")
        meta[c] = vals[0]

    Q_nom_Ah = float(meta["Q_nom_Ah"])
    I_DC_A = parsed["DC_C"] * Q_nom_Ah
    I_AC_A = parsed["AC_C"] * Q_nom_Ah
    f_anchor_Hz = float(meta["f_anchor_Hz"])

    print(
        f"[Day18 v4 v2] {param_set} | {anchor_label} | {protocol_DCAC} | "
        f"phase={phase} | f={f_anchor_Hz:.9g} Hz | n={len(g)}"
    )

    gg = recompute_geom_from_metadata_group(
        group_df=g,
        dataset_name="day18_v4_charge_first",
        Q_col_Ah="Q_Ah",
        stored_geom_col="dtQ_geom_s",
        phase=phase,
        I_DC_A=I_DC_A,
        I_AC_A=I_AC_A,
        f_Hz=f_anchor_Hz,
        metadata_extra={
            "metadata_source": "day18_step2_dt_Q_audit_v4_charge_first.csv",
            "protocol_label_for_parse": protocol_DCAC,
            "anchor_label_for_join": anchor_label,
            "param_set_meta": param_set,
            "Q_nom_Ah_used": Q_nom_Ah,
            "DC_C_used": parsed["DC_C"],
            "AC_C_used": parsed["AC_C"],
            "n_tau_used": parsed["n_tau"],
            "tau95_eq_s_used": np.nan,
        },
    )

    day18v4_rows.append(gg)

day18v4_cmp = pd.concat(day18v4_rows, ignore_index=True)


# ---------------------------------------------------------------------
# Combine + summarize
# ---------------------------------------------------------------------

cmp_df = pd.concat([day18b_cmp, day18v4_cmp], ignore_index=True)
cmp_df["abs_err_s"] = cmp_df["geom_recompute_err_s"].abs()

summary = (
    cmp_df
    .groupby(
        [
            "dataset",
            "param_set",
            "protocol_label_for_parse",
            "phase_used_for_recompute",
            "metadata_source",
        ],
        dropna=False,
    )
    .agg(
        n=("geom_recompute_err_s", "size"),
        max_abs_err_s=("abs_err_s", "max"),
        median_abs_err_s=("abs_err_s", "median"),
        mean_abs_err_s=("abs_err_s", "mean"),
        mean_err_s=("geom_recompute_err_s", "mean"),
        stored_geom_median_s=("dt_geom_stored_s", "median"),
        recomputed_geom_median_s=("dt_geom_recomputed_s", "median"),
        Q_nom_Ah_used=("Q_nom_Ah_used", "first"),
        I_DC_A_used=("I_DC_A_used", "first"),
        I_AC_A_used=("I_AC_A_used", "first"),
        f_Hz_used=("f_Hz_used", "first"),
        n_tau_used=("n_tau_used", "first"),
        tau95_eq_s_used=("tau95_eq_s_used", "first"),
        dt_eval_s_used=("dt_eval_s_used", "first"),
    )
    .reset_index()
)

summary["validation_status"] = np.where(
    summary["max_abs_err_s"] <= TOL_GROUP_MAX_ABS_S,
    "ok",
    "outside_tolerance",
)

global_summary = pd.DataFrame([{
    "n_rows": len(cmp_df),
    "n_groups": len(summary),
    "global_max_abs_err_s": float(cmp_df["abs_err_s"].max()),
    "global_median_abs_err_s": float(cmp_df["abs_err_s"].median()),
    "global_mean_abs_err_s": float(cmp_df["abs_err_s"].mean()),
    "global_mean_err_s": float(cmp_df["geom_recompute_err_s"].mean()),
    "global_status": (
        "ok"
        if float(cmp_df["abs_err_s"].max()) <= TOL_GLOBAL_MAX_ABS_S
        else "outside_tolerance"
    ),
    "tol_global_max_abs_s": TOL_GLOBAL_MAX_ABS_S,
    "tol_group_max_abs_s": TOL_GROUP_MAX_ABS_S,
}])

for c in global_summary.columns:
    summary[c] = global_summary.loc[0, c]


# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_long = DATA / "day19A_step2_geometry_helper_validation_long.csv"
out_summary = DATA / "day19A_step2_geometry_helper_validation_summary.csv"

cmp_df.to_csv(out_long, index=False)
summary.to_csv(out_summary, index=False)

print(f"\nWrote: {out_long} ({len(cmp_df)} rows)")
print(f"Wrote: {out_summary} ({len(summary)} groups)")

display(global_summary)
display(summary.sort_values("max_abs_err_s", ascending=False).head(25))


# ---------------------------------------------------------------------
# Hard-fail if helper fails to reproduce stored geometry
# ---------------------------------------------------------------------

bad = summary[summary["validation_status"] != "ok"]

assert bad.empty, (
    "Some geometry groups exceed tolerance:\n"
    f"{bad[['dataset', 'param_set', 'protocol_label_for_parse', 'max_abs_err_s', 'validation_status']]}"
)

assert global_summary.loc[0, "global_status"] == "ok", (
    "Global geometry validation failed:\n"
    f"{global_summary}"
)

print("\n" + "=" * 72)
print("Cell 4D v2 PASSED — geometry helper reproduces stored Day18B / Day18 v4 geometry columns")
print("=" * 72)

[Day18B v2] Ecker2015 | DCAC_DC0p2C_AC0p4C_0p1tau | phase=charge_first | f=0.0973429154 Hz | n=29
[Day18B v2] Ecker2015 | DCAC_DC0p2C_AC0p4C_1tau | phase=charge_first | f=0.00973429154 Hz | n=29
[Day18B v2] Ecker2015 | DCAC_DC0p2C_AC0p4C_10tau | phase=charge_first | f=0.000973429154 Hz | n=29
[Day18B v2] Ecker2015 | DCAC_DC0p4C_AC0p6C_0p1tau | phase=charge_first | f=0.0973429154 Hz | n=28
[Day18B v2] Ecker2015 | DCAC_DC0p4C_AC0p6C_1tau | phase=charge_first | f=0.00973429154 Hz | n=28
[Day18B v2] Ecker2015 | DCAC_DC0p4C_AC0p6C_10tau | phase=charge_first | f=0.000973429154 Hz | n=28
[Day18B v2] Chen2020 | DCAC_DC0p2C_AC0p4C_0p1tau | phase=charge_first | f=0.0407578731 Hz | n=79
[Day18B v2] Chen2020 | DCAC_DC0p2C_AC0p4C_1tau | phase=charge_first | f=0.00407578731 Hz | n=75
[Day18B v2] Chen2020 | DCAC_DC0p2C_AC0p4C_10tau | phase=charge_first | f=0.000407578731 Hz | n=78
[Day18B v2] Chen2020 | DCAC_DC0p4C_AC0p6C_0p1tau | phase=charge_first | f=0.0407578731 Hz | n=67
[Day18B v2] Chen2020 | D

,n_rows,n_groups,global_max_abs_err_s,global_median_abs_err_s,global_mean_abs_err_s,global_mean_err_s,global_status,tol_global_max_abs_s,tol_group_max_abs_s
0,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1


,dataset,param_set,protocol_label_for_parse,phase_used_for_recompute,metadata_source,n,max_abs_err_s,median_abs_err_s,mean_abs_err_s,mean_err_s,...,validation_status,n_rows,n_groups,global_max_abs_err_s,global_median_abs_err_s,global_mean_abs_err_s,global_mean_err_s,global_status,tol_global_max_abs_s,tol_group_max_abs_s
13,day18B_smoke,ORegan2022,DCAC_DC0p2C_AC0p4C_10tau,charge_first,day18B_protocol_design_smoke.csv,85,0.098797,0.000248,0.001823,0.001473,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
10,day18B_smoke,Ecker2015,DCAC_DC0p4C_AC0p6C_10tau,charge_first,day18B_protocol_design_smoke.csv,28,0.035911,0.000907,0.002790,0.001499,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
18,day18_v4_charge_first,Chen2020,DCAC_DC0p2C_AC0p5C_10tau,charge_first,day18_step2_dt_Q_audit_v4_charge_first.csv,80,0.034863,0.001374,0.002739,0.001992,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
19,day18_v4_charge_first,OKane2022,DCAC_DC0p2C_AC0p5C_10tau,charge_first,day18_step2_dt_Q_audit_v4_charge_first.csv,80,0.017776,0.001806,0.003035,0.002261,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
20,day18_v4_charge_first,ORegan2022,DCAC_DC0p2C_AC0p5C_10tau,charge_first,day18_step2_dt_Q_audit_v4_charge_first.csv,80,0.017041,0.001161,0.002179,0.001436,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
17,day18B_smoke,ORegan2022,DCAC_DC0p4C_AC0p6C_1tau,charge_first,day18B_protocol_design_smoke.csv,55,0.015398,0.000327,0.001046,0.000461,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
2,day18B_smoke,Chen2020,DCAC_DC0p2C_AC0p4C_1tau,charge_first,day18B_protocol_design_smoke.csv,75,0.014505,0.000177,0.000795,0.000642,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
7,day18B_smoke,Ecker2015,DCAC_DC0p2C_AC0p4C_10tau,charge_first,day18B_protocol_design_smoke.csv,29,0.014228,0.000973,0.002236,0.000825,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
1,day18B_smoke,Chen2020,DCAC_DC0p2C_AC0p4C_10tau,charge_first,day18B_protocol_design_smoke.csv,78,0.013792,0.000275,0.001224,0.000662,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1
6,day18B_smoke,Ecker2015,DCAC_DC0p2C_AC0p4C_0p1tau,charge_first,day18B_protocol_design_smoke.csv,29,0.009542,0.000528,0.001032,-0.000988,...,ok,1272,21,0.098797,0.000278,0.001171,0.000546,ok,0.1,0.1



Cell 4D v2 PASSED — geometry helper reproduces stored Day18B / Day18 v4 geometry columns


In [29]:
# Cell 5A — Legacy Day8/Day9 DC-vs-DCAC pair inventory + Q-window audit
#
# Purpose:
#   Build a pair-level inventory for historical Day8/Day9 JSON outputs.
#   For each OK DCAC record:
#     - find matching DC baseline by I_DC_Crate
#     - clean both trajectories via Cell 4C keep-last dedup
#     - infer phase from early current slope
#     - infer CC current magnitudes from CC portion of I_chg
#     - define shared CC-only Q-window:
#           q_lo = 0.05 * Q_ref
#           q_hi = min(Q_CC_end_DC, Q_CC_end_DCAC) - 0.02 * Q_ref
#       where Q_ref = DC final Q_mAh
#
# Outputs:
#   data/day19A_step3_legacy_pair_inventory.csv
#
# Notes:
#   This cell does not yet compute Δt_resid. It only determines which pairs
#   are admissible for Cell 5B residual decomposition.

import json as json_module
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# Input JSON files
# ---------------------------------------------------------------------

legacy_json_files = [
    DATA / "results_day8_x2_dfn_mj1freq_v2.json",
    DATA / "results_day8_x4_dfn_chenfreq.json",
    DATA / "results_day8_x5A_dfn_corrected_mj1freq.json",
    DATA / "results_day8_x5BC_dfn_corrected_chenfreq.json",
    DATA / "results_day9_x5beta_v2_truephase0_mj1freq.json",
    DATA / "results_day9_x6alpha_composite_sigmoid_mj1freq.json",
    DATA / "results_day9_x6beta_v2_truephase0_mj1freq.json",
]

legacy_json_files = [p for p in legacy_json_files if p.exists()]
assert legacy_json_files, "No legacy Day8/Day9 JSON files found."

print(f"Legacy JSON files selected: {len(legacy_json_files)}")
for p in legacy_json_files:
    print(f"  - {p.relative_to(REPO)}")


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def interp_Q_at_time(t_s, Q_mAh, t_query_s):
    """
    Interpolate Q(t_query). Returns NaN if t_query outside range or invalid.
    """
    t_s = np.asarray(t_s, dtype=float)
    Q_mAh = np.asarray(Q_mAh, dtype=float)

    if not np.isfinite(t_query_s):
        return np.nan
    if len(t_s) == 0:
        return np.nan
    if t_query_s < t_s[0] or t_query_s > t_s[-1]:
        return np.nan

    return float(np.interp(t_query_s, t_s, Q_mAh))


def infer_phase_from_current(t_s, I_A, dc_A_est=None, n_probe=20, eps=1e-9):
    """
    Infer charge_first vs discharge_first from early current movement.

    PyBaMM convention:
        charge_first    : I moves more negative after t=0
        discharge_first : I moves more positive after t=0

    Returns
    -------
    phase : str
        charge_first / discharge_first / dc_only / unknown
    metric : float
        median(I[1:n] - I[0])
    """
    t_s = np.asarray(t_s, dtype=float)
    I_A = np.asarray(I_A, dtype=float)

    if len(I_A) < 3:
        return "unknown", np.nan

    n = min(len(I_A), max(3, n_probe))
    delta = np.nanmedian(I_A[1:n] - I_A[0])

    if not np.isfinite(delta) or abs(delta) <= eps:
        return "unknown", float(delta) if np.isfinite(delta) else np.nan

    if delta < 0:
        return PHASE_CHARGE_FIRST, float(delta)
    else:
        return PHASE_DISCHARGE_FIRST, float(delta)


def cc_mask_from_record(t_s, record):
    """
    Boolean mask for CC segment based on CC_time_s.
    Includes samples t <= CC_time_s.
    """
    t_s = np.asarray(t_s, dtype=float)
    cc_time_s = record.get("CC_time_s", np.nan)

    try:
        cc_time_s = float(cc_time_s)
    except Exception:
        cc_time_s = np.nan

    if not np.isfinite(cc_time_s):
        # fallback: use full trajectory, but mark later via metadata
        return np.ones_like(t_s, dtype=bool), np.nan, "missing_CC_time_use_full"

    return t_s <= cc_time_s + 1e-9, cc_time_s, "ok"


def infer_current_magnitudes_from_cc(record, traj, eps=1e-12):
    """
    Infer I_DC_A and I_AC_A from CC portion of stored I_chg.

    Uses current extrema in CC:
        I_DC_A ≈ -0.5 * (I_min + I_max)
        I_AC_A ≈  0.5 * (I_max - I_min)

    For DC baseline, I_AC_A is set to 0 and I_DC_A is inferred from median/negative current.

    Returns dict with diagnostics.
    """
    t = traj["t_s"]
    I = traj["I_A"]

    mask, cc_time_s, cc_status = cc_mask_from_record(t, record)
    if mask.sum() < 3:
        mask = np.ones_like(t, dtype=bool)
        cc_status = "too_few_CC_samples_use_full"

    I_cc = I[mask]

    I_min = float(np.nanmin(I_cc))
    I_max = float(np.nanmax(I_cc))
    I_median = float(np.nanmedian(I_cc))

    A_C = float(record.get("A_Crate", 0.0) or 0.0)
    DC_C = float(record.get("I_DC_Crate", np.nan))

    if A_C > eps:
        I_DC_A_est = -0.5 * (I_min + I_max)
        I_AC_A_est = 0.5 * (I_max - I_min)
    else:
        # DC baseline: current is negative during charge.
        # Median is safer than max because CV tail can approach zero.
        I_DC_A_est = abs(I_median)
        I_AC_A_est = 0.0

    # C-rate implied nominal capacity estimates
    I_nom_from_DC = I_DC_A_est / DC_C if np.isfinite(DC_C) and DC_C > 0 else np.nan
    I_nom_from_AC = I_AC_A_est / A_C if A_C > eps else np.nan

    return {
        "I_DC_A_est": float(I_DC_A_est),
        "I_AC_A_est": float(I_AC_A_est),
        "I_min_CC_A": I_min,
        "I_max_CC_A": I_max,
        "I_median_CC_A": I_median,
        "I_nom_from_DC_A": float(I_nom_from_DC) if np.isfinite(I_nom_from_DC) else np.nan,
        "I_nom_from_AC_A": float(I_nom_from_AC) if np.isfinite(I_nom_from_AC) else np.nan,
        "CC_time_s": cc_time_s,
        "CC_mask_status": cc_status,
        "n_CC_samples": int(mask.sum()),
    }


def make_shared_cc_q_window(dc_traj, dcac_traj, dc_record, dcac_record):
    """
    Define shared CC-only Q-window for residual audit.
    """
    Q_ref_mAh = float(dc_traj["Q_recomputed_mAh"][-1])

    dc_cc_time = float(dc_record.get("CC_time_s", np.nan))
    dcac_cc_time = float(dcac_record.get("CC_time_s", np.nan))

    Q_CC_end_DC_mAh = interp_Q_at_time(
        dc_traj["t_s"],
        dc_traj["Q_recomputed_mAh"],
        dc_cc_time,
    )
    Q_CC_end_DCAC_mAh = interp_Q_at_time(
        dcac_traj["t_s"],
        dcac_traj["Q_recomputed_mAh"],
        dcac_cc_time,
    )

    q_lo_mAh = 0.05 * Q_ref_mAh
    q_hi_mAh = min(Q_CC_end_DC_mAh, Q_CC_end_DCAC_mAh) - 0.02 * Q_ref_mAh

    valid = (
        np.isfinite(q_lo_mAh)
        and np.isfinite(q_hi_mAh)
        and q_hi_mAh > q_lo_mAh
    )

    return {
        "Q_ref_mAh": Q_ref_mAh,
        "Q_CC_end_DC_mAh": Q_CC_end_DC_mAh,
        "Q_CC_end_DCAC_mAh": Q_CC_end_DCAC_mAh,
        "q_lo_mAh": q_lo_mAh,
        "q_hi_mAh": q_hi_mAh,
        "q_window_valid": bool(valid),
        "q_window_span_mAh": float(q_hi_mAh - q_lo_mAh) if valid else np.nan,
    }


def classify_pair_admissibility(row):
    """
    Decide whether a pair can enter Cell 5B.
    """
    if row["dc_status"] != "ok" or row["dcac_status"] != "ok":
        return "exclude_non_ok"
    if not row["q_window_valid"]:
        return "exclude_invalid_q_window"
    if row["inferred_phase"] not in [PHASE_CHARGE_FIRST, PHASE_DISCHARGE_FIRST]:
        return "exclude_unknown_phase"
    if not np.isfinite(row["I_DC_A_used"]) or not np.isfinite(row["I_AC_A_used"]):
        return "exclude_bad_current_inference"
    if row["I_DC_A_used"] <= 0:
        return "exclude_nonpositive_I_DC"
    if row["I_AC_A_used"] < 0:
        return "exclude_negative_I_AC"
    if not np.isfinite(row["f_Hz"]) or row["f_Hz"] <= 0:
        return "exclude_bad_frequency"
    return "admissible"


# ---------------------------------------------------------------------
# Build pair inventory
# ---------------------------------------------------------------------

pair_rows = []

for json_path in legacy_json_files:
    records = load_legacy_json_records(json_path)

    print(f"\n[{json_path.name}] records={len(records)}")

    # Build baseline lookup by DC C-rate
    dc_baselines = {}
    for idx, rec in enumerate(records):
        if rec.get("status") != "ok":
            continue
        if not is_dc_baseline_record(rec):
            continue

        try:
            dc_c = float(rec.get("I_DC_Crate"))
        except Exception:
            continue

        if dc_c not in dc_baselines:
            dc_baselines[dc_c] = (idx, rec)

    print(f"  DC baselines found: {sorted(dc_baselines.keys())}")

    for idx, rec in enumerate(records):
        status = rec.get("status")
        if status != "ok":
            # Keep non-ok in inventory only if useful? Here skip to avoid noise.
            continue

        if is_dc_baseline_record(rec):
            continue

        try:
            dc_c = float(rec.get("I_DC_Crate"))
        except Exception:
            dc_c = np.nan

        try:
            ac_c = float(rec.get("A_Crate", np.nan))
        except Exception:
            ac_c = np.nan

        if not np.isfinite(dc_c) or dc_c not in dc_baselines:
            pair_rows.append({
                "source_json": str(json_path.relative_to(REPO)),
                "dcac_record_idx": idx,
                "dcac_case_id": rec.get("case_id"),
                "dcac_condition": rec.get("condition"),
                "dcac_status": status,
                "dc_record_idx": np.nan,
                "dc_case_id": None,
                "dc_status": None,
                "pair_status": "missing_matching_dc_baseline",
            })
            continue

        dc_idx, dc_rec = dc_baselines[dc_c]

        # Extract / clean trajectories
        dc_traj = extract_legacy_record_trajectory(dc_rec, clean=True)
        dcac_traj = extract_legacy_record_trajectory(rec, clean=True)

        # Infer current magnitudes from DCAC CC current
        mag = infer_current_magnitudes_from_cc(rec, dcac_traj)

        # Phase inference from DCAC current
        inferred_phase, phase_metric = infer_phase_from_current(
            dcac_traj["t_s"],
            dcac_traj["I_A"],
            dc_A_est=mag["I_DC_A_est"],
        )

        # Q-window
        qwin = make_shared_cc_q_window(dc_traj, dcac_traj, dc_rec, rec)

        f_Hz = float(rec.get("f_Hz", np.nan))
        kappa_meta = float(rec.get("kappa", np.nan)) if rec.get("kappa") is not None else np.nan
        kappa_used = (
            mag["I_AC_A_est"] / mag["I_DC_A_est"]
            if mag["I_DC_A_est"] > 0
            else np.nan
        )

        row = {
            "source_json": str(json_path.relative_to(REPO)),
            "source_file": json_path.name,

            "dc_record_idx": dc_idx,
            "dc_case_id": dc_rec.get("case_id"),
            "dc_condition": dc_rec.get("condition"),
            "dc_status": dc_rec.get("status"),

            "dcac_record_idx": idx,
            "dcac_case_id": rec.get("case_id"),
            "dcac_condition": rec.get("condition"),
            "dcac_status": rec.get("status"),

            "DC_C": dc_c,
            "AC_C": ac_c,
            "f_Hz": f_Hz,
            "tau_label": rec.get("tau_label"),
            "kappa_meta": kappa_meta,

            "inferred_phase": inferred_phase,
            "phase_metric_median_delta_I_A": phase_metric,

            "I_DC_A_used": mag["I_DC_A_est"],
            "I_AC_A_used": mag["I_AC_A_est"],
            "I_min_CC_A": mag["I_min_CC_A"],
            "I_max_CC_A": mag["I_max_CC_A"],
            "I_median_CC_A": mag["I_median_CC_A"],
            "I_nom_from_DC_A": mag["I_nom_from_DC_A"],
            "I_nom_from_AC_A": mag["I_nom_from_AC_A"],
            "kappa_used": kappa_used,

            "CC_time_DC_s": dc_traj["metadata"]["CC_time_s"],
            "CC_time_DCAC_s": dcac_traj["metadata"]["CC_time_s"],
            "CC_mask_status_DCAC": mag["CC_mask_status"],
            "n_CC_samples_DCAC": mag["n_CC_samples"],

            "dc_clean_removed": dc_traj["cleaning"]["n_removed"],
            "dcac_clean_removed": dcac_traj["cleaning"]["n_removed"],
            "dc_q_match_max_abs_mAh": dc_traj["q_match_max_abs_mAh"],
            "dcac_q_match_max_abs_mAh": dcac_traj["q_match_max_abs_mAh"],

            **qwin,
        }

        row["pair_status"] = classify_pair_admissibility(row)

        pair_rows.append(row)

pair_df = pd.DataFrame(pair_rows)

out_pair = DATA / "day19A_step3_legacy_pair_inventory.csv"
pair_df.to_csv(out_pair, index=False)

print(f"\nWrote: {out_pair}")
print(f"Pairs found: {len(pair_df)}")

if len(pair_df):
    print("\nPair status counts:")
    print(pair_df["pair_status"].value_counts(dropna=False).to_string())

    print("\nInferred phase counts:")
    print(pair_df["inferred_phase"].value_counts(dropna=False).to_string())

    print("\nSource-file counts:")
    print(pair_df.groupby(["source_file", "pair_status"]).size().to_string())

display_cols = [
    "source_file",
    "dcac_case_id",
    "dc_case_id",
    "DC_C",
    "AC_C",
    "tau_label",
    "f_Hz",
    "inferred_phase",
    "I_DC_A_used",
    "I_AC_A_used",
    "kappa_meta",
    "kappa_used",
    "q_lo_mAh",
    "q_hi_mAh",
    "q_window_span_mAh",
    "pair_status",
]

display(pair_df[display_cols].head(50))

# Hard check: no unknown phase among otherwise admissible-ish records
admissible = pair_df[pair_df["pair_status"].eq("admissible")]
assert len(admissible) > 0, "No admissible legacy pairs found."

print("\nCell 5A PASSED — legacy pair inventory and Q-window audit complete.")

Legacy JSON files selected: 7
  - data/results_day8_x2_dfn_mj1freq_v2.json
  - data/results_day8_x4_dfn_chenfreq.json
  - data/results_day8_x5A_dfn_corrected_mj1freq.json
  - data/results_day8_x5BC_dfn_corrected_chenfreq.json
  - data/results_day9_x5beta_v2_truephase0_mj1freq.json
  - data/results_day9_x6alpha_composite_sigmoid_mj1freq.json
  - data/results_day9_x6beta_v2_truephase0_mj1freq.json

[results_day8_x2_dfn_mj1freq_v2.json] records=31
  DC baselines found: [0.1, 0.2, 0.3, 0.4, 0.5, 0.9]

[results_day8_x4_dfn_chenfreq.json] records=62
  DC baselines found: [0.1, 0.2, 0.3, 0.4, 0.5, 0.9]

[results_day8_x5A_dfn_corrected_mj1freq.json] records=31
  DC baselines found: [0.1, 0.2, 0.3, 0.4, 0.5, 0.9]

[results_day8_x5BC_dfn_corrected_chenfreq.json] records=62
  DC baselines found: [0.1, 0.2, 0.3, 0.4, 0.5, 0.9]

[results_day9_x5beta_v2_truephase0_mj1freq.json] records=31
  DC baselines found: [0.1, 0.2, 0.3, 0.4, 0.5, 0.9]

[results_day9_x6alpha_composite_sigmoid_mj1freq.json] reco

,source_file,dcac_case_id,dc_case_id,DC_C,AC_C,tau_label,f_Hz,inferred_phase,I_DC_A_used,I_AC_A_used,kappa_meta,kappa_used,q_lo_mAh,q_hi_mAh,q_window_span_mAh,pair_status
0,results_day8_x2_dfn_mj1freq_v2.json,DC0.10C+AC0.90C_f0.01430Hz,DC0.10C_baseline,0.1,0.9,1τ,0.014300,discharge_first,0.500000,4.500000,9.000000,8.999999,256.315654,4051.559566,3795.243913,admissible
1,results_day8_x2_dfn_mj1freq_v2.json,DC0.10C+AC0.20C_f0.01430Hz,DC0.10C_baseline,0.1,0.2,1τ,0.014300,discharge_first,0.500000,1.000000,2.000000,2.000000,256.315654,4757.174713,4500.859060,admissible
2,results_day8_x2_dfn_mj1freq_v2.json,DC0.20C+AC0.30C_f0.01430Hz,DC0.20C_baseline,0.2,0.3,1τ,0.014300,discharge_first,1.000000,1.500000,1.500000,1.500000,256.316526,4509.134265,4252.817740,admissible
3,results_day8_x2_dfn_mj1freq_v2.json,DC0.20C+AC0.30C_f0.00143Hz,DC0.20C_baseline,0.2,0.3,10τ,0.001430,discharge_first,1.000004,1.499996,1.500000,1.499991,256.316526,4433.308675,4176.992149,admissible
4,results_day8_x2_dfn_mj1freq_v2.json,DC0.20C+AC0.30C_f0.00041Hz,DC0.20C_baseline,0.2,0.3,34.8τ,0.000410,discharge_first,1.000003,1.499983,1.500000,1.499978,256.316526,4292.888201,4036.571675,admissible
5,results_day8_x2_dfn_mj1freq_v2.json,DC0.20C+AC0.80C_f0.14300Hz,DC0.20C_baseline,0.2,0.8,0.1τ,0.143000,discharge_first,1.000000,4.000000,4.000000,4.000000,256.316526,4043.374541,3787.058015,admissible
6,results_day8_x2_dfn_mj1freq_v2.json,DC0.20C+AC0.80C_f0.01430Hz,DC0.20C_baseline,0.2,0.8,1τ,0.014300,discharge_first,1.000000,4.000000,4.000000,4.000000,256.316526,3961.878700,3705.562175,admissible
7,results_day8_x2_dfn_mj1freq_v2.json,DC0.30C+AC0.70C_f0.14300Hz,DC0.30C_baseline,0.3,0.7,0.1τ,0.143000,discharge_first,1.500000,3.500000,2.333333,2.333333,256.299412,3960.368022,3704.068609,admissible
8,results_day8_x2_dfn_mj1freq_v2.json,DC0.30C+AC0.70C_f0.01430Hz,DC0.30C_baseline,0.3,0.7,1τ,0.014300,discharge_first,1.500000,3.500000,2.333333,2.333334,256.299412,3897.861558,3641.562146,admissible
9,results_day8_x2_dfn_mj1freq_v2.json,DC0.30C+AC0.70C_f0.00143Hz,DC0.30C_baseline,0.3,0.7,10τ,0.001430,discharge_first,1.499899,3.499899,2.333333,2.333423,256.299412,3739.989226,3483.689813,admissible



Cell 5A PASSED — legacy pair inventory and Q-window audit complete.


In [30]:
# Cell 5B — Legacy Day8/Day9 DC-vs-DCAC residual decomposition
#
# Inputs:
#   data/day19A_step3_legacy_pair_inventory.csv
#   legacy results_day8/results_day9 JSON files
#
# Outputs:
#   data/day19A_step3_legacy_dt_resid_curves.csv
#   data/day19A_step3_legacy_dt_resid_summary.csv
#
# Definition:
#   Δt_model(Q) = t_DC(Q) − t_DCAC(Q) from stored PyBaMM trajectories
#   Δt_geom(Q)  = waveform-only geometry baseline from prescribed current
#   Δt_resid(Q) = Δt_model(Q) − Δt_geom(Q)
#
# Convention:
#   - raw strict-net first-passage
#   - no cummax by default
#   - positive Δt means DCAC reaches Q* earlier than DC
#
# Classification:
#   null_residual:       max|Δt_resid| < 1 s
#   small_residual:      max|Δt_resid| < 10 s
#   inspect_residual:    otherwise

import json as json_module
import numpy as np
import pandas as pd
from pathlib import Path

PAIR_INV = DATA / "day19A_step3_legacy_pair_inventory.csv"
assert PAIR_INV.exists(), f"Missing pair inventory: {PAIR_INV}"

pair_df = pd.read_csv(PAIR_INV)

admissible_pairs = pair_df[pair_df["pair_status"].eq("admissible")].copy()
assert len(admissible_pairs) > 0, "No admissible pairs found."

print(f"Admissible legacy pairs: {len(admissible_pairs)}")

N_Q_GRID = 80
FP_MODE_AUDIT = FP_MODE_RAW
DT_EVAL_GEOM_S = 0.1

RESID_NULL_THR_S = 1.0
RESID_SMALL_THR_S = 10.0

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def classify_residual(max_abs_resid_s, mean_abs_resid_s=None):
    if not np.isfinite(max_abs_resid_s):
        return "invalid_residual"
    if max_abs_resid_s < RESID_NULL_THR_S:
        return "null_residual"
    if max_abs_resid_s < RESID_SMALL_THR_S:
        return "small_residual"
    return "inspect_residual"


def sign_topology(x, threshold_s=1.0):
    """
    Coarse sign topology with deadband threshold.
    """
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)
    if not finite.any():
        return "invalid"

    xx = x[finite]
    pos = np.any(xx > threshold_s)
    neg = np.any(xx < -threshold_s)

    if pos and neg:
        return "mixed"
    if pos:
        return "positive_only"
    if neg:
        return "negative_only"
    return "near_zero"


def safe_ratio_abs(num, den, eps=1e-12):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)
    out = np.full_like(num, np.nan, dtype=float)
    mask = np.isfinite(num) & np.isfinite(den) & (np.abs(den) > eps)
    out[mask] = np.abs(num[mask]) / np.abs(den[mask])
    return out


# Cache loaded JSON files
records_cache = {}

def get_records_for_source(source_json):
    p = REPO / source_json
    if source_json not in records_cache:
        records_cache[source_json] = load_legacy_json_records(p)
    return records_cache[source_json]


# ---------------------------------------------------------------------
# Main decomposition
# ---------------------------------------------------------------------

curve_rows = []
summary_rows = []
error_rows = []

for pair_idx, row in admissible_pairs.reset_index(drop=True).iterrows():
    source_json = row["source_json"]
    records = get_records_for_source(source_json)

    dc_idx = int(row["dc_record_idx"])
    dcac_idx = int(row["dcac_record_idx"])

    dc_record = records[dc_idx]
    dcac_record = records[dcac_idx]

    q_lo = float(row["q_lo_mAh"])
    q_hi = float(row["q_hi_mAh"])

    if not np.isfinite(q_lo) or not np.isfinite(q_hi) or q_hi <= q_lo:
        error_rows.append({
            "pair_idx": pair_idx,
            "source_json": source_json,
            "dcac_case_id": row["dcac_case_id"],
            "error": f"Invalid Q window: q_lo={q_lo}, q_hi={q_hi}",
        })
        continue

    Q_grid_mAh = np.linspace(q_lo, q_hi, N_Q_GRID)

    try:
        # Δt_model from stored PyBaMM trajectories
        model_out = dt_from_legacy_records(
            dc_record=dc_record,
            dcac_record=dcac_record,
            Q_grid_mAh=Q_grid_mAh,
            fp_mode=FP_MODE_AUDIT,
            clean=True,
        )

        # Δt_geom from waveform-only helper
        phase = str(row["inferred_phase"])
        I_DC_A = float(row["I_DC_A_used"])
        I_AC_A = float(row["I_AC_A_used"])
        f_Hz = float(row["f_Hz"])

        geom_out = dt_geom_from_protocol(
            I_DC_A=I_DC_A,
            I_AC_A=I_AC_A,
            f_Hz=f_Hz,
            phase=phase,
            Q_grid_mAh=Q_grid_mAh,
            dt_eval_s=DT_EVAL_GEOM_S,
            fp_mode=FP_MODE_AUDIT,
        )

        dt_model_s = np.asarray(model_out["dt_s"], dtype=float)
        dt_geom_s = np.asarray(geom_out["dt_geom_s"], dtype=float)
        dt_resid_s = dt_model_s - dt_geom_s

        finite = (
            np.isfinite(dt_model_s)
            & np.isfinite(dt_geom_s)
            & np.isfinite(dt_resid_s)
        )

        if not finite.all():
            n_bad = int((~finite).sum())
        else:
            n_bad = 0

        resid_abs = np.abs(dt_resid_s[finite])
        geom_abs = np.abs(dt_geom_s[finite])
        model_abs = np.abs(dt_model_s[finite])
        resid_over_geom = safe_ratio_abs(dt_resid_s[finite], dt_geom_s[finite])
        resid_over_model = safe_ratio_abs(dt_resid_s[finite], dt_model_s[finite])

        max_abs_resid = float(np.nanmax(resid_abs)) if len(resid_abs) else np.nan
        mean_abs_resid = float(np.nanmean(resid_abs)) if len(resid_abs) else np.nan
        median_abs_resid = float(np.nanmedian(resid_abs)) if len(resid_abs) else np.nan

        residual_class = classify_residual(max_abs_resid, mean_abs_resid)

        # Long curve rows
        for j, q in enumerate(Q_grid_mAh):
            curve_rows.append({
                "pair_idx": pair_idx,
                "source_json": source_json,
                "source_file": row["source_file"],

                "dc_record_idx": dc_idx,
                "dc_case_id": row["dc_case_id"],
                "dc_condition": row["dc_condition"],

                "dcac_record_idx": dcac_idx,
                "dcac_case_id": row["dcac_case_id"],
                "dcac_condition": row["dcac_condition"],

                "DC_C": float(row["DC_C"]),
                "AC_C": float(row["AC_C"]),
                "tau_label": row["tau_label"],
                "f_Hz": f_Hz,
                "phase": phase,

                "Q_mAh": float(q),
                "Q_Ah": float(q / 1000.0),

                "t_DC_model_s": float(model_out["t_DC_s"][j]),
                "t_DCAC_model_s": float(model_out["t_DCAC_s"][j]),
                "dt_model_s": float(dt_model_s[j]),

                "t_DC_geom_s": float(geom_out["t_DC_s"][j]),
                "t_DCAC_geom_s": float(geom_out["t_DCAC_s"][j]),
                "dt_geom_s": float(dt_geom_s[j]),

                "dt_resid_s": float(dt_resid_s[j]),

                "I_DC_A_used": I_DC_A,
                "I_AC_A_used": I_AC_A,
                "kappa_used": float(row["kappa_used"]),
                "kappa_meta": float(row["kappa_meta"]) if np.isfinite(row["kappa_meta"]) else np.nan,

                "q_lo_mAh": q_lo,
                "q_hi_mAh": q_hi,
                "Q_ref_mAh": float(row["Q_ref_mAh"]),
                "Q_CC_end_DC_mAh": float(row["Q_CC_end_DC_mAh"]),
                "Q_CC_end_DCAC_mAh": float(row["Q_CC_end_DCAC_mAh"]),

                "fp_mode": FP_MODE_AUDIT,
                "N_Q_GRID": N_Q_GRID,
                "dt_eval_geom_s": DT_EVAL_GEOM_S,

                "dc_clean_removed": int(row["dc_clean_removed"]),
                "dcac_clean_removed": int(row["dcac_clean_removed"]),
                "finite": bool(finite[j]),
            })

        # Summary row
        summary_rows.append({
            "pair_idx": pair_idx,
            "source_json": source_json,
            "source_file": row["source_file"],

            "dc_case_id": row["dc_case_id"],
            "dc_condition": row["dc_condition"],
            "dcac_case_id": row["dcac_case_id"],
            "dcac_condition": row["dcac_condition"],

            "DC_C": float(row["DC_C"]),
            "AC_C": float(row["AC_C"]),
            "tau_label": row["tau_label"],
            "f_Hz": f_Hz,
            "phase": phase,
            "I_DC_A_used": I_DC_A,
            "I_AC_A_used": I_AC_A,
            "kappa_used": float(row["kappa_used"]),
            "kappa_meta": float(row["kappa_meta"]) if np.isfinite(row["kappa_meta"]) else np.nan,

            "q_lo_mAh": q_lo,
            "q_hi_mAh": q_hi,
            "Q_ref_mAh": float(row["Q_ref_mAh"]),
            "Q_window_span_mAh": float(row["q_window_span_mAh"]),
            "N_Q": N_Q_GRID,
            "n_finite": int(finite.sum()),
            "n_nonfinite": n_bad,

            "dt_model_mean_s": float(np.nanmean(dt_model_s)),
            "dt_model_median_s": float(np.nanmedian(dt_model_s)),
            "dt_model_min_s": float(np.nanmin(dt_model_s)),
            "dt_model_max_s": float(np.nanmax(dt_model_s)),
            "dt_model_sign_topology": sign_topology(dt_model_s),

            "dt_geom_mean_s": float(np.nanmean(dt_geom_s)),
            "dt_geom_median_s": float(np.nanmedian(dt_geom_s)),
            "dt_geom_min_s": float(np.nanmin(dt_geom_s)),
            "dt_geom_max_s": float(np.nanmax(dt_geom_s)),
            "dt_geom_sign_topology": sign_topology(dt_geom_s),

            "dt_resid_mean_s": float(np.nanmean(dt_resid_s)),
            "dt_resid_median_s": float(np.nanmedian(dt_resid_s)),
            "dt_resid_min_s": float(np.nanmin(dt_resid_s)),
            "dt_resid_max_s": float(np.nanmax(dt_resid_s)),
            "dt_resid_mean_abs_s": mean_abs_resid,
            "dt_resid_median_abs_s": median_abs_resid,
            "dt_resid_max_abs_s": max_abs_resid,
            "dt_resid_p95_abs_s": float(np.nanquantile(resid_abs, 0.95)) if len(resid_abs) else np.nan,
            "dt_resid_sign_topology": sign_topology(dt_resid_s),

            "resid_over_geom_median": float(np.nanmedian(resid_over_geom)) if len(resid_over_geom) else np.nan,
            "resid_over_geom_p95": float(np.nanquantile(resid_over_geom, 0.95)) if len(resid_over_geom) else np.nan,
            "resid_over_geom_max": float(np.nanmax(resid_over_geom)) if len(resid_over_geom) else np.nan,

            "resid_over_model_median": float(np.nanmedian(resid_over_model)) if len(resid_over_model) else np.nan,
            "resid_over_model_p95": float(np.nanquantile(resid_over_model, 0.95)) if len(resid_over_model) else np.nan,
            "resid_over_model_max": float(np.nanmax(resid_over_model)) if len(resid_over_model) else np.nan,

            "residual_class": residual_class,
            "pair_status": row["pair_status"],
            "fp_mode": FP_MODE_AUDIT,
            "dt_eval_geom_s": DT_EVAL_GEOM_S,

            "dc_clean_removed": int(row["dc_clean_removed"]),
            "dcac_clean_removed": int(row["dcac_clean_removed"]),
            "dc_q_match_max_abs_mAh": float(row["dc_q_match_max_abs_mAh"]),
            "dcac_q_match_max_abs_mAh": float(row["dcac_q_match_max_abs_mAh"]),
        })

    except Exception as e:
        error_rows.append({
            "pair_idx": pair_idx,
            "source_json": source_json,
            "source_file": row.get("source_file"),
            "dc_case_id": row.get("dc_case_id"),
            "dcac_case_id": row.get("dcac_case_id"),
            "error": str(e)[:500],
        })

# ---------------------------------------------------------------------
# Build DataFrames + persist
# ---------------------------------------------------------------------

curves_df = pd.DataFrame(curve_rows)
summary_df = pd.DataFrame(summary_rows)
errors_df = pd.DataFrame(error_rows)

out_curves = DATA / "day19A_step3_legacy_dt_resid_curves.csv"
out_summary = DATA / "day19A_step3_legacy_dt_resid_summary.csv"
out_errors = DATA / "day19A_step3_legacy_dt_resid_errors.csv"

curves_df.to_csv(out_curves, index=False)
summary_df.to_csv(out_summary, index=False)
errors_df.to_csv(out_errors, index=False)

print(f"Wrote: {out_curves} ({len(curves_df)} rows)")
print(f"Wrote: {out_summary} ({len(summary_df)} rows)")
print(f"Wrote: {out_errors} ({len(errors_df)} rows)")

print("\nPair decomposition status:")
print(f"  requested admissible pairs : {len(admissible_pairs)}")
print(f"  decomposed pairs           : {len(summary_df)}")
print(f"  errors                     : {len(errors_df)}")

if len(errors_df):
    display(errors_df.head(20))

# ---------------------------------------------------------------------
# Summary reports
# ---------------------------------------------------------------------

assert len(summary_df) > 0, "No residual decomposition summary rows produced."

print("\nResidual class counts:")
print(summary_df["residual_class"].value_counts(dropna=False).to_string())

print("\nPhase counts:")
print(summary_df["phase"].value_counts(dropna=False).to_string())

print("\nBy source file and residual class:")
print(summary_df.groupby(["source_file", "residual_class"]).size().to_string())

print("\nTop 20 by max |residual|:")
display(
    summary_df.sort_values("dt_resid_max_abs_s", ascending=False)
    [
        [
            "source_file", "dcac_case_id", "DC_C", "AC_C", "tau_label",
            "f_Hz", "phase",
            "dt_model_median_s", "dt_geom_median_s",
            "dt_resid_median_s", "dt_resid_max_abs_s",
            "resid_over_geom_median",
            "residual_class",
        ]
    ]
    .head(20)
)

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert len(errors_df) == 0, (
    "Some admissible pairs failed decomposition:\n"
    f"{errors_df.head(20)}"
)

assert len(summary_df) == len(admissible_pairs), (
    f"Expected {len(admissible_pairs)} decomposed pairs, got {len(summary_df)}"
)

assert (summary_df["n_nonfinite"] == 0).all(), (
    "Some decomposed pairs contain non-finite curve values:\n"
    f"{summary_df[summary_df['n_nonfinite'] != 0][['source_file', 'dcac_case_id', 'n_nonfinite']].head(20)}"
)

print("\n" + "=" * 72)
print("Cell 5B PASSED — legacy Day8/Day9 residual decomposition complete")
print("=" * 72)

Admissible legacy pairs: 198
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step3_legacy_dt_resid_curves.csv (15840 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step3_legacy_dt_resid_summary.csv (198 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step3_legacy_dt_resid_errors.csv (0 rows)

Pair decomposition status:
  requested admissible pairs : 198
  decomposed pairs           : 198
  errors                     : 0

Residual class counts:
residual_class
inspect_residual    103
small_residual       58
null_residual        37

Phase counts:
phase
discharge_first    198

By source file and residual class:
source_file                                          residual_class  
results_day8_x2_dfn_mj1freq_v2.json                  inspect_residual    10
                                                     null_residual        4
                                                     small_residual       9
results_day8_x4_dfn_chenfreq.json        

,source_file,dcac_case_id,DC_C,AC_C,tau_label,f_Hz,phase,dt_model_median_s,dt_geom_median_s,dt_resid_median_s,dt_resid_max_abs_s,resid_over_geom_median,residual_class
165,results_day9_x6alpha_composite_sigmoid_mj1freq...,DC0.20C+AC0.30C_f0.00041Hz_X6alpha,0.2,0.3,34.8τ,0.000410,discharge_first,-530.591058,-507.445721,-0.404284,1043.695786,0.001106,inspect_residual
124,results_day8_x5BC_dfn_corrected_chenfreq.json,DC0.30C+AC0.70C_f0.00080Hz_X5,0.3,0.7,10τ,0.000805,discharge_first,-303.253648,-297.364170,-0.204967,716.860145,0.001107,inspect_residual
77,results_day8_x5A_dfn_corrected_mj1freq.json,DC0.30C+AC0.70C_f0.00143Hz_X5,0.3,0.7,10τ,0.001430,discharge_first,-157.094272,-155.952048,-0.000413,409.327049,0.000934,inspect_residual
23,results_day8_x4_dfn_chenfreq.json,DC0.10C+AC0.90C_f0.00666Hz,0.1,0.9,1τ,0.006661,discharge_first,-43.522176,-53.505846,2.368233,129.516380,0.047215,inspect_residual
92,results_day8_x5BC_dfn_corrected_chenfreq.json,DC0.10C+AC0.90C_f0.00666Hz_X5,0.1,0.9,1τ,0.006661,discharge_first,-59.935120,-56.925938,-0.886603,127.048152,0.012110,inspect_residual
162,results_day9_x6alpha_composite_sigmoid_mj1freq...,DC0.10C+AC0.20C_f0.01430Hz_X6alpha,0.1,0.2,1τ,0.014300,discharge_first,-68.297936,-17.098782,-58.626813,125.775147,2.889090,inspect_residual
107,results_day8_x5BC_dfn_corrected_chenfreq.json,DC0.40C+AC0.60C_f0.00401Hz_X5,0.4,0.6,1.67τ,0.004006,discharge_first,-54.555071,-51.921928,-0.618171,113.124783,0.009299,inspect_residual
98,results_day8_x5BC_dfn_corrected_chenfreq.json,DC0.20C+AC0.80C_f0.00666Hz_X5,0.2,0.8,1τ,0.006661,discharge_first,-52.516676,-48.108309,-1.063629,111.501110,0.019359,inspect_residual
45,results_day8_x4_dfn_chenfreq.json,DC0.10C+AC0.90C_f0.00805Hz,0.1,0.9,1τ,0.008049,discharge_first,-34.422034,-46.390334,2.045642,107.297241,0.040381,inspect_residual
29,results_day8_x4_dfn_chenfreq.json,DC0.20C+AC0.80C_f0.00666Hz,0.2,0.8,1τ,0.006661,discharge_first,-49.297087,-45.569427,0.126396,105.754380,0.003478,inspect_residual



Cell 5B PASSED — legacy Day8/Day9 residual decomposition complete


In [31]:
# Cell 5C — Residual topology / spike audit for legacy Day8/Day9 decomposition
#
# Inputs:
#   data/day19A_step3_legacy_dt_resid_curves.csv
#   data/day19A_step3_legacy_dt_resid_summary.csv
#
# Outputs:
#   data/day19A_step3_legacy_residual_taxonomy.csv
#
# Purpose:
#   Refine Cell 5B max-based residual classification. Large max|resid| alone is
#   not sufficient evidence for non-geometric state-layer residual because it can
#   arise from isolated first-passage spikes, branch jumps, or endpoint effects.

import numpy as np
import pandas as pd

curves_path = DATA / "day19A_step3_legacy_dt_resid_curves.csv"
summary_path = DATA / "day19A_step3_legacy_dt_resid_summary.csv"

assert curves_path.exists(), f"Missing: {curves_path}"
assert summary_path.exists(), f"Missing: {summary_path}"

curves = pd.read_csv(curves_path)
summary = pd.read_csv(summary_path)

# Thresholds
THR_NEAR_ZERO_S = 1.0
THR_WEAK_S = 10.0
THR_STRONG_S = 60.0

EDGE_FRAC = 0.05
ISOLATED_FRAC_THR = 0.05      # <=5% points above threshold => isolated
DISTRIBUTED_FRAC_THR = 0.20   # >=20% points above threshold => distributed

rows = []

for pair_idx, g in curves.groupby("pair_idx", sort=False):
    g = g.sort_values("Q_mAh").reset_index(drop=True)
    n = len(g)

    resid = pd.to_numeric(g["dt_resid_s"], errors="coerce").to_numpy()
    geom = pd.to_numeric(g["dt_geom_s"], errors="coerce").to_numpy()
    model = pd.to_numeric(g["dt_model_s"], errors="coerce").to_numpy()
    Q = pd.to_numeric(g["Q_mAh"], errors="coerce").to_numpy()

    finite = np.isfinite(resid) & np.isfinite(geom) & np.isfinite(model) & np.isfinite(Q)

    if finite.sum() == 0:
        rows.append({
            "pair_idx": pair_idx,
            "taxonomy": "invalid_no_finite_points",
            "n": n,
            "n_finite": 0,
        })
        continue

    rr = resid[finite]
    gg_geom = geom[finite]
    mm = model[finite]
    qq = Q[finite]

    abs_r = np.abs(rr)

    max_abs = float(np.nanmax(abs_r))
    p95_abs = float(np.nanquantile(abs_r, 0.95))
    median_abs = float(np.nanmedian(abs_r))
    mean_abs = float(np.nanmean(abs_r))

    frac_abs_gt_1s = float(np.mean(abs_r > THR_NEAR_ZERO_S))
    frac_abs_gt_10s = float(np.mean(abs_r > THR_WEAK_S))
    frac_abs_gt_60s = float(np.mean(abs_r > THR_STRONG_S))

    # max residual location
    max_i_local = int(np.nanargmax(abs_r))
    max_i_global = int(np.flatnonzero(finite)[max_i_local])
    max_frac = max_i_global / max(n - 1, 1)

    edge_hit = bool(max_frac <= EDGE_FRAC or max_frac >= 1.0 - EDGE_FRAC)

    # sign topology with deadband
    pos = np.any(rr > THR_NEAR_ZERO_S)
    neg = np.any(rr < -THR_NEAR_ZERO_S)
    if pos and neg:
        sign_topology = "mixed"
    elif pos:
        sign_topology = "positive_only"
    elif neg:
        sign_topology = "negative_only"
    else:
        sign_topology = "near_zero"

    # residual / geometry ratios
    geom_abs = np.abs(gg_geom)
    valid_ratio = geom_abs > 1e-12
    if valid_ratio.any():
        resid_over_geom_median = float(np.nanmedian(abs_r[valid_ratio] / geom_abs[valid_ratio]))
        resid_over_geom_p95 = float(np.nanquantile(abs_r[valid_ratio] / geom_abs[valid_ratio], 0.95))
        resid_over_geom_max = float(np.nanmax(abs_r[valid_ratio] / geom_abs[valid_ratio]))
    else:
        resid_over_geom_median = np.nan
        resid_over_geom_p95 = np.nan
        resid_over_geom_max = np.nan

    # shape classification
    if p95_abs < THR_NEAR_ZERO_S:
        taxonomy = "null_residual"
    elif max_abs >= THR_WEAK_S and frac_abs_gt_10s <= ISOLATED_FRAC_THR:
        taxonomy = "isolated_spike"
    elif edge_hit and frac_abs_gt_10s <= DISTRIBUTED_FRAC_THR:
        taxonomy = "edge_or_boundary_artifact"
    elif p95_abs < THR_WEAK_S:
        taxonomy = "weak_distributed_residual"
    elif frac_abs_gt_10s >= DISTRIBUTED_FRAC_THR:
        taxonomy = "distributed_residual_requires_inspection"
    else:
        taxonomy = "mixed_or_intermediate"

    # Add stable-region verdict: remove edge 5% and recompute p95
    if n >= 20:
        stable_mask_global = np.zeros(n, dtype=bool)
        lo = int(np.ceil(EDGE_FRAC * n))
        hi = int(np.floor((1.0 - EDGE_FRAC) * n))
        stable_mask_global[lo:hi] = True
        stable_mask = stable_mask_global[finite]
    else:
        stable_mask = np.ones_like(rr, dtype=bool)

    if stable_mask.any():
        abs_stable = abs_r[stable_mask]
        stable_p95_abs = float(np.nanquantile(abs_stable, 0.95))
        stable_max_abs = float(np.nanmax(abs_stable))
        stable_frac_gt_10s = float(np.mean(abs_stable > THR_WEAK_S))
    else:
        stable_p95_abs = np.nan
        stable_max_abs = np.nan
        stable_frac_gt_10s = np.nan

    if np.isfinite(stable_p95_abs) and stable_p95_abs < THR_NEAR_ZERO_S:
        stable_region_verdict = "stable_null"
    elif np.isfinite(stable_p95_abs) and stable_p95_abs < THR_WEAK_S:
        stable_region_verdict = "stable_weak"
    elif np.isfinite(stable_frac_gt_10s) and stable_frac_gt_10s >= DISTRIBUTED_FRAC_THR:
        stable_region_verdict = "stable_distributed_nonzero"
    else:
        stable_region_verdict = "stable_intermediate_or_spiky"

    # Pull pair metadata from first row and summary
    first = g.iloc[0].to_dict()
    srow = summary[summary["pair_idx"].eq(pair_idx)]
    if len(srow):
        srow = srow.iloc[0].to_dict()
    else:
        srow = {}

    rows.append({
        "pair_idx": pair_idx,
        "source_file": first.get("source_file"),
        "source_json": first.get("source_json"),
        "dcac_case_id": first.get("dcac_case_id"),
        "dcac_condition": first.get("dcac_condition"),
        "dc_case_id": first.get("dc_case_id"),
        "dc_condition": first.get("dc_condition"),

        "DC_C": first.get("DC_C"),
        "AC_C": first.get("AC_C"),
        "tau_label": first.get("tau_label"),
        "f_Hz": first.get("f_Hz"),
        "phase": first.get("phase"),
        "kappa_used": first.get("kappa_used"),

        "n": n,
        "n_finite": int(finite.sum()),

        "dt_resid_median_s": float(np.nanmedian(rr)),
        "dt_resid_mean_s": float(np.nanmean(rr)),
        "dt_resid_median_abs_s": median_abs,
        "dt_resid_mean_abs_s": mean_abs,
        "dt_resid_p95_abs_s": p95_abs,
        "dt_resid_max_abs_s": max_abs,

        "frac_abs_gt_1s": frac_abs_gt_1s,
        "frac_abs_gt_10s": frac_abs_gt_10s,
        "frac_abs_gt_60s": frac_abs_gt_60s,

        "max_resid_Q_mAh": float(qq[max_i_local]),
        "max_resid_Q_frac_window": float(max_frac),
        "max_resid_is_edge": edge_hit,

        "dt_model_median_s": float(np.nanmedian(mm)),
        "dt_geom_median_s": float(np.nanmedian(gg_geom)),
        "resid_over_geom_median": resid_over_geom_median,
        "resid_over_geom_p95": resid_over_geom_p95,
        "resid_over_geom_max": resid_over_geom_max,

        "sign_topology": sign_topology,
        "taxonomy": taxonomy,

        "stable_p95_abs_s": stable_p95_abs,
        "stable_max_abs_s": stable_max_abs,
        "stable_frac_gt_10s": stable_frac_gt_10s,
        "stable_region_verdict": stable_region_verdict,

        # Previous 5B max-based class for comparison
        "cell5B_residual_class": srow.get("residual_class"),
    })

tax_df = pd.DataFrame(rows)

out_tax = DATA / "day19A_step3_legacy_residual_taxonomy.csv"
tax_df.to_csv(out_tax, index=False)

print(f"Wrote: {out_tax} ({len(tax_df)} rows)")

print("\nTaxonomy counts:")
print(tax_df["taxonomy"].value_counts(dropna=False).to_string())

print("\nStable-region verdict counts:")
print(tax_df["stable_region_verdict"].value_counts(dropna=False).to_string())

print("\nCell5B class vs Cell5C taxonomy:")
print(pd.crosstab(tax_df["cell5B_residual_class"], tax_df["taxonomy"]).to_string())

print("\nTop 25 by stable p95 |residual|:")
display(
    tax_df.sort_values("stable_p95_abs_s", ascending=False)
    [
        [
            "source_file", "dcac_case_id", "DC_C", "AC_C", "tau_label",
            "phase", "dt_model_median_s", "dt_geom_median_s",
            "dt_resid_median_s",
            "dt_resid_p95_abs_s", "dt_resid_max_abs_s",
            "frac_abs_gt_10s",
            "max_resid_Q_frac_window",
            "max_resid_is_edge",
            "taxonomy", "stable_region_verdict",
        ]
    ]
    .head(25)
)

# Hard checks
assert len(tax_df) == len(summary), (
    f"Expected taxonomy rows = summary rows ({len(summary)}), got {len(tax_df)}"
)

print("\n" + "=" * 72)
print("Cell 5C PASSED — residual topology / spike audit complete")
print("=" * 72)

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step3_legacy_residual_taxonomy.csv (198 rows)

Taxonomy counts:
taxonomy
isolated_spike                              59
null_residual                               50
weak_distributed_residual                   32
edge_or_boundary_artifact                   22
distributed_residual_requires_inspection    19
mixed_or_intermediate                       16

Stable-region verdict counts:
stable_region_verdict
stable_weak                     102
stable_null                      53
stable_intermediate_or_spiky     24
stable_distributed_nonzero       19

Cell5B class vs Cell5C taxonomy:
taxonomy               distributed_residual_requires_inspection  edge_or_boundary_artifact  isolated_spike  mixed_or_intermediate  null_residual  weak_distributed_residual
cell5B_residual_class                                                                                                                                                      
inspect_res

,source_file,dcac_case_id,DC_C,AC_C,tau_label,phase,dt_model_median_s,dt_geom_median_s,dt_resid_median_s,dt_resid_p95_abs_s,dt_resid_max_abs_s,frac_abs_gt_10s,max_resid_Q_frac_window,max_resid_is_edge,taxonomy,stable_region_verdict
162,results_day9_x6alpha_composite_sigmoid_mj1freq...,DC0.10C+AC0.20C_f0.01430Hz_X6alpha,0.1,0.2,1τ,discharge_first,-68.297936,-17.098782,-58.626813,120.040304,125.775147,0.7500,0.974684,True,distributed_residual_requires_inspection,stable_distributed_nonzero
45,results_day8_x4_dfn_chenfreq.json,DC0.10C+AC0.90C_f0.00805Hz,0.1,0.9,1τ,discharge_first,-34.422034,-46.390334,2.045642,105.439801,107.297241,0.0875,0.582278,False,mixed_or_intermediate,stable_intermediate_or_spiky
115,results_day8_x5BC_dfn_corrected_chenfreq.json,DC0.10C+AC0.90C_f0.00805Hz_X5,0.1,0.9,1τ,discharge_first,-61.895370,-53.157021,-2.650534,104.461944,105.382727,0.1000,0.063291,False,mixed_or_intermediate,stable_intermediate_or_spiky
121,results_day8_x5BC_dfn_corrected_chenfreq.json,DC0.20C+AC0.80C_f0.00805Hz_X5,0.2,0.8,1τ,discharge_first,-48.258560,-38.620965,-2.316465,91.162016,94.886118,0.0625,0.734177,False,mixed_or_intermediate,stable_intermediate_or_spiky
161,results_day9_x6alpha_composite_sigmoid_mj1freq...,DC0.10C+AC0.90C_f0.01430Hz_X6alpha,0.1,0.9,1τ,discharge_first,-59.034735,-21.342782,-58.803457,68.531808,69.053420,0.5375,0.974684,True,distributed_residual_requires_inspection,stable_distributed_nonzero
67,results_day8_x5A_dfn_corrected_mj1freq.json,DC0.10C+AC0.90C_f0.01430Hz_X5,0.1,0.9,1τ,discharge_first,-59.165304,-26.091040,-7.931127,68.607503,68.984987,0.4625,1.000000,True,distributed_residual_requires_inspection,stable_distributed_nonzero
0,results_day8_x2_dfn_mj1freq_v2.json,DC0.10C+AC0.90C_f0.01430Hz,0.1,0.9,1τ,discharge_first,-16.806135,-28.213584,2.337113,61.754534,62.701679,0.1500,0.784810,False,mixed_or_intermediate,stable_intermediate_or_spiky
138,results_day9_x5beta_v2_truephase0_mj1freq.json,DC0.10C+AC0.90C_f0.01430Hz_X5beta_v2,0.1,0.9,1τ,discharge_first,-18.833715,-27.061583,1.663072,61.080006,62.008892,0.1375,0.658228,False,mixed_or_intermediate,stable_intermediate_or_spiky
23,results_day8_x4_dfn_chenfreq.json,DC0.10C+AC0.90C_f0.00666Hz,0.1,0.9,1τ,discharge_first,-43.522176,-53.505846,2.368233,13.373271,129.516380,0.0500,0.848101,False,isolated_spike,stable_intermediate_or_spiky
73,results_day8_x5A_dfn_corrected_mj1freq.json,DC0.20C+AC0.80C_f0.01430Hz_X5,0.2,0.8,1τ,discharge_first,-31.907401,-22.165227,-2.777727,55.564462,58.059203,0.1500,1.000000,True,edge_or_boundary_artifact,stable_intermediate_or_spiky



Cell 5C PASSED — residual topology / spike audit complete


In [32]:
# Cell 5D — Lineage-level verdict aggregation for legacy Day8/Day9 residual audit
#
# Inputs:
#   data/day19A_step3_legacy_residual_taxonomy.csv
#   data/day19A_step3_legacy_dt_resid_summary.csv
#
# Outputs:
#   data/day19A_step3_legacy_lineage_verdict_summary.csv
#
# Purpose:
#   Aggregate Cell 5C pair-level taxonomy into lineage-level verdicts:
#     - clean historical DCAC branches
#     - exploratory X6 combined branches
#
# Rationale:
#   Cell 5B max-based residual classification over-flags isolated spikes.
#   Cell 5C taxonomy refines pair-level behavior.
#   Cell 5D summarizes evidential level by lineage before moving to nb20.

import numpy as np
import pandas as pd

tax_path = DATA / "day19A_step3_legacy_residual_taxonomy.csv"
summary_path = DATA / "day19A_step3_legacy_dt_resid_summary.csv"

assert tax_path.exists(), f"Missing taxonomy file: {tax_path}"
assert summary_path.exists(), f"Missing summary file: {summary_path}"

tax_df = pd.read_csv(tax_path)
summary_df = pd.read_csv(summary_path)

assert len(tax_df) > 0, "Empty taxonomy table."

# ---------------------------------------------------------------------
# Lineage classifier
# ---------------------------------------------------------------------

def classify_lineage(source_file):
    """
    Map legacy source files to lineage-level interpretation classes.

    clean_historical_dcac:
        Main historical DC-vs-DCAC protocol families.
        These are useful for phase-aware / geometry-corrected retro audit.

    exploratory_combined_branch:
        Day9 X6 branches include combined / non-clean mechanism changes
        and must not be interpreted as single-variable mechanism evidence.
    """
    s = str(source_file)

    if "x6alpha" in s.lower():
        return "day9_x6alpha_exploratory_combined"
    if "x6beta" in s.lower():
        return "day9_x6beta_exploratory_combined"
    if "x5beta" in s.lower():
        return "day9_x5beta_phase_path"
    if "x5bc" in s.lower():
        return "day8_x5BC_corrected_chenfreq"
    if "x5a" in s.lower():
        return "day8_x5A_corrected_mj1freq"
    if "x4" in s.lower():
        return "day8_x4_chenfreq"
    if "x2" in s.lower():
        return "day8_x2_mj1freq"

    return "other_legacy"


def classify_branch_type(lineage_tag):
    if "x6" in lineage_tag:
        return "exploratory_combined_branch"
    if "x5beta" in lineage_tag:
        return "phase_path_branch"
    if lineage_tag.startswith("day8_"):
        return "clean_historical_dcac_branch"
    return "other_branch"


tax_df["lineage_tag"] = tax_df["source_file"].apply(classify_lineage)
tax_df["branch_type"] = tax_df["lineage_tag"].apply(classify_branch_type)

# ---------------------------------------------------------------------
# Boolean flags
# ---------------------------------------------------------------------

tax_df["is_stable_null"] = tax_df["stable_region_verdict"].eq("stable_null")
tax_df["is_stable_weak"] = tax_df["stable_region_verdict"].eq("stable_weak")
tax_df["is_stable_null_or_weak"] = tax_df["stable_region_verdict"].isin(["stable_null", "stable_weak"])
tax_df["is_stable_distributed_nonzero"] = tax_df["stable_region_verdict"].eq("stable_distributed_nonzero")
tax_df["is_intermediate_or_spiky"] = tax_df["stable_region_verdict"].eq("stable_intermediate_or_spiky")

tax_df["is_isolated_spike"] = tax_df["taxonomy"].eq("isolated_spike")
tax_df["is_edge_artifact"] = tax_df["taxonomy"].eq("edge_or_boundary_artifact")
tax_df["is_distributed_taxonomy"] = tax_df["taxonomy"].eq("distributed_residual_requires_inspection")
tax_df["is_weak_distributed_taxonomy"] = tax_df["taxonomy"].eq("weak_distributed_residual")
tax_df["is_null_taxonomy"] = tax_df["taxonomy"].eq("null_residual")

# ---------------------------------------------------------------------
# Aggregation
# ---------------------------------------------------------------------

group_cols = ["lineage_tag", "branch_type", "phase"]

agg = (
    tax_df
    .groupby(group_cols, dropna=False)
    .agg(
        n_pairs=("pair_idx", "size"),

        n_stable_null=("is_stable_null", "sum"),
        n_stable_weak=("is_stable_weak", "sum"),
        n_stable_null_or_weak=("is_stable_null_or_weak", "sum"),
        n_stable_distributed_nonzero=("is_stable_distributed_nonzero", "sum"),
        n_intermediate_or_spiky=("is_intermediate_or_spiky", "sum"),

        n_isolated_spike=("is_isolated_spike", "sum"),
        n_edge_artifact=("is_edge_artifact", "sum"),
        n_distributed_taxonomy=("is_distributed_taxonomy", "sum"),
        n_weak_distributed_taxonomy=("is_weak_distributed_taxonomy", "sum"),
        n_null_taxonomy=("is_null_taxonomy", "sum"),

        median_dt_resid_p95_abs_s=("dt_resid_p95_abs_s", "median"),
        max_dt_resid_p95_abs_s=("dt_resid_p95_abs_s", "max"),
        median_stable_p95_abs_s=("stable_p95_abs_s", "median"),
        max_stable_p95_abs_s=("stable_p95_abs_s", "max"),
        median_resid_over_geom=("resid_over_geom_median", "median"),
        max_resid_over_geom=("resid_over_geom_max", "max"),
    )
    .reset_index()
)

# Fractions
for col in [
    "stable_null",
    "stable_weak",
    "stable_null_or_weak",
    "stable_distributed_nonzero",
    "intermediate_or_spiky",
    "isolated_spike",
    "edge_artifact",
    "distributed_taxonomy",
    "weak_distributed_taxonomy",
    "null_taxonomy",
]:
    agg[f"frac_{col}"] = agg[f"n_{col}"] / agg["n_pairs"]

# ---------------------------------------------------------------------
# Lineage verdict logic
# ---------------------------------------------------------------------

def lineage_verdict(row):
    branch = row["branch_type"]

    frac_null_weak = row["frac_stable_null_or_weak"]
    frac_distributed = row["frac_stable_distributed_nonzero"]
    frac_spiky = row["frac_intermediate_or_spiky"]
    frac_iso = row["frac_isolated_spike"]
    n_distributed = row["n_stable_distributed_nonzero"]

    if branch == "exploratory_combined_branch":
        if n_distributed > 0:
            return "distributed_candidates_in_exploratory_branch"
        if frac_null_weak >= 0.75:
            return "exploratory_branch_mostly_weak_or_null"
        return "exploratory_branch_mixed"

    if branch in ["clean_historical_dcac_branch", "phase_path_branch"]:
        if frac_null_weak >= 0.75 and n_distributed == 0:
            return "geometry_dominated_or_weak_residual"
        if n_distributed == 0 and (frac_iso + frac_spiky) > 0:
            return "geometry_dominated_with_spikes"
        if n_distributed > 0:
            return "mixed_with_distributed_candidates"
        return "mixed_unclear"

    return "other_unclassified"


def evidence_level(row):
    verdict = row["lineage_verdict"]

    if verdict == "geometry_dominated_or_weak_residual":
        return "high_for_geometry_dominated_reclassification"

    if verdict == "geometry_dominated_with_spikes":
        return "moderate_spike_artifacts_present"

    if verdict == "mixed_with_distributed_candidates":
        return "requires_pair_level_review"

    if verdict == "distributed_candidates_in_exploratory_branch":
        return "not_clean_mechanism_evidence"

    if verdict.startswith("exploratory"):
        return "exploratory_only"

    return "unclassified"


agg["lineage_verdict"] = agg.apply(lineage_verdict, axis=1)
agg["evidence_level"] = agg.apply(evidence_level, axis=1)

# ---------------------------------------------------------------------
# Add representative cases
# ---------------------------------------------------------------------

rep_rows = []

for _, row in agg.iterrows():
    sub = tax_df[
        (tax_df["lineage_tag"] == row["lineage_tag"])
        & (tax_df["branch_type"] == row["branch_type"])
        & (tax_df["phase"] == row["phase"])
    ].copy()

    top_distributed = (
        sub[sub["stable_region_verdict"].eq("stable_distributed_nonzero")]
        .sort_values("stable_p95_abs_s", ascending=False)
        .head(5)
    )

    top_spikes = (
        sub[sub["taxonomy"].eq("isolated_spike")]
        .sort_values("dt_resid_max_abs_s", ascending=False)
        .head(5)
    )

    rep_rows.append({
        "lineage_tag": row["lineage_tag"],
        "branch_type": row["branch_type"],
        "phase": row["phase"],
        "representative_distributed_cases": "; ".join(
            top_distributed["dcac_case_id"].astype(str).tolist()
        ),
        "representative_spike_cases": "; ".join(
            top_spikes["dcac_case_id"].astype(str).tolist()
        ),
    })

rep_df = pd.DataFrame(rep_rows)

agg = agg.merge(rep_df, on=["lineage_tag", "branch_type", "phase"], how="left")

# ---------------------------------------------------------------------
# Persist
# ---------------------------------------------------------------------

out_lineage = DATA / "day19A_step3_legacy_lineage_verdict_summary.csv"
agg.to_csv(out_lineage, index=False)

print(f"Wrote: {out_lineage} ({len(agg)} rows)")

display_cols = [
    "lineage_tag",
    "branch_type",
    "phase",
    "n_pairs",
    "n_stable_null_or_weak",
    "n_stable_distributed_nonzero",
    "n_intermediate_or_spiky",
    "n_isolated_spike",
    "n_edge_artifact",
    "frac_stable_null_or_weak",
    "frac_stable_distributed_nonzero",
    "median_stable_p95_abs_s",
    "max_stable_p95_abs_s",
    "lineage_verdict",
    "evidence_level",
    "representative_distributed_cases",
]

display(
    agg[display_cols]
    .sort_values(["branch_type", "lineage_tag"])
    .reset_index(drop=True)
)

print("\nLineage verdict counts:")
print(agg["lineage_verdict"].value_counts(dropna=False).to_string())

print("\nEvidence level counts:")
print(agg["evidence_level"].value_counts(dropna=False).to_string())

# ---------------------------------------------------------------------
# Global Day8/Day9 summary for notes
# ---------------------------------------------------------------------

global_counts = {
    "n_pairs": int(len(tax_df)),
    "n_stable_null_or_weak": int(tax_df["is_stable_null_or_weak"].sum()),
    "n_stable_distributed_nonzero": int(tax_df["is_stable_distributed_nonzero"].sum()),
    "n_intermediate_or_spiky": int(tax_df["is_intermediate_or_spiky"].sum()),
    "n_isolated_spike": int(tax_df["is_isolated_spike"].sum()),
    "n_edge_artifact": int(tax_df["is_edge_artifact"].sum()),
}

print("\nGlobal Day8/Day9 residual topology summary:")
for k, v in global_counts.items():
    print(f"  {k:<35}: {v}")

# Hard sanity checks
assert agg["phase"].eq(PHASE_DISCHARGE_FIRST).all(), (
    "Expected all legacy Day8/Day9 lineages to be discharge_first."
)
assert global_counts["n_pairs"] == 198, (
    f"Expected 198 pairs from Cell 5A/5B/5C, got {global_counts['n_pairs']}"
)

print("\n" + "=" * 72)
print("Cell 5D PASSED — lineage-level legacy verdict aggregation complete")
print("=" * 72)

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step3_legacy_lineage_verdict_summary.csv (7 rows)


,lineage_tag,branch_type,phase,n_pairs,n_stable_null_or_weak,n_stable_distributed_nonzero,n_intermediate_or_spiky,n_isolated_spike,n_edge_artifact,frac_stable_null_or_weak,frac_stable_distributed_nonzero,median_stable_p95_abs_s,max_stable_p95_abs_s,lineage_verdict,evidence_level,representative_distributed_cases
0,day8_x2_mj1freq,clean_historical_dcac_branch,discharge_first,23,20,0,3,7,4,0.869565,0.000000,2.090449,61.901764,geometry_dominated_or_weak_residual,high_for_geometry_dominated_reclassification,
1,day8_x4_chenfreq,clean_historical_dcac_branch,discharge_first,44,38,1,5,14,4,0.863636,0.022727,1.712104,105.459634,mixed_with_distributed_candidates,requires_pair_level_review,DC0.30C+AC0.70C_f0.06661Hz
2,day8_x5A_corrected_mj1freq,clean_historical_dcac_branch,discharge_first,25,15,4,6,6,4,0.600000,0.160000,2.788153,67.858778,mixed_with_distributed_candidates,requires_pair_level_review,DC0.10C+AC0.90C_f0.01430Hz_X5; DC0.20C+AC0.80C...
3,day8_x5BC_corrected_chenfreq,clean_historical_dcac_branch,discharge_first,46,38,6,2,13,3,0.826087,0.130435,2.819621,104.592494,mixed_with_distributed_candidates,requires_pair_level_review,DC0.20C+AC0.80C_f0.06661Hz_X5; DC0.20C+AC0.80C...
4,day9_x6alpha_exploratory_combined,exploratory_combined_branch,discharge_first,22,12,6,4,6,2,0.545455,0.272727,7.509058,119.835118,distributed_candidates_in_exploratory_branch,not_clean_mechanism_evidence,DC0.10C+AC0.20C_f0.01430Hz_X6alpha; DC0.10C+AC...
5,day9_x6beta_exploratory_combined,exploratory_combined_branch,discharge_first,15,10,2,3,5,1,0.666667,0.133333,5.291956,52.713368,distributed_candidates_in_exploratory_branch,not_clean_mechanism_evidence,DC0.10C+AC0.20C_f0.01430Hz_X6beta_v2; DC0.20C+...
6,day9_x5beta_phase_path,phase_path_branch,discharge_first,23,22,0,1,8,4,0.956522,0.000000,2.491201,61.089423,geometry_dominated_or_weak_residual,high_for_geometry_dominated_reclassification,



Lineage verdict counts:
lineage_verdict
mixed_with_distributed_candidates               3
geometry_dominated_or_weak_residual             2
distributed_candidates_in_exploratory_branch    2

Evidence level counts:
evidence_level
requires_pair_level_review                      3
high_for_geometry_dominated_reclassification    2
not_clean_mechanism_evidence                    2

Global Day8/Day9 residual topology summary:
  n_pairs                            : 198
  n_stable_null_or_weak              : 155
  n_stable_distributed_nonzero       : 19
  n_intermediate_or_spiky            : 24
  n_isolated_spike                   : 59
  n_edge_artifact                    : 22

Cell 5D PASSED — lineage-level legacy verdict aggregation complete


In [33]:
# Cell 6A — nb20 / Day16 Q-window compatibility gate
#
# Input:
#   data/day16_step3a_dtQ_curves_long.csv.gz
#   notebooks/20_param_set_wiring_audit.ipynb
#
# Outputs:
#   data/day19A_step4_day16_related_files.csv
#   data/day19A_step4_day16_notebook_window_provenance.csv
#   data/day19A_step4_day16_window_audit.csv
#
# Purpose:
#   Day16 / nb20 is a DC-vs-DCAC parameter-set audit and must be included in
#   Day19A retro audit if its Q-window is compatible.
#
# Compatibility gate:
#   1. The curve file must be internally valid:
#        dtQ_s = t_DC_s − t_protocol_s
#        Q_Ah lies within [Q_low_Ah, Q_hi_Ah]
#        Q_low_Ah / Q_hi_Ah finite and q_hi > q_low
#
#   2. The notebook provenance should support that Q_hi is a shared-Q window
#      below the limiting voltage event, e.g. min(Q_to_Vmax_DC, Q_to_Vmax_DCAC)
#      minus safety margin.
#
#   3. If internal validity passes but provenance is not confirmed, classify as:
#        needs_recompute_or_provenance
#
#   4. Only rows/groups classified compatible should feed the next residual audit.

import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

DAY16_CURVES = DATA / "day16_step3a_dtQ_curves_long.csv.gz"
NB20_PATH = NB_DIR / "20_param_set_wiring_audit.ipynb"

assert DAY16_CURVES.exists(), f"Missing Day16 curves file: {DAY16_CURVES}"
assert NB20_PATH.exists(), f"Missing nb20 notebook: {NB20_PATH}"

# ---------------------------------------------------------------------
# 1. Related Day16 file inventory
# ---------------------------------------------------------------------

related_patterns = [
    "*day16*",
    "*Day16*",
    "*step3a*",
    "*param_set*wiring*",
    "*wiring*audit*",
]

related = []
seen = set()

for pat in related_patterns:
    for p in REPO.rglob(pat):
        if not p.is_file():
            continue
        if ".ipynb_checkpoints" in p.parts or "__pycache__" in p.parts:
            continue
        if p.name.startswith(("day19A_", "day19B_", "day19D_")):
            continue
        key = p.resolve()
        if key in seen:
            continue
        seen.add(key)

        related.append({
            "path": str(p.relative_to(REPO)),
            "name": p.name,
            "suffixes": "".join(p.suffixes),
            "size_kb": round(p.stat().st_size / 1024, 2),
            "mtime": pd.Timestamp.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
        })

related_df = pd.DataFrame(related).sort_values("path").reset_index(drop=True)

out_related = DATA / "day19A_step4_day16_related_files.csv"
related_df.to_csv(out_related, index=False)

print(f"Wrote: {out_related}")
display(related_df)


# ---------------------------------------------------------------------
# 2. Notebook provenance scan for Q-window definition
# ---------------------------------------------------------------------

with open(NB20_PATH, encoding="utf-8") as fh:
    nb = json.load(fh)

provenance_rows = []

patterns = [
    "Q_hi", "Q_low", "Q_to_Vmax", "Q_to_Vmax_DC", "Q_to_Vmax_DCAC",
    "Vmax", "V_max", "voltage", "min(", "minimum", "safety", "margin",
    "window", "shared", "strict_Q", "Q_window",
]

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    src = "".join(cell.get("source", []))
    src_lower = src.lower()

    hits = [p for p in patterns if p.lower() in src_lower]

    if hits:
        # Keep a bounded excerpt around first relevant occurrence
        first_positions = [
            src_lower.find(p.lower()) for p in hits
            if src_lower.find(p.lower()) >= 0
        ]
        pos = min(first_positions) if first_positions else 0
        start = max(0, pos - 800)
        end = min(len(src), pos + 2200)

        provenance_rows.append({
            "cell_index": i,
            "cell_type": cell.get("cell_type"),
            "hits": json.dumps(hits, ensure_ascii=False),
            "contains_Q_hi": "q_hi" in src_lower,
            "contains_Q_low": "q_low" in src_lower,
            "contains_Q_to_Vmax": "q_to_vmax" in src_lower,
            "contains_min": "min(" in src_lower or "minimum" in src_lower,
            "contains_safety_margin": "safety" in src_lower or "margin" in src_lower,
            "contains_shared_or_window": "shared" in src_lower or "window" in src_lower,
            "excerpt": src[start:end],
        })

prov_df = pd.DataFrame(provenance_rows)

out_prov = DATA / "day19A_step4_day16_notebook_window_provenance.csv"
prov_df.to_csv(out_prov, index=False)

print(f"\nWrote: {out_prov}")
display(
    prov_df[
        [
            "cell_index", "hits", "contains_Q_hi", "contains_Q_low",
            "contains_Q_to_Vmax", "contains_min",
            "contains_safety_margin", "contains_shared_or_window",
        ]
    ].head(20)
)

# Overall provenance flags
if len(prov_df):
    provenance_confirms_shared_window = bool(
        prov_df["contains_Q_hi"].any()
        and prov_df["contains_Q_low"].any()
        and prov_df["contains_Q_to_Vmax"].any()
        and prov_df["contains_min"].any()
    )
else:
    provenance_confirms_shared_window = False

print("\nNotebook provenance summary:")
print(f"  provenance_confirms_shared_window = {provenance_confirms_shared_window}")
print(f"  cells_with_window_hits            = {len(prov_df)}")


# ---------------------------------------------------------------------
# 3. Day16 curve internal validity audit
# ---------------------------------------------------------------------

df = pd.read_csv(DAY16_CURVES, compression="gzip")

required = {
    "param_set", "chem_tag", "condition", "pair_role",
    "DC_C", "AC_C", "tau_label",
    "Q_Ah", "t_DC_s", "t_protocol_s", "dtQ_s",
    "Q_nom_Ah", "Q_low_Ah", "Q_hi_Ah",
}
missing = required - set(df.columns)
assert not missing, f"Day16 curves missing columns: {missing}"

# Numeric coercion
num_cols = [
    "DC_C", "AC_C", "Q_Ah", "t_DC_s", "t_protocol_s", "dtQ_s",
    "Q_nom_Ah", "Q_low_Ah", "Q_hi_Ah",
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Row-level checks
df["dtQ_recalc_s"] = df["t_DC_s"] - df["t_protocol_s"]
df["dtQ_sign_err_s"] = df["dtQ_s"] - df["dtQ_recalc_s"]
df["Q_inside_window"] = (
    (df["Q_Ah"] >= df["Q_low_Ah"] - 1e-12)
    & (df["Q_Ah"] <= df["Q_hi_Ah"] + 1e-12)
)
df["Q_window_valid_row"] = (
    np.isfinite(df["Q_low_Ah"])
    & np.isfinite(df["Q_hi_Ah"])
    & (df["Q_hi_Ah"] > df["Q_low_Ah"])
)

# Group audit
group_cols = [
    "param_set", "chem_tag", "condition", "pair_role",
    "DC_C", "AC_C", "tau_label", "Q_nom_Ah", "Q_low_Ah", "Q_hi_Ah",
]

rows = []

for keys, g in df.groupby(group_cols, dropna=False, sort=False):
    group = dict(zip(group_cols, keys))

    q = g["Q_Ah"].to_numpy(dtype=float)
    dt_err = g["dtQ_sign_err_s"].to_numpy(dtype=float)

    finite_core = (
        np.isfinite(g["Q_Ah"])
        & np.isfinite(g["t_DC_s"])
        & np.isfinite(g["t_protocol_s"])
        & np.isfinite(g["dtQ_s"])
    )

    q_sorted = np.all(np.diff(np.sort(q[np.isfinite(q)])) >= -1e-12) if np.isfinite(q).any() else False

    q_low = group["Q_low_Ah"]
    q_hi = group["Q_hi_Ah"]
    q_nom = group["Q_nom_Ah"]

    row = {
        **group,

        "n_rows": len(g),
        "n_finite_core": int(finite_core.sum()),
        "n_nonfinite_core": int((~finite_core).sum()),

        "Q_min_Ah": float(np.nanmin(q)),
        "Q_max_Ah": float(np.nanmax(q)),
        "Q_span_Ah": float(np.nanmax(q) - np.nanmin(q)),
        "Q_grid_unique": int(pd.Series(q).dropna().nunique()),
        "Q_sorted_if_sorted": bool(q_sorted),

        "Q_inside_window_all": bool(g["Q_inside_window"].all()),
        "Q_window_valid_all": bool(g["Q_window_valid_row"].all()),
        "Q_low_Ah_unique": int(g["Q_low_Ah"].nunique(dropna=True)),
        "Q_hi_Ah_unique": int(g["Q_hi_Ah"].nunique(dropna=True)),
        "Q_nom_Ah_unique": int(g["Q_nom_Ah"].nunique(dropna=True)),
        "Q_hi_gt_Q_low": bool(q_hi > q_low) if np.isfinite(q_hi) and np.isfinite(q_low) else False,
        "Q_hi_le_Q_nom": bool(q_hi <= q_nom) if np.isfinite(q_hi) and np.isfinite(q_nom) else False,

        "dtQ_sign_err_max_abs_s": float(np.nanmax(np.abs(dt_err))),
        "dtQ_sign_err_median_s": float(np.nanmedian(dt_err)),
        "dtQ_internal_sign_ok": bool(np.nanmax(np.abs(dt_err)) < 1e-6),

        "t_DC_median_s": float(np.nanmedian(g["t_DC_s"])),
        "t_protocol_median_s": float(np.nanmedian(g["t_protocol_s"])),
        "dtQ_median_s": float(np.nanmedian(g["dtQ_s"])),

        "provenance_confirms_shared_window": provenance_confirms_shared_window,
        "notebook_window_cells_found": len(prov_df),
    }

    # Compatibility classification
    if row["n_nonfinite_core"] > 0:
        compatibility = "incompatible_nonfinite_core"
    elif not row["dtQ_internal_sign_ok"]:
        compatibility = "incompatible_dt_sign"
    elif not row["Q_window_valid_all"] or not row["Q_hi_gt_Q_low"]:
        compatibility = "incompatible_invalid_window"
    elif not row["Q_inside_window_all"]:
        compatibility = "incompatible_Q_outside_window"
    elif not row["Q_hi_le_Q_nom"]:
        compatibility = "needs_review_Q_hi_exceeds_Q_nom"
    elif provenance_confirms_shared_window:
        compatibility = "compatible"
    else:
        compatibility = "needs_recompute_or_provenance"

    row["window_compatibility"] = compatibility

    rows.append(row)

audit_df = pd.DataFrame(rows)

out_audit = DATA / "day19A_step4_day16_window_audit.csv"
audit_df.to_csv(out_audit, index=False)

print(f"\nWrote: {out_audit}")
print(f"Groups audited: {len(audit_df)}")

print("\nCompatibility counts:")
print(audit_df["window_compatibility"].value_counts(dropna=False).to_string())

display_cols = [
    "param_set", "chem_tag", "condition", "pair_role",
    "DC_C", "AC_C", "tau_label",
    "n_rows", "Q_min_Ah", "Q_max_Ah", "Q_low_Ah", "Q_hi_Ah",
    "Q_inside_window_all", "dtQ_internal_sign_ok",
    "provenance_confirms_shared_window",
    "window_compatibility",
]

display(audit_df[display_cols].head(60))

# Do not hard-fail if provenance is missing; this is a gate.
assert len(audit_df) > 0, "No Day16 groups audited."

print("\nCell 6A PASSED — Day16 / nb20 Q-window compatibility gate complete.")

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step4_day16_related_files.csv


,path,name,suffixes,size_kb,mtime
0,data/day16_step1_axis_a_audit.csv,day16_step1_axis_a_audit.csv,.csv,9.85,2026-05-03 12:29
1,data/day16_step2_feasibility_matrix_rev3.csv,day16_step2_feasibility_matrix_rev3.csv,.csv,13.93,2026-05-03 13:21
2,data/day16_step2_trajectories_rev3.npz,day16_step2_trajectories_rev3.npz,.npz,3766.30,2026-05-03 13:21
3,data/day16_step3a_dtQ_curves_long.csv.gz,day16_step3a_dtQ_curves_long.csv.gz,.csv.gz,129.47,2026-05-03 13:25
4,data/day16_step3a_dtQ_table.csv,day16_step3a_dtQ_table.csv,.csv,13.40,2026-05-03 13:25
5,notebooks/20_param_set_wiring_audit.ipynb,20_param_set_wiring_audit.ipynb,.ipynb,139.86,2026-05-03 13:52



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step4_day16_notebook_window_provenance.csv


,cell_index,hits,contains_Q_hi,contains_Q_low,contains_Q_to_Vmax,contains_min,contains_safety_margin,contains_shared_or_window
0,1,"[""Vmax"", ""V_max"", ""voltage""]",False,False,False,False,False,False
1,4,"[""Q_to_Vmax"", ""Vmax"", ""V_max"", ""voltage"", ""min(""]",False,False,True,True,False,False
2,5,"[""V_max"", ""voltage"", ""min("", ""minimum""]",False,False,False,True,False,False
3,6,"[""Vmax"", ""V_max"", ""voltage"", ""min("", ""minimum""]",False,False,False,True,False,False
4,7,"[""Q_hi"", ""Q_low"", ""Q_to_Vmax"", ""Q_to_Vmax_DC"",...",True,True,True,True,False,True
5,8,"[""Vmax"", ""min("", ""window""]",False,False,False,True,False,True



Notebook provenance summary:
  provenance_confirms_shared_window = True
  cells_with_window_hits            = 6

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step4_day16_window_audit.csv
Groups audited: 36

Compatibility counts:
window_compatibility
compatible                         25
needs_review_Q_hi_exceeds_Q_nom    11


,param_set,chem_tag,condition,pair_role,DC_C,AC_C,tau_label,n_rows,Q_min_Ah,Q_max_Ah,Q_low_Ah,Q_hi_Ah,Q_inside_window_all,dtQ_internal_sign_ok,provenance_confirms_shared_window,window_compatibility
0,Ai2020,Enertech LCO/graphite,DC_0p2C,DC_sanity,0.2,0.0,0.0,100,0.114000,2.244355,0.114000,2.244355,True,True,True,compatible
1,Ai2020,Enertech LCO/graphite,DC0p2_AC0p5_1tau,DCAC,0.2,0.5,1.0,100,0.114000,2.119968,0.114000,2.119968,True,True,True,compatible
2,Ai2020,Enertech LCO/graphite,DC0p2_AC0p5_10tau,DCAC,0.2,0.5,10.0,100,0.114000,2.102389,0.114000,2.102389,True,True,True,compatible
3,Ai2020,Enertech LCO/graphite,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,100,0.114000,2.017151,0.114000,2.017151,True,True,True,compatible
4,Ai2020,Enertech LCO/graphite,DC0p2_AC1p0_10tau,DCAC,0.2,1.0,10.0,100,0.114000,1.974823,0.114000,1.974823,True,True,True,compatible
5,Chen2020,LG M50 lineage,DC_0p2C,DC_sanity,0.2,0.0,0.0,100,0.250000,4.509473,0.250000,4.509473,True,True,True,compatible
6,Chen2020,LG M50 lineage,DC0p2_AC0p5_1tau,DCAC,0.2,0.5,1.0,100,0.250000,3.931157,0.250000,3.931157,True,True,True,compatible
7,Chen2020,LG M50 lineage,DC0p2_AC0p5_10tau,DCAC,0.2,0.5,10.0,100,0.250000,3.800601,0.250000,3.800601,True,True,True,compatible
8,Chen2020,LG M50 lineage,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,100,0.250000,3.617658,0.250000,3.617658,True,True,True,compatible
9,Ecker2015,Kokam-Ecker Co-rich Mn-free,DC_0p2C,DC_sanity,0.2,0.0,0.0,100,0.007812,0.157226,0.007812,0.157226,True,True,True,needs_review_Q_hi_exceeds_Q_nom



Cell 6A PASSED — Day16 / nb20 Q-window compatibility gate complete.


In [34]:
# Cell 6A.1 — refine Day16 window compatibility for capacity-scale warnings
#
# Input:
#   data/day19A_step4_day16_window_audit.csv
#
# Output:
#   data/day19A_step4_day16_window_audit_refined.csv
#
# Purpose:
#   Convert groups with Q_hi > Q_nom but otherwise valid shared windows into
#   compatible_with_capacity_scale_warning rather than excluding them.

import numpy as np
import pandas as pd

audit_path = DATA / "day19A_step4_day16_window_audit.csv"
assert audit_path.exists(), f"Missing: {audit_path}"

audit = pd.read_csv(audit_path)

def refine_window_status(row):
    status = row["window_compatibility"]

    if status == "compatible":
        return "compatible"

    if status == "needs_review_Q_hi_exceeds_Q_nom":
        if (
            bool(row["Q_inside_window_all"])
            and bool(row["dtQ_internal_sign_ok"])
            and bool(row["provenance_confirms_shared_window"])
            and bool(row["Q_window_valid_all"])
            and bool(row["Q_hi_gt_Q_low"])
        ):
            return "compatible_with_capacity_scale_warning"

    return status

audit["window_compatibility_refined"] = audit.apply(refine_window_status, axis=1)

audit["capacity_scale_warning"] = audit["window_compatibility"].eq("needs_review_Q_hi_exceeds_Q_nom")

out = DATA / "day19A_step4_day16_window_audit_refined.csv"
audit.to_csv(out, index=False)

print(f"Wrote: {out}")

print("\nOriginal compatibility counts:")
print(audit["window_compatibility"].value_counts(dropna=False).to_string())

print("\nRefined compatibility counts:")
print(audit["window_compatibility_refined"].value_counts(dropna=False).to_string())

display(
    audit[
        [
            "param_set", "chem_tag", "condition", "pair_role",
            "DC_C", "AC_C", "tau_label",
            "Q_low_Ah", "Q_hi_Ah", "Q_nom_Ah",
            "Q_inside_window_all",
            "dtQ_internal_sign_ok",
            "provenance_confirms_shared_window",
            "window_compatibility",
            "window_compatibility_refined",
            "capacity_scale_warning",
        ]
    ].sort_values(["window_compatibility_refined", "param_set", "condition"])
)

# Hard checks for entering 6B
allowed = {
    "compatible",
    "compatible_with_capacity_scale_warning",
}
not_allowed = audit[~audit["window_compatibility_refined"].isin(allowed)]

assert not_allowed.empty, (
    "Some Day16 groups still not eligible for 6B:\n"
    f"{not_allowed[['param_set', 'condition', 'window_compatibility_refined']]}"
)

print("\nCell 6A.1 PASSED — all Day16 groups eligible for 6B, with warnings where needed.")

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step4_day16_window_audit_refined.csv

Original compatibility counts:
window_compatibility
compatible                         25
needs_review_Q_hi_exceeds_Q_nom    11

Refined compatibility counts:
window_compatibility_refined
compatible                                25
compatible_with_capacity_scale_warning    11


,param_set,chem_tag,condition,pair_role,DC_C,AC_C,tau_label,Q_low_Ah,Q_hi_Ah,Q_nom_Ah,Q_inside_window_all,dtQ_internal_sign_ok,provenance_confirms_shared_window,window_compatibility,window_compatibility_refined,capacity_scale_warning
2,Ai2020,Enertech LCO/graphite,DC0p2_AC0p5_10tau,DCAC,0.2,0.5,10.0,0.114000,2.102389,2.280000,True,True,True,compatible,compatible,False
1,Ai2020,Enertech LCO/graphite,DC0p2_AC0p5_1tau,DCAC,0.2,0.5,1.0,0.114000,2.119968,2.280000,True,True,True,compatible,compatible,False
4,Ai2020,Enertech LCO/graphite,DC0p2_AC1p0_10tau,DCAC,0.2,1.0,10.0,0.114000,1.974823,2.280000,True,True,True,compatible,compatible,False
3,Ai2020,Enertech LCO/graphite,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,0.114000,2.017151,2.280000,True,True,True,compatible,compatible,False
0,Ai2020,Enertech LCO/graphite,DC_0p2C,DC_sanity,0.2,0.0,0.0,0.114000,2.244355,2.280000,True,True,True,compatible,compatible,False
7,Chen2020,LG M50 lineage,DC0p2_AC0p5_10tau,DCAC,0.2,0.5,10.0,0.250000,3.800601,5.000000,True,True,True,compatible,compatible,False
6,Chen2020,LG M50 lineage,DC0p2_AC0p5_1tau,DCAC,0.2,0.5,1.0,0.250000,3.931157,5.000000,True,True,True,compatible,compatible,False
8,Chen2020,LG M50 lineage,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,0.250000,3.617658,5.000000,True,True,True,compatible,compatible,False
5,Chen2020,LG M50 lineage,DC_0p2C,DC_sanity,0.2,0.0,0.0,0.250000,4.509473,5.000000,True,True,True,compatible,compatible,False
11,Ecker2015,Kokam-Ecker Co-rich Mn-free,DC0p2_AC0p5_10tau,DCAC,0.2,0.5,10.0,0.007812,0.153984,0.156250,True,True,True,compatible,compatible,False



Cell 6A.1 PASSED — all Day16 groups eligible for 6B, with warnings where needed.


In [35]:
# Cell 6B — Day16 / nb20 stored-first-passage residual decomposition
#
# Inputs:
#   data/day16_step3a_dtQ_curves_long.csv.gz
#   data/day19A_step4_day16_window_audit_refined.csv
#
# Outputs:
#   data/day19A_step4_day16_dt_resid_curves.csv
#   data/day19A_step4_day16_dt_resid_summary.csv
#   data/day19A_step4_day16_dt_resid_errors.csv
#
# Purpose:
#   Decompose Day16 / nb20 stored Δt(Q) into:
#       dt_model_s = stored dtQ_s
#       dt_geom_s  = waveform-only geometry baseline
#       dt_resid_s = dt_model_s − dt_geom_s
#
# Evidence level:
#   This is NOT raw-trajectory residual audit. Day16 curves_long contains stored
#   first-passage times but no raw current trajectory. Therefore this is:
#       stored-first-passage residual audit
#
# Historical phase:
#   Day16 / nb20 belongs to the historical discharge-first implementation lineage:
#       I_py = -|I_DC| + |I_AC| sin(ωt)
#
# Frequency basis:
#   Day16 is fixed-label / MJ1-label convention:
#       f = 1 / (2π · tau_label · 11.1 s)
#
# DC_sanity rows:
#   Retained as reference/sanity rows but excluded from DCAC mechanism verdict.

import numpy as np
import pandas as pd

DAY16_CURVES = DATA / "day16_step3a_dtQ_curves_long.csv.gz"
DAY16_AUDIT = DATA / "day19A_step4_day16_window_audit_refined.csv"

assert DAY16_CURVES.exists(), f"Missing: {DAY16_CURVES}"
assert DAY16_AUDIT.exists(), f"Missing: {DAY16_AUDIT}"

TAU_LABEL_FIXED_S = 11.1
PHASE_DAY16 = PHASE_DISCHARGE_FIRST
FP_MODE_DAY16 = FP_MODE_RAW
DT_EVAL_GEOM_S = 0.1

RESID_NULL_THR_S = 1.0
RESID_SMALL_THR_S = 10.0
EDGE_FRAC = 0.05
DISTRIBUTED_FRAC_THR = 0.20

ALLOWED_WINDOW_STATUSES = {
    "compatible",
    "compatible_with_capacity_scale_warning",
}

# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def tau_label_to_f_Hz(tau_label):
    tau_label = float(tau_label)
    if tau_label <= 0:
        return np.nan
    return 1.0 / (2.0 * np.pi * tau_label * TAU_LABEL_FIXED_S)


def sign_topology_local(x, threshold_s=1.0):
    x = np.asarray(x, dtype=float)
    finite = np.isfinite(x)

    if not finite.any():
        return "invalid"

    xx = x[finite]
    pos = np.any(xx > threshold_s)
    neg = np.any(xx < -threshold_s)

    if pos and neg:
        return "mixed"
    if pos:
        return "positive_only"
    if neg:
        return "negative_only"
    return "near_zero"


def classify_residual_max(max_abs_resid_s):
    if not np.isfinite(max_abs_resid_s):
        return "invalid_residual"
    if max_abs_resid_s < RESID_NULL_THR_S:
        return "null_residual"
    if max_abs_resid_s < RESID_SMALL_THR_S:
        return "small_residual"
    return "inspect_residual"


def classify_stable_region(resid_s, edge_frac=EDGE_FRAC):
    resid_s = np.asarray(resid_s, dtype=float)
    finite = np.isfinite(resid_s)

    if not finite.any():
        return {
            "stable_p95_abs_s": np.nan,
            "stable_max_abs_s": np.nan,
            "stable_frac_gt_10s": np.nan,
            "stable_region_verdict": "invalid",
        }

    rr = resid_s[finite]
    n = len(rr)

    if n >= 20:
        lo = int(np.ceil(edge_frac * n))
        hi = int(np.floor((1.0 - edge_frac) * n))
        rr_stable = rr[lo:hi]
    else:
        rr_stable = rr

    if len(rr_stable) == 0:
        rr_stable = rr

    abs_stable = np.abs(rr_stable)

    stable_p95 = float(np.nanquantile(abs_stable, 0.95))
    stable_max = float(np.nanmax(abs_stable))
    stable_frac_gt_10 = float(np.mean(abs_stable > RESID_SMALL_THR_S))

    if stable_p95 < RESID_NULL_THR_S:
        verdict = "stable_null"
    elif stable_p95 < RESID_SMALL_THR_S:
        verdict = "stable_weak"
    elif stable_frac_gt_10 >= DISTRIBUTED_FRAC_THR:
        verdict = "stable_distributed_nonzero"
    else:
        verdict = "stable_intermediate_or_spiky"

    return {
        "stable_p95_abs_s": stable_p95,
        "stable_max_abs_s": stable_max,
        "stable_frac_gt_10s": stable_frac_gt_10,
        "stable_region_verdict": verdict,
    }


def classify_pair_taxonomy(resid_s):
    resid_s = np.asarray(resid_s, dtype=float)
    finite = np.isfinite(resid_s)

    if not finite.any():
        return "invalid_no_finite_points"

    rr = resid_s[finite]
    abs_r = np.abs(rr)
    n = len(abs_r)

    max_abs = float(np.nanmax(abs_r))
    p95_abs = float(np.nanquantile(abs_r, 0.95))
    frac_gt_10 = float(np.mean(abs_r > RESID_SMALL_THR_S))

    max_i = int(np.nanargmax(abs_r))
    max_frac = max_i / max(n - 1, 1)
    edge_hit = bool(max_frac <= EDGE_FRAC or max_frac >= 1.0 - EDGE_FRAC)

    if p95_abs < RESID_NULL_THR_S:
        return "null_residual"
    if max_abs >= RESID_SMALL_THR_S and frac_gt_10 <= 0.05:
        return "isolated_spike"
    if edge_hit and frac_gt_10 <= DISTRIBUTED_FRAC_THR:
        return "edge_or_boundary_artifact"
    if p95_abs < RESID_SMALL_THR_S:
        return "weak_distributed_residual"
    if frac_gt_10 >= DISTRIBUTED_FRAC_THR:
        return "distributed_residual_requires_inspection"
    return "mixed_or_intermediate"


def safe_ratio_abs(num, den, eps=1e-12):
    num = np.asarray(num, dtype=float)
    den = np.asarray(den, dtype=float)

    out = np.full_like(num, np.nan, dtype=float)
    mask = np.isfinite(num) & np.isfinite(den) & (np.abs(den) > eps)
    out[mask] = np.abs(num[mask]) / np.abs(den[mask])
    return out


# ---------------------------------------------------------------------
# Load + join refined window audit
# ---------------------------------------------------------------------

curves = pd.read_csv(DAY16_CURVES, compression="gzip")
audit = pd.read_csv(DAY16_AUDIT)

required_curves = {
    "param_set", "chem_tag", "condition", "pair_role",
    "DC_C", "AC_C", "tau_label",
    "Q_Ah", "t_DC_s", "t_protocol_s", "dtQ_s",
    "Q_nom_Ah", "Q_low_Ah", "Q_hi_Ah",
}
required_audit = {
    "param_set", "chem_tag", "condition", "pair_role",
    "DC_C", "AC_C", "tau_label",
    "Q_nom_Ah", "Q_low_Ah", "Q_hi_Ah",
    "window_compatibility_refined",
    "capacity_scale_warning",
}

missing_curves = required_curves - set(curves.columns)
missing_audit = required_audit - set(audit.columns)

assert not missing_curves, f"Day16 curves missing columns: {missing_curves}"
assert not missing_audit, f"Day16 refined audit missing columns: {missing_audit}"

join_keys = [
    "param_set", "chem_tag", "condition", "pair_role",
    "DC_C", "AC_C", "tau_label",
    "Q_nom_Ah", "Q_low_Ah", "Q_hi_Ah",
]

audit_meta = audit[
    join_keys
    + [
        "window_compatibility",
        "window_compatibility_refined",
        "capacity_scale_warning",
        "provenance_confirms_shared_window",
        "dtQ_internal_sign_ok",
        "Q_inside_window_all",
    ]
].drop_duplicates(join_keys)

merged = curves.merge(
    audit_meta,
    on=join_keys,
    how="left",
    validate="many_to_one",
    indicator=True,
)

missing_join = merged[merged["_merge"] != "both"]
assert missing_join.empty, (
    "Some Day16 curve rows failed to join refined audit:\n"
    f"{missing_join[join_keys].drop_duplicates().head(20)}"
)

merged = merged.drop(columns=["_merge"])

eligible = merged[
    merged["window_compatibility_refined"].isin(ALLOWED_WINDOW_STATUSES)
].copy()

assert len(eligible) == len(merged), (
    "Some Day16 rows not eligible for 6B:\n"
    f"{merged[~merged['window_compatibility_refined'].isin(ALLOWED_WINDOW_STATUSES)][join_keys + ['window_compatibility_refined']].drop_duplicates()}"
)

# Numeric coercion
num_cols = [
    "DC_C", "AC_C", "tau_label", "Q_Ah",
    "t_DC_s", "t_protocol_s", "dtQ_s",
    "Q_nom_Ah", "Q_low_Ah", "Q_hi_Ah",
]
for c in num_cols:
    eligible[c] = pd.to_numeric(eligible[c], errors="coerce")


# ---------------------------------------------------------------------
# Main decomposition
# ---------------------------------------------------------------------

group_cols = join_keys

curve_rows = []
summary_rows = []
error_rows = []

for keys, g in eligible.groupby(group_cols, dropna=False, sort=False):
    meta = dict(zip(group_cols, keys))

    condition = str(meta["condition"])
    pair_role = str(meta["pair_role"])
    AC_C = float(meta["AC_C"])
    DC_C = float(meta["DC_C"])
    tau_label = float(meta["tau_label"])
    Q_nom_Ah = float(meta["Q_nom_Ah"])

    Q_grid_mAh = pd.to_numeric(g["Q_Ah"], errors="coerce").to_numpy(dtype=float) * 1000.0
    dt_model_s = pd.to_numeric(g["dtQ_s"], errors="coerce").to_numpy(dtype=float)

    try:
        if pair_role == "DC_sanity" or AC_C == 0.0:
            # DC sanity: no AC geometry. Keep row as reference; not mechanism evidence.
            dt_geom_s = np.zeros_like(dt_model_s)
            t_DC_geom_s = np.full_like(dt_model_s, np.nan, dtype=float)
            t_DCAC_geom_s = np.full_like(dt_model_s, np.nan, dtype=float)
            phase_used = "dc_sanity_no_ac"
            f_Hz = np.nan
            I_DC_A = DC_C * Q_nom_Ah
            I_AC_A = 0.0
            geom_metadata = {
                "t_max_s": np.nan,
                "Q_DC_final_mAh": np.nan,
                "Q_DCAC_final_mAh": np.nan,
                "n_t_points": np.nan,
            }
        else:
            f_Hz = tau_label_to_f_Hz(tau_label)
            if not np.isfinite(f_Hz) or f_Hz <= 0:
                raise ValueError(f"Invalid f_Hz from tau_label={tau_label}")

            I_DC_A = DC_C * Q_nom_Ah
            I_AC_A = AC_C * Q_nom_Ah
            phase_used = PHASE_DAY16

            geom = dt_geom_from_protocol(
                I_DC_A=I_DC_A,
                I_AC_A=I_AC_A,
                f_Hz=f_Hz,
                phase=phase_used,
                Q_grid_mAh=Q_grid_mAh,
                dt_eval_s=DT_EVAL_GEOM_S,
                fp_mode=FP_MODE_DAY16,
            )

            dt_geom_s = np.asarray(geom["dt_geom_s"], dtype=float)
            t_DC_geom_s = np.asarray(geom["t_DC_s"], dtype=float)
            t_DCAC_geom_s = np.asarray(geom["t_DCAC_s"], dtype=float)
            geom_metadata = geom["metadata"]

        dt_resid_s = dt_model_s - dt_geom_s
        finite = np.isfinite(dt_model_s) & np.isfinite(dt_geom_s) & np.isfinite(dt_resid_s)

        resid_abs = np.abs(dt_resid_s[finite])
        resid_over_geom = safe_ratio_abs(dt_resid_s[finite], dt_geom_s[finite])
        resid_over_model = safe_ratio_abs(dt_resid_s[finite], dt_model_s[finite])

        stable = classify_stable_region(dt_resid_s)
        taxonomy = classify_pair_taxonomy(dt_resid_s)

        max_abs_resid = float(np.nanmax(resid_abs)) if len(resid_abs) else np.nan
        residual_class = (
            "dc_sanity_reference"
            if pair_role == "DC_sanity" or AC_C == 0.0
            else classify_residual_max(max_abs_resid)
        )

        # Long rows
        for j, (_, r) in enumerate(g.reset_index(drop=True).iterrows()):
            curve_rows.append({
                "param_set": meta["param_set"],
                "chem_tag": meta["chem_tag"],
                "condition": meta["condition"],
                "pair_role": pair_role,
                "DC_C": DC_C,
                "AC_C": AC_C,
                "tau_label": tau_label,
                "phase": phase_used,

                "Q_Ah": float(r["Q_Ah"]),
                "Q_mAh": float(r["Q_Ah"] * 1000.0),
                "t_DC_model_s": float(r["t_DC_s"]),
                "t_DCAC_model_s": float(r["t_protocol_s"]),
                "dt_model_s": float(dt_model_s[j]),

                "t_DC_geom_s": float(t_DC_geom_s[j]) if np.isfinite(t_DC_geom_s[j]) else np.nan,
                "t_DCAC_geom_s": float(t_DCAC_geom_s[j]) if np.isfinite(t_DCAC_geom_s[j]) else np.nan,
                "dt_geom_s": float(dt_geom_s[j]),
                "dt_resid_s": float(dt_resid_s[j]),

                "I_DC_A_used": float(I_DC_A),
                "I_AC_A_used": float(I_AC_A),
                "f_Hz_used": float(f_Hz) if np.isfinite(f_Hz) else np.nan,
                "kappa_used": float(I_AC_A / I_DC_A) if I_DC_A > 0 else np.nan,

                "Q_nom_Ah": Q_nom_Ah,
                "Q_low_Ah": float(meta["Q_low_Ah"]),
                "Q_hi_Ah": float(meta["Q_hi_Ah"]),
                "window_compatibility_refined": r["window_compatibility_refined"],
                "capacity_scale_warning": bool(r["capacity_scale_warning"]),

                "fp_mode": FP_MODE_DAY16,
                "dt_eval_geom_s": DT_EVAL_GEOM_S,
                "finite": bool(finite[j]),
                "evidence_type": "stored_first_passage_residual_audit",
            })

        # Summary row
        summary_rows.append({
            "param_set": meta["param_set"],
            "chem_tag": meta["chem_tag"],
            "condition": meta["condition"],
            "pair_role": pair_role,
            "DC_C": DC_C,
            "AC_C": AC_C,
            "tau_label": tau_label,
            "phase": phase_used,
            "evidence_type": "stored_first_passage_residual_audit",

            "N_Q": len(g),
            "n_finite": int(finite.sum()),
            "n_nonfinite": int((~finite).sum()),

            "Q_nom_Ah": Q_nom_Ah,
            "Q_low_Ah": float(meta["Q_low_Ah"]),
            "Q_hi_Ah": float(meta["Q_hi_Ah"]),
            "window_compatibility_refined": g["window_compatibility_refined"].iloc[0],
            "capacity_scale_warning": bool(g["capacity_scale_warning"].iloc[0]),

            "I_DC_A_used": float(I_DC_A),
            "I_AC_A_used": float(I_AC_A),
            "f_Hz_used": float(f_Hz) if np.isfinite(f_Hz) else np.nan,
            "kappa_used": float(I_AC_A / I_DC_A) if I_DC_A > 0 else np.nan,

            "dt_model_mean_s": float(np.nanmean(dt_model_s)),
            "dt_model_median_s": float(np.nanmedian(dt_model_s)),
            "dt_model_min_s": float(np.nanmin(dt_model_s)),
            "dt_model_max_s": float(np.nanmax(dt_model_s)),
            "dt_model_sign_topology": sign_topology_local(dt_model_s),

            "dt_geom_mean_s": float(np.nanmean(dt_geom_s)),
            "dt_geom_median_s": float(np.nanmedian(dt_geom_s)),
            "dt_geom_min_s": float(np.nanmin(dt_geom_s)),
            "dt_geom_max_s": float(np.nanmax(dt_geom_s)),
            "dt_geom_sign_topology": sign_topology_local(dt_geom_s),

            "dt_resid_mean_s": float(np.nanmean(dt_resid_s)),
            "dt_resid_median_s": float(np.nanmedian(dt_resid_s)),
            "dt_resid_min_s": float(np.nanmin(dt_resid_s)),
            "dt_resid_max_s": float(np.nanmax(dt_resid_s)),
            "dt_resid_mean_abs_s": float(np.nanmean(resid_abs)) if len(resid_abs) else np.nan,
            "dt_resid_median_abs_s": float(np.nanmedian(resid_abs)) if len(resid_abs) else np.nan,
            "dt_resid_p95_abs_s": float(np.nanquantile(resid_abs, 0.95)) if len(resid_abs) else np.nan,
            "dt_resid_max_abs_s": max_abs_resid,
            "dt_resid_sign_topology": sign_topology_local(dt_resid_s),

            "resid_over_geom_median": float(np.nanmedian(resid_over_geom)) if len(resid_over_geom) else np.nan,
            "resid_over_geom_p95": float(np.nanquantile(resid_over_geom, 0.95)) if len(resid_over_geom) else np.nan,
            "resid_over_geom_max": float(np.nanmax(resid_over_geom)) if len(resid_over_geom) else np.nan,

            "resid_over_model_median": float(np.nanmedian(resid_over_model)) if len(resid_over_model) else np.nan,
            "resid_over_model_p95": float(np.nanquantile(resid_over_model, 0.95)) if len(resid_over_model) else np.nan,
            "resid_over_model_max": float(np.nanmax(resid_over_model)) if len(resid_over_model) else np.nan,

            "residual_class": residual_class,
            "taxonomy": taxonomy,
            **stable,

            "t_max_s_geom": geom_metadata["t_max_s"],
            "Q_DC_final_mAh_geom": geom_metadata["Q_DC_final_mAh"],
            "Q_DCAC_final_mAh_geom": geom_metadata["Q_DCAC_final_mAh"],
            "n_t_points_geom": geom_metadata["n_t_points"],
            "fp_mode": FP_MODE_DAY16,
            "dt_eval_geom_s": DT_EVAL_GEOM_S,
        })

    except Exception as e:
        error_rows.append({
            **meta,
            "error": str(e)[:800],
        })

# ---------------------------------------------------------------------
# Build dataframes + persist
# ---------------------------------------------------------------------

curves_out = pd.DataFrame(curve_rows)
summary_out = pd.DataFrame(summary_rows)
errors_out = pd.DataFrame(error_rows)

out_curves = DATA / "day19A_step4_day16_dt_resid_curves.csv"
out_summary = DATA / "day19A_step4_day16_dt_resid_summary.csv"
out_errors = DATA / "day19A_step4_day16_dt_resid_errors.csv"

curves_out.to_csv(out_curves, index=False)
summary_out.to_csv(out_summary, index=False)
errors_out.to_csv(out_errors, index=False)

print(f"Wrote: {out_curves} ({len(curves_out)} rows)")
print(f"Wrote: {out_summary} ({len(summary_out)} rows)")
print(f"Wrote: {out_errors} ({len(errors_out)} rows)")

print("\nDecomposition status:")
print(f"  groups requested : {eligible[group_cols].drop_duplicates().shape[0]}")
print(f"  groups decomposed: {len(summary_out)}")
print(f"  errors           : {len(errors_out)}")

if len(errors_out):
    display(errors_out)

print("\nResidual class counts:")
print(summary_out["residual_class"].value_counts(dropna=False).to_string())

print("\nStable-region verdict counts:")
print(summary_out["stable_region_verdict"].value_counts(dropna=False).to_string())

print("\nTaxonomy counts:")
print(summary_out["taxonomy"].value_counts(dropna=False).to_string())

print("\nBy param_set and stable-region verdict:")
print(summary_out.groupby(["param_set", "stable_region_verdict"]).size().to_string())

print("\nTop 20 Day16 groups by stable p95 |residual|:")
display(
    summary_out.sort_values("stable_p95_abs_s", ascending=False)
    [
        [
            "param_set", "chem_tag", "condition", "pair_role",
            "DC_C", "AC_C", "tau_label", "phase",
            "capacity_scale_warning",
            "dt_model_median_s", "dt_geom_median_s",
            "dt_resid_median_s", "stable_p95_abs_s",
            "dt_resid_max_abs_s", "taxonomy", "stable_region_verdict",
        ]
    ]
    .head(20)
)

# ---------------------------------------------------------------------
# Hard checks
# ---------------------------------------------------------------------

assert len(errors_out) == 0, (
    "Some Day16 groups failed decomposition:\n"
    f"{errors_out.head(20)}"
)

expected_groups = eligible[group_cols].drop_duplicates().shape[0]
assert len(summary_out) == expected_groups, (
    f"Expected {expected_groups} decomposed groups, got {len(summary_out)}"
)

assert (summary_out["n_nonfinite"] == 0).all(), (
    "Some Day16 decompositions contain non-finite curve values:\n"
    f"{summary_out[summary_out['n_nonfinite'] != 0][['param_set', 'condition', 'n_nonfinite']]}"
)

print("\n" + "=" * 72)
print("Cell 6B PASSED — Day16 / nb20 stored-first-passage residual decomposition complete")
print("=" * 72)

/var/folders/cc/4060jtz167l1hw2178_m6pyh0000gn/T/ipykernel_25728/184765735.py:438: RuntimeWarning: All-NaN slice encountered
  "resid_over_geom_median": float(np.nanmedian(resid_over_geom)) if len(resid_over_geom) else np.nan,
/Users/louislu/pybamm-dcac-superimposed/.venv/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1573: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(
/var/folders/cc/4060jtz167l1hw2178_m6pyh0000gn/T/ipykernel_25728/184765735.py:440: RuntimeWarning: All-NaN slice encountered
  "resid_over_geom_max": float(np.nanmax(resid_over_geom)) if len(resid_over_geom) else np.nan,
/var/folders/cc/4060jtz167l1hw2178_m6pyh0000gn/T/ipykernel_25728/184765735.py:442: RuntimeWarning: All-NaN slice encountered
  "resid_over_model_median": float(np.nanmedian(resid_over_model)) if len(resid_over_model) else np.nan,
/var/folders/cc/4060jtz167l1hw2178_m6pyh0000gn/T/ipykernel_25728/184765735.py:444: RuntimeWarning: All-NaN slice encountered
  "resid_

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step4_day16_dt_resid_curves.csv (3600 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step4_day16_dt_resid_summary.csv (36 rows)
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step4_day16_dt_resid_errors.csv (0 rows)

Decomposition status:
  groups requested : 36
  groups decomposed: 36
  errors           : 0

Residual class counts:
residual_class
inspect_residual       21
dc_sanity_reference     8
small_residual          7

Stable-region verdict counts:
stable_region_verdict
stable_weak                     13
stable_null                     11
stable_intermediate_or_spiky     9
stable_distributed_nonzero       3

Taxonomy counts:
taxonomy
null_residual                               11
isolated_spike                               8
mixed_or_intermediate                        6
weak_distributed_residual                    6
edge_or_boundary_artifact                    3
distributed_residual_requires

,param_set,chem_tag,condition,pair_role,DC_C,AC_C,tau_label,phase,capacity_scale_warning,dt_model_median_s,dt_geom_median_s,dt_resid_median_s,stable_p95_abs_s,dt_resid_max_abs_s,taxonomy,stable_region_verdict
17,Marquis2019,Kokam-Marquis LCO/graphite,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,discharge_first,True,-42.281577,-23.263933,-5.100560,61.917295,63.130774,distributed_residual_requires_inspection,stable_distributed_nonzero
3,Ai2020,Enertech LCO/graphite,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,discharge_first,False,-38.061210,-22.893701,-3.531598,59.454005,60.903283,distributed_residual_requires_inspection,stable_distributed_nonzero
22,Mohtat2020,graphite/NMC532,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,discharge_first,False,-35.526022,-24.308529,-2.870397,58.207056,59.805546,mixed_or_intermediate,stable_intermediate_or_spiky
8,Chen2020,LG M50 lineage,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,discharge_first,False,-32.695640,-22.488447,-2.401852,57.555524,58.430780,mixed_or_intermediate,stable_intermediate_or_spiky
35,ORegan2022,LG M50 lineage,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,discharge_first,False,-27.886582,-22.941479,-0.934093,54.022276,55.942663,mixed_or_intermediate,stable_intermediate_or_spiky
31,OKane2022,LG M50 lineage,DC0p2_AC1p0_1tau,DCAC,0.2,1.0,1.0,discharge_first,False,-28.251597,-23.698262,-1.087780,53.506135,57.277768,edge_or_boundary_artifact,stable_intermediate_or_spiky
15,Marquis2019,Kokam-Marquis LCO/graphite,DC0p2_AC0p5_1tau,DCAC,0.2,0.5,1.0,discharge_first,True,-31.542249,-18.706704,-4.889044,52.659091,54.269052,edge_or_boundary_artifact,stable_distributed_nonzero
6,Chen2020,LG M50 lineage,DC0p2_AC0p5_1tau,DCAC,0.2,0.5,1.0,discharge_first,False,-28.838461,-18.417007,-3.719249,50.347197,52.686697,edge_or_boundary_artifact,stable_intermediate_or_spiky
29,OKane2022,LG M50 lineage,DC0p2_AC0p5_1tau,DCAC,0.2,0.5,1.0,discharge_first,False,-26.645940,-18.557657,-3.111632,49.555189,51.512142,mixed_or_intermediate,stable_intermediate_or_spiky
1,Ai2020,Enertech LCO/graphite,DC0p2_AC0p5_1tau,DCAC,0.2,0.5,1.0,discharge_first,False,-24.108271,-19.104661,-1.905715,45.559308,49.519712,mixed_or_intermediate,stable_intermediate_or_spiky



Cell 6B PASSED — Day16 / nb20 stored-first-passage residual decomposition complete


In [36]:
# Cell 7A — DCAC-vs-DCAC ablation phase audit
#
# Purpose:
#   Audit the phase lineage and evidential status of historical DCAC-vs-DCAC
#   ablation notebooks:
#     - nb14 / Day11: plating ablation
#     - nb15 / Day12: OCP scan
#     - nb16 / Day13: PE transport / kinetics scan
#     - nb18 / Day14 Task 1: AsymBV alpha scan, mixed lineage
#     - nb19 / Day14 Task 2: X6 clean test, discharge-first variants
#
# Outputs:
#   data/day19A_step5_ablation_notebook_phase_scan.csv
#   data/day19A_step5_ablation_output_inventory.csv
#   data/day19A_step5_nb18_phase_pair_inventory.csv
#   data/day19A_step5_ablation_phase_audit_summary.csv
#
# Interpretation:
#   Same-waveform DCAC-vs-DCAC ablations can retain internal validity because
#   Δt_geom cancels when baseline and ablation share the same current waveform.
#   Cross-notebook absolute Δt comparisons are invalid unless phase, Q-window,
#   sign convention, model topology, and output lineage are all matched.

import json
import re
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

notebook_specs = [
    {
        "notebook_id": "nb14",
        "notebook_file": "14_plating_ablation.ipynb",
        "task_label": "Day11 plating ablation",
        "expected_closure_lineage": "discharge_first",
        "comparison_type": "DCAC-vs-DCAC same-phase ablation",
        "notes": "Historical Day11 ablation lineage; expected nb16-style discharge-first convention.",
    },
    {
        "notebook_id": "nb15",
        "notebook_file": "15_ocp_scan.ipynb",
        "task_label": "Day12 OCP scan",
        "expected_closure_lineage": "discharge_first",
        "comparison_type": "DCAC-vs-DCAC same-phase ablation",
        "notes": "Historical Day12 ablation lineage; expected discharge-first convention.",
    },
    {
        "notebook_id": "nb16",
        "notebook_file": "16_pe_ocp_transport_kinetics_scan.ipynb",
        "task_label": "Day13 PE/OCP/transport/kinetics scan",
        "expected_closure_lineage": "discharge_first",
        "comparison_type": "DCAC-vs-DCAC same-phase ablation",
        "notes": "Historical Day13 branch that canonized nb16-aligned discharge-first convention.",
    },
    {
        "notebook_id": "nb18",
        "notebook_file": "18_asymbv_alpha_scan_day14_task1.ipynb",
        "task_label": "Day14 Task1 AsymBV alpha scan",
        "expected_closure_lineage": "mixed_v1_charge_first_v2_discharge_first",
        "comparison_type": "mixed phase lineage with paired v1/v2 evidence",
        "notes": "Cell 5 v1 charge-first was historically labelled wrong_phase; Cell 5 v2 nb16-aligned discharge-first was final closure.",
    },
    {
        "notebook_id": "nb19",
        "notebook_file": "19_x6_phase_clean_test_day14_task2.ipynb",
        "task_label": "Day14 Task2 X6 phase clean test",
        "expected_closure_lineage": "discharge_first",
        "comparison_type": "DCAC-vs-DCAC same-phase ablation with t_offset variants",
        "notes": "Expected discharge-first / nb16-aligned variants; t_offset audit may change waveform start but remains same-phase within output lineage.",
    },
]


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def read_notebook_source(path):
    with open(path, encoding="utf-8") as fh:
        nb = json.load(fh)

    cells = []
    for i, cell in enumerate(nb.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell.get("source", []))
        cells.append((i, src))
    return cells


def compact_excerpt(src, pos, span_before=500, span_after=1000):
    start = max(0, pos - span_before)
    end = min(len(src), pos + span_after)
    return src[start:end]


def detect_phase_signals(src):
    """
    Heuristic phase-pattern detector.

    Returns dict with charge/discharge signal counts and label hits.
    This is not the sole truth source; closure_lineage is interpreted with
    notebook/output history.
    """
    s = src
    sl = src.lower()

    charge_patterns = [
        r"-\s*\(\s*I_DC\s*\+\s*I_AC\s*\*.*sin",
        r"-\s*\(\s*i_dc\s*\+\s*i_ac\s*\*.*sin",
        r"-\s*I_DC\s*-\s*I_AC\s*\*.*sin",
        r"-\s*i_dc\s*-\s*i_ac\s*\*.*sin",
        r"charge[_ -]?first",
    ]

    discharge_patterns = [
        r"I_DC_A\s*\+\s*A_A\s*\*.*sin",
        r"I_DC_s\s*\+\s*A_s\s*\*.*sin",
        r"-\s*abs\s*\(.*DC.*\)\s*\+\s*abs\s*\(.*AC.*\).*sin",
        r"-\s*\|\s*I_DC\s*\|\s*\+\s*\|\s*I_AC\s*\|.*sin",
        r"discharge[_ -]?first",
        r"nb16[-_ ]aligned",
    ]

    label_patterns = [
        "wrong_phase",
        "aligned_phase",
        "phase-sensitivity",
        "phase sensitivity",
        "nb16-aligned",
        "charge-first",
        "discharge-first",
        "t_offset",
        "phase_offset",
    ]

    charge_hits = []
    discharge_hits = []
    label_hits = []

    for pat in charge_patterns:
        if re.search(pat, s, flags=re.IGNORECASE | re.DOTALL):
            charge_hits.append(pat)

    for pat in discharge_patterns:
        if re.search(pat, s, flags=re.IGNORECASE | re.DOTALL):
            discharge_hits.append(pat)

    for pat in label_patterns:
        if pat.lower() in sl:
            label_hits.append(pat)

    return {
        "charge_signal": len(charge_hits) > 0,
        "discharge_signal": len(discharge_hits) > 0,
        "charge_hit_patterns": charge_hits,
        "discharge_hit_patterns": discharge_hits,
        "label_hits": label_hits,
    }


def infer_observed_phase_from_scan(charge_count, discharge_count):
    if charge_count > 0 and discharge_count > 0:
        return "mixed"
    if charge_count > 0:
        return "charge_first"
    if discharge_count > 0:
        return "discharge_first"
    return "unknown"


def classify_ablation_evidence(row):
    closure = row["closure_lineage"]

    if row["notebook_id"] == "nb18":
        return (
            "mixed_lineage_phase_pair_evidence",
            "v1/v2 pair is valuable for phase-sensitivity evidence; final closure output is discharge-first nb16-aligned.",
            "cross_notebook_absolute_dt_invalid_without_output_lineage_tracing",
        )

    if closure == "discharge_first":
        return (
            "same_phase_ablation_valid_internal_only",
            "Within-notebook ablation verdict retains internal validity because current geometry cancels under same waveform.",
            "cross_notebook_absolute_dt_not_comparable_to_charge_first_mainline",
        )

    if closure == "charge_first":
        return (
            "same_phase_charge_first_ablation_valid_internal_only",
            "Within-notebook ablation verdict retains internal validity; check whether it is final output lineage.",
            "cross_notebook_absolute_dt_requires_phase_match",
        )

    return (
        "phase_lineage_uncertain",
        "Notebook scan did not establish a clear output-generating phase lineage.",
        "manual_review_required",
    )


# ---------------------------------------------------------------------
# Part A — notebook source scan
# ---------------------------------------------------------------------

scan_rows = []

for spec in notebook_specs:
    nb_path = NB_DIR / spec["notebook_file"]

    if not nb_path.exists():
        scan_rows.append({
            **spec,
            "notebook_path": str(nb_path.relative_to(REPO)) if nb_path.exists() else str(nb_path),
            "exists": False,
            "n_code_cells": 0,
            "n_phase_relevant_cells": 0,
            "charge_signal_cells": 0,
            "discharge_signal_cells": 0,
            "observed_phase_from_scan": "missing_notebook",
            "closure_lineage": spec["expected_closure_lineage"],
            "representative_excerpts": "",
        })
        continue

    cells = read_notebook_source(nb_path)

    phase_cells = []
    charge_cells = []
    discharge_cells = []
    excerpts = []

    for cell_idx, src in cells:
        relevant = (
            "sin(" in src
            or "Current function" in src
            or "current function" in src.lower()
            or "I_DC" in src
            or "I_AC" in src
            or "phase" in src.lower()
            or "wrong_phase" in src.lower()
            or "nb16" in src.lower()
        )

        if not relevant:
            continue

        signals = detect_phase_signals(src)
        if signals["charge_signal"] or signals["discharge_signal"] or signals["label_hits"]:
            phase_cells.append(cell_idx)

            if signals["charge_signal"]:
                charge_cells.append(cell_idx)
            if signals["discharge_signal"]:
                discharge_cells.append(cell_idx)

            # excerpt around first occurrence of sin / phase / wrong_phase / nb16
            positions = []
            for term in ["sin(", "phase", "wrong_phase", "nb16"]:
                pos = src.lower().find(term.lower())
                if pos >= 0:
                    positions.append(pos)
            pos = min(positions) if positions else 0

            excerpts.append({
                "cell_index": cell_idx,
                "signals": signals,
                "excerpt": compact_excerpt(src, pos),
            })

    observed = infer_observed_phase_from_scan(len(charge_cells), len(discharge_cells))

    # Closure lineage is not purely heuristic; use expected closure lineage from
    # manually verified notebook/output lineage.
    closure_lineage = spec["expected_closure_lineage"]

    evidence_status, evidence_note, comparability = classify_ablation_evidence({
        "notebook_id": spec["notebook_id"],
        "closure_lineage": closure_lineage,
    })

    scan_rows.append({
        **spec,
        "notebook_path": str(nb_path.relative_to(REPO)),
        "exists": True,
        "n_code_cells": len(cells),
        "n_phase_relevant_cells": len(phase_cells),
        "charge_signal_cells": len(set(charge_cells)),
        "discharge_signal_cells": len(set(discharge_cells)),
        "phase_relevant_cell_indices": json.dumps(sorted(set(phase_cells))),
        "charge_signal_cell_indices": json.dumps(sorted(set(charge_cells))),
        "discharge_signal_cell_indices": json.dumps(sorted(set(discharge_cells))),
        "observed_phase_from_scan": observed,
        "closure_lineage": closure_lineage,
        "evidence_status": evidence_status,
        "evidence_note": evidence_note,
        "cross_notebook_comparability": comparability,
        "representative_excerpts": json.dumps(excerpts[:8], ensure_ascii=False),
    })

scan_df = pd.DataFrame(scan_rows)

out_scan = DATA / "day19A_step5_ablation_notebook_phase_scan.csv"
scan_df.to_csv(out_scan, index=False)

print(f"Wrote: {out_scan}")
display(
    scan_df[
        [
            "notebook_id", "task_label", "exists",
            "n_phase_relevant_cells",
            "charge_signal_cells", "discharge_signal_cells",
            "observed_phase_from_scan", "closure_lineage",
            "evidence_status", "cross_notebook_comparability",
        ]
    ]
)


# ---------------------------------------------------------------------
# Part B — output inventory
# ---------------------------------------------------------------------

output_patterns_by_nb = {
    "nb14": [
        "results_day11*.csv",
        "day11*.csv",
        "*plating*.csv",
    ],
    "nb15": [
        "results_day12*.csv",
        "day12*.csv",
        "*OCP*.csv",
        "*ocp*.csv",
    ],
    "nb16": [
        "results_day13*.csv",
        "day13*.csv",
        "*PE_transport*.csv",
        "*transport_kinetics*.csv",
    ],
    "nb18": [
        "day14_step5_delta_tQ_curves*.csv",
        "results_day14*.csv",
        "*asymbv*.csv",
        "*AsymmetricBV*.csv",
    ],
    "nb19": [
        "*x6_phase*.csv",
        "*X6*.csv",
        "*x6*.csv",
        "*day15*x6*.csv",
        "*day14*x6*.csv",
        "*t_offset*.csv",
    ],
}

output_rows = []
seen = set()

for spec in notebook_specs:
    nbid = spec["notebook_id"]

    for pat in output_patterns_by_nb.get(nbid, []):
        for p in DATA.glob(pat):
            if not p.is_file():
                continue
            if p.name.startswith(("day19A_", "day19B_", "day19D_")):
                continue

            key = (nbid, p.resolve())
            if key in seen:
                continue
            seen.add(key)

            cols = []
            n_rows = np.nan
            read_ok = False
            error = ""

            try:
                df_head = pd.read_csv(p, compression="infer", nrows=5)
                cols = list(df_head.columns)
                read_ok = True
                try:
                    n_rows = sum(len(chunk) for chunk in pd.read_csv(p, compression="infer", chunksize=100000))
                except Exception:
                    n_rows = np.nan
            except Exception as e:
                error = str(e)[:300]

            phase_like_cols = [
                c for c in cols
                if "phase" in c.lower() or "offset" in c.lower()
            ]
            dt_like_cols = [
                c for c in cols
                if "dt" in c.lower() or "delta_t" in c.lower()
            ]
            protocol_like_cols = [
                c for c in cols
                if any(s in c.lower() for s in ["protocol", "condition", "case", "ablation"])
            ]

            output_rows.append({
                "notebook_id": nbid,
                "task_label": spec["task_label"],
                "file": p.name,
                "path": str(p.relative_to(REPO)),
                "size_kb": round(p.stat().st_size / 1024, 2),
                "mtime": pd.Timestamp.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
                "read_ok": read_ok,
                "n_rows": n_rows,
                "n_cols": len(cols),
                "columns": json.dumps(cols, ensure_ascii=False),
                "phase_like_cols": json.dumps(phase_like_cols, ensure_ascii=False),
                "dt_like_cols": json.dumps(dt_like_cols, ensure_ascii=False),
                "protocol_like_cols": json.dumps(protocol_like_cols, ensure_ascii=False),
                "error": error,
            })

output_df = pd.DataFrame(output_rows)

out_outputs = DATA / "day19A_step5_ablation_output_inventory.csv"
output_df.to_csv(out_outputs, index=False)

print(f"\nWrote: {out_outputs}")
display(
    output_df[
        [
            "notebook_id", "file", "n_rows",
            "phase_like_cols", "dt_like_cols", "protocol_like_cols",
        ]
    ].sort_values(["notebook_id", "file"])
)


# ---------------------------------------------------------------------
# Part C — nb18 paired phase evidence inventory
# ---------------------------------------------------------------------

nb18_files = {
    "v1_charge_first_historical_wrong_phase": DATA / "day14_step5_delta_tQ_curves_v1_wrong_phase.csv",
    "v2_discharge_first_production": DATA / "day14_step5_delta_tQ_curves.csv",
    "v2_discharge_first_alias": DATA / "day14_step5_delta_tQ_curves_v2_aligned_phase.csv",
}

nb18_rows = []

sha_map = {}

for label, path in nb18_files.items():
    exists = path.exists()

    if exists:
        sha = sha256_file(path)
        sha_map[label] = sha
        size = path.stat().st_size
        try:
            df_head = pd.read_csv(path, nrows=5)
            cols = list(df_head.columns)
            n_rows = sum(len(chunk) for chunk in pd.read_csv(path, chunksize=100000))
        except Exception:
            cols = []
            n_rows = np.nan
    else:
        sha = ""
        size = np.nan
        cols = []
        n_rows = np.nan

    if label == "v1_charge_first_historical_wrong_phase":
        actual_phase = PHASE_CHARGE_FIRST
        reclassified_status = "experiment_faithful_charge_first_branch"
        evidence_use = "JES2 §4 candidate: phase-sensitivity evidence before Δt_geom correction"
        historical_label = "wrong_phase"
        alias_of = ""
    elif label == "v2_discharge_first_production":
        actual_phase = PHASE_DISCHARGE_FIRST
        reclassified_status = "historical_nb16_aligned_production_branch"
        evidence_use = "same-waveform ablation closure lineage"
        historical_label = "production / nb16-aligned"
        alias_of = "day14_step5_delta_tQ_curves_v2_aligned_phase.csv if byte-identical"
    else:
        actual_phase = PHASE_DISCHARGE_FIRST
        reclassified_status = "byte_identical_alias_of_v2_production_if_sha_matches"
        evidence_use = "alias only; do not count as independent phase record"
        historical_label = "v2_aligned_phase"
        alias_of = "day14_step5_delta_tQ_curves.csv"

    nb18_rows.append({
        "notebook_id": "nb18",
        "file_label": label,
        "file": path.name,
        "path": str(path.relative_to(REPO)) if exists else str(path),
        "exists": exists,
        "n_rows": n_rows,
        "size_bytes": size,
        "sha256": sha,
        "columns": json.dumps(cols, ensure_ascii=False),
        "historical_label": historical_label,
        "actual_phase_after_day18B_alignment": actual_phase,
        "reclassified_status": reclassified_status,
        "alias_of": alias_of,
        "evidence_use": evidence_use,
    })

nb18_pair_df = pd.DataFrame(nb18_rows)

# Byte identity flag between v2 production and v2 alias
v2_prod_sha = sha_map.get("v2_discharge_first_production")
v2_alias_sha = sha_map.get("v2_discharge_first_alias")
v2_alias_identical = bool(v2_prod_sha and v2_alias_sha and v2_prod_sha == v2_alias_sha)

nb18_pair_df["v2_prod_alias_byte_identical"] = v2_alias_identical

out_nb18 = DATA / "day19A_step5_nb18_phase_pair_inventory.csv"
nb18_pair_df.to_csv(out_nb18, index=False)

print(f"\nWrote: {out_nb18}")
display(
    nb18_pair_df[
        [
            "file_label", "file", "exists", "n_rows",
            "historical_label", "actual_phase_after_day18B_alignment",
            "reclassified_status", "v2_prod_alias_byte_identical",
            "evidence_use",
        ]
    ]
)


# ---------------------------------------------------------------------
# Part D — audit summary table
# ---------------------------------------------------------------------

summary_rows = []

for _, r in scan_df.iterrows():
    nbid = r["notebook_id"]
    outs = output_df[output_df["notebook_id"].eq(nbid)]

    if nbid == "nb18":
        final_verdict = (
            "mixed lineage. v1 is charge-first / experiment-faithful branch; "
            "v2 is discharge-first / nb16-aligned production. The v1-v2 pair "
            "is retained as direct phase-sensitivity evidence."
        )
    elif r["closure_lineage"] == "discharge_first":
        final_verdict = (
            "same-phase discharge-first ablation evidence retained internally; "
            "not directly comparable to charge-first mainline absolute Δt."
        )
    else:
        final_verdict = "manual review required"

    summary_rows.append({
        "notebook_id": nbid,
        "task_label": r["task_label"],
        "closure_lineage": r["closure_lineage"],
        "observed_phase_from_scan": r["observed_phase_from_scan"],
        "comparison_type": r["comparison_type"],
        "evidence_status": r["evidence_status"],
        "n_output_files_found": len(outs),
        "output_files": "; ".join(outs["file"].astype(str).tolist()),
        "cross_notebook_comparability": r["cross_notebook_comparability"],
        "final_verdict": final_verdict,
    })

audit_summary_df = pd.DataFrame(summary_rows)

out_summary = DATA / "day19A_step5_ablation_phase_audit_summary.csv"
audit_summary_df.to_csv(out_summary, index=False)

print(f"\nWrote: {out_summary}")
display(audit_summary_df)

print("\nEvidence-status counts:")
print(audit_summary_df["evidence_status"].value_counts(dropna=False).to_string())

# Hard checks
assert nb18_pair_df["exists"].all(), (
    "Expected all nb18 phase-pair files to exist:\n"
    f"{nb18_pair_df[['file_label', 'file', 'exists']]}"
)

assert v2_alias_identical, (
    "nb18 v2 production and v2_aligned_phase alias are not byte-identical."
)

print("\n" + "=" * 72)
print("Cell 7A PASSED — DCAC-vs-DCAC ablation phase audit complete")
print("=" * 72)

Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step5_ablation_notebook_phase_scan.csv


,notebook_id,task_label,exists,n_phase_relevant_cells,charge_signal_cells,discharge_signal_cells,observed_phase_from_scan,closure_lineage,evidence_status,cross_notebook_comparability
0,nb14,Day11 plating ablation,True,2,0,2,discharge_first,discharge_first,same_phase_ablation_valid_internal_only,cross_notebook_absolute_dt_not_comparable_to_c...
1,nb15,Day12 OCP scan,True,5,0,5,discharge_first,discharge_first,same_phase_ablation_valid_internal_only,cross_notebook_absolute_dt_not_comparable_to_c...
2,nb16,Day13 PE/OCP/transport/kinetics scan,True,4,0,4,discharge_first,discharge_first,same_phase_ablation_valid_internal_only,cross_notebook_absolute_dt_not_comparable_to_c...
3,nb18,Day14 Task1 AsymBV alpha scan,True,5,3,1,mixed,mixed_v1_charge_first_v2_discharge_first,mixed_lineage_phase_pair_evidence,cross_notebook_absolute_dt_invalid_without_out...
4,nb19,Day14 Task2 X6 phase clean test,True,5,0,2,discharge_first,discharge_first,same_phase_ablation_valid_internal_only,cross_notebook_absolute_dt_not_comparable_to_c...



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step5_ablation_output_inventory.csv


,notebook_id,file,n_rows,phase_like_cols,dt_like_cols,protocol_like_cols
2,nb14,results_day11_curve_metrics.csv,24,[],"[""X5A_avg_dt"", ""X5A_A_dt"", ""pl_avg_dt"", ""pl_A_...","[""condition""]"
1,nb14,results_day11_metric_stability_scan.csv,24,[],[],"[""condition""]"
0,nb14,results_day11_plating_dt_Q80.csv,24,[],"[""dt_Q80_plating_min"", ""dt_Q80_X5A_min"", ""dt_Q...","[""condition""]"
5,nb15,day12_pre_audit_Q_at_CC_end.csv,24,[],[],"[""condition""]"
6,nb15,day12_xn_range_probe.csv,5,[],[],"[""condition""]"
3,nb15,results_day12_OCP_ablations.csv,96,[],"[""A_dt"", ""avg_dt""]","[""condition"", ""ablation"", ""Q_CC_min_ablation"",..."
4,nb15,results_day12_step0_A_dt_CC_only.csv,24,[],"[""avg_dt_OLD_Day11"", ""avg_dt_CC_only_NEW"", ""A_...","[""condition""]"
10,nb16,day13_pre_run_Q_CC_end.csv,179,[],[],"[""condition"", ""ablation""]"
11,nb16,day13_strict_Qhi_map.csv,24,[],[],"[""condition"", ""min_ablation_Q_CC_end""]"
12,nb16,day13_xp_range_probe.csv,5,[],[],"[""condition""]"



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step5_nb18_phase_pair_inventory.csv


,file_label,file,exists,n_rows,historical_label,actual_phase_after_day18B_alignment,reclassified_status,v2_prod_alias_byte_identical,evidence_use
0,v1_charge_first_historical_wrong_phase,day14_step5_delta_tQ_curves_v1_wrong_phase.csv,True,5760,wrong_phase,charge_first,experiment_faithful_charge_first_branch,True,JES2 §4 candidate: phase-sensitivity evidence ...
1,v2_discharge_first_production,day14_step5_delta_tQ_curves.csv,True,5760,production / nb16-aligned,discharge_first,historical_nb16_aligned_production_branch,True,same-waveform ablation closure lineage
2,v2_discharge_first_alias,day14_step5_delta_tQ_curves_v2_aligned_phase.csv,True,5760,v2_aligned_phase,discharge_first,byte_identical_alias_of_v2_production_if_sha_m...,True,alias only; do not count as independent phase ...



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step5_ablation_phase_audit_summary.csv


,notebook_id,task_label,closure_lineage,observed_phase_from_scan,comparison_type,evidence_status,n_output_files_found,output_files,cross_notebook_comparability,final_verdict
0,nb14,Day11 plating ablation,discharge_first,discharge_first,DCAC-vs-DCAC same-phase ablation,same_phase_ablation_valid_internal_only,3,results_day11_plating_dt_Q80.csv; results_day1...,cross_notebook_absolute_dt_not_comparable_to_c...,same-phase discharge-first ablation evidence r...
1,nb15,Day12 OCP scan,discharge_first,discharge_first,DCAC-vs-DCAC same-phase ablation,same_phase_ablation_valid_internal_only,4,results_day12_OCP_ablations.csv; results_day12...,cross_notebook_absolute_dt_not_comparable_to_c...,same-phase discharge-first ablation evidence r...
2,nb16,Day13 PE/OCP/transport/kinetics scan,discharge_first,discharge_first,DCAC-vs-DCAC same-phase ablation,same_phase_ablation_valid_internal_only,6,results_day13_PE_transport_kinetics_ablations....,cross_notebook_absolute_dt_not_comparable_to_c...,same-phase discharge-first ablation evidence r...
3,nb18,Day14 Task1 AsymBV alpha scan,mixed_v1_charge_first_v2_discharge_first,mixed,mixed phase lineage with paired v1/v2 evidence,mixed_lineage_phase_pair_evidence,4,day14_step5_delta_tQ_curves_v1_wrong_phase.csv...,cross_notebook_absolute_dt_invalid_without_out...,mixed lineage. v1 is charge-first / experiment...
4,nb19,Day14 Task2 X6 phase clean test,discharge_first,discharge_first,DCAC-vs-DCAC same-phase ablation with t_offset...,same_phase_ablation_valid_internal_only,9,results_day9_x6alpha_4way_dual_Q80.csv; result...,cross_notebook_absolute_dt_not_comparable_to_c...,same-phase discharge-first ablation evidence r...



Evidence-status counts:
evidence_status
same_phase_ablation_valid_internal_only    4
mixed_lineage_phase_pair_evidence          1

Cell 7A PASSED — DCAC-vs-DCAC ablation phase audit complete


In [37]:
# Cell 8A — Day19A final verdict aggregation
#
# Inputs:
#   Step 1: inventory / sign / trajectory / anomaly / residual-null audits
#   Step 2: geometry helper validation
#   Step 3: Day8/Day9 legacy residual audit
#   Step 4: Day16 / nb20 residual audit
#   Step 5: DCAC-vs-DCAC ablation phase audit
#
# Outputs:
#   data/day19A_step6_evidence_register.csv
#   data/day19A_step6_final_verdict_summary.csv
#
# Purpose:
#   Convert Day19A technical audit outputs into a compact evidence register and
#   final verdict table suitable for docs/day19A_retrospective_audit.md,
#   ROADMAP.md, README phase-convention section, and final Day19A commit.

import json
import numpy as np
import pandas as pd
from pathlib import Path


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def read_csv_if_exists(path, **kwargs):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path, **kwargs)
    return None


def vc_str(series):
    if series is None:
        return ""
    vc = series.value_counts(dropna=False)
    return "; ".join([f"{idx}={int(val)}" for idx, val in vc.items()])


def safe_get(df, col, default=np.nan):
    if df is None or col not in df.columns or len(df) == 0:
        return default
    return df[col].iloc[0]


def add_evidence(rows, block, status, key_metrics, finding, evidence_level, files):
    rows.append({
        "block": block,
        "status": status,
        "key_metrics": key_metrics,
        "finding": finding,
        "evidence_level": evidence_level,
        "files": "; ".join(files),
    })


# ---------------------------------------------------------------------
# Load prior outputs
# ---------------------------------------------------------------------

paths = {
    # Step 1
    "inventory": DATA / "day19A_step1_result_curve_inventory.csv",
    "sign_audit": DATA / "day19A_step1_csv_timepair_sign_self_audit.csv",
    "legacy_json_validation": DATA / "day19A_step1_legacy_json_trajectory_validation.csv",
    "day18B_resid_magnitude": DATA / "day19A_step1_day18B_smoke_identity_resid_magnitude_audit.csv",
    "t_axis_anomaly": DATA / "day19A_step1_t_axis_anomaly_diagnostic.csv",

    # Step 2
    "mj1_anchor_summary": DATA / "day19A_step2_MJ1_0p3_0p7_10tau_geom_anchor_summary.csv",
    "geom_helper_validation": DATA / "day19A_step2_geometry_helper_validation_summary.csv",
    "trajectory_helper_selftest": DATA / "day19A_step2_trajectory_helper_dedup_selftest.csv",

    # Step 3
    "legacy_dt_resid_summary": DATA / "day19A_step3_legacy_dt_resid_summary.csv",
    "legacy_taxonomy": DATA / "day19A_step3_legacy_residual_taxonomy.csv",
    "legacy_lineage": DATA / "day19A_step3_legacy_lineage_verdict_summary.csv",

    # Step 4
    "day16_window": DATA / "day19A_step4_day16_window_audit_refined.csv",
    "day16_resid_summary": DATA / "day19A_step4_day16_dt_resid_summary.csv",

    # Step 5
    "ablation_phase_summary": DATA / "day19A_step5_ablation_phase_audit_summary.csv",
    "nb18_phase_pair": DATA / "day19A_step5_nb18_phase_pair_inventory.csv",
}

data = {k: read_csv_if_exists(v) for k, v in paths.items()}

missing = [k for k, df in data.items() if df is None]
if missing:
    print("[WARN] Missing optional/expected files:")
    for k in missing:
        print(f"  - {k}: {paths[k]}")
else:
    print("All expected Day19A output files found.")


# ---------------------------------------------------------------------
# Evidence register
# ---------------------------------------------------------------------

evidence_rows = []

# Step 1 — inventory
inv = data["inventory"]
if inv is not None:
    add_evidence(
        evidence_rows,
        block="19A.1 result/curve inventory",
        status="ok",
        key_metrics=(
            f"n_files={len(inv)}; "
            f"families={vc_str(inv['file_family'])}; "
            f"audit_readiness={vc_str(inv['audit_readiness'])}"
        ),
        finding=(
            "Historical outputs were classified by file family. Day8/Day9 CSVs "
            "are summary-level but paired JSON payloads are available for raw-trajectory audit."
        ),
        evidence_level="inventory_support",
        files=[str(paths["inventory"].relative_to(REPO))],
    )

# Step 1 — CSV sign audit
sign_df = data["sign_audit"]
if sign_df is not None:
    n_ok = int((sign_df["audit_status"] == "ok").sum())
    n_total = len(sign_df)
    inferred = vc_str(sign_df["inferred_sign"])
    add_evidence(
        evidence_rows,
        block="19A.1 CSV sign self-audit",
        status="ok" if n_ok == n_total else "review",
        key_metrics=f"n_ok={n_ok}/{n_total}; inferred_sign={inferred}",
        finding=(
            "All time-pair-auditable CSV files store Δt with the official sign "
            "convention t_ref − t_protocol. The Day18B discrepancy is not caused "
            "by an on-disk sign inversion."
        ),
        evidence_level="high_for_excluding_sign_bug",
        files=[str(paths["sign_audit"].relative_to(REPO))],
    )

# Step 1 — legacy JSON trajectory validation
val = data["legacy_json_validation"]
if val is not None:
    n_usable = int((val["audit_use"] == "usable_recompute_strict_net_from_t_I").sum())
    n_total = len(val)
    q_mode_counts = vc_str(val.loc[val["audit_use"] == "usable_recompute_strict_net_from_t_I", "q_match_mode"])
    add_evidence(
        evidence_rows,
        block="19A.1 legacy JSON strict-net validation",
        status="ok",
        key_metrics=(
            f"usable_records={n_usable}/{n_total}; "
            f"q_match_mode={q_mode_counts}"
        ),
        finding=(
            "Legacy Day8/Day9 JSON payloads preserve t_chg, I_chg, and Q_net_trajectory. "
            "For all usable records, stored Q_net_trajectory matches strict-net integration "
            "Q=-∫I dt."
        ),
        evidence_level="high_raw_trajectory_ground_truth",
        files=[str(paths["legacy_json_validation"].relative_to(REPO))],
    )

# Step 1 — Day18B residual magnitude null baseline
resid0 = data["day18B_resid_magnitude"]
if resid0 is not None and len(resid0):
    r = resid0.iloc[0]
    add_evidence(
        evidence_rows,
        block="19A.1 Day18B residual magnitude null baseline",
        status=str(r.get("residual_magnitude_status", "unknown")),
        key_metrics=(
            f"max_abs_identity_err_s={r['max_abs_identity_err_s']:.3e}; "
            f"dt_resid_max_abs_s={r['dt_resid_max_abs_s']:.6f}; "
            f"dt_resid_p95_abs_s={r['dt_resid_p95_abs_s']:.6f}; "
            f"resid_over_geom_median={r['resid_over_geom_median']:.6g}"
        ),
        finding=(
            "Day18B prescribed-current CC residual is a near-zero null baseline. "
            "Raw Δt is carried by current geometry, not by non-geometric model response."
        ),
        evidence_level="high_for_geometry_null_baseline",
        files=[str(paths["day18B_resid_magnitude"].relative_to(REPO))],
    )

# Step 1 — time-axis anomalies
anom = data["t_axis_anomaly"]
if anom is not None:
    add_evidence(
        evidence_rows,
        block="19A.1 legacy time-axis anomaly audit",
        status="ok",
        key_metrics=(
            f"n_anomalies={len(anom)}; "
            f"classes={vc_str(anom['anomaly_class'])}; "
            f"recoverable={int(anom['recoverable_via_dedup'].sum())}/{len(anom)}"
        ),
        finding=(
            "All known legacy time-axis anomalies are duplicate-timestamp-only cases "
            "and are repairable by keep-last de-duplication. No successful legacy record "
            "needs to be excluded on time-axis grounds."
        ),
        evidence_level="high_for_trajectory_cleaning",
        files=[str(paths["t_axis_anomaly"].relative_to(REPO))],
    )

# Step 2 — MJ1 anchor
mj1 = data["mj1_anchor_summary"]
if mj1 is not None:
    cf_raw = mj1[(mj1["phase"] == "charge_first") & (mj1["fp_mode"] == "raw")]
    df_raw = mj1[(mj1["phase"] == "discharge_first") & (mj1["fp_mode"] == "raw")]
    mean_cf = float(cf_raw["mean_dt_geom_s"].iloc[0]) if len(cf_raw) else np.nan
    mean_df = float(df_raw["mean_dt_geom_s"].iloc[0]) if len(df_raw) else np.nan
    raw_cummax_max = float(mj1["raw_vs_cummax_max_per_Q_diff_s"].max())
    add_evidence(
        evidence_rows,
        block="19A.2 MJ1 waveform-geometry anchor",
        status="ok",
        key_metrics=(
            f"charge_first_raw_mean={mean_cf:.6f}s; "
            f"discharge_first_raw_mean={mean_df:.6f}s; "
            f"raw_cummax_max_diff={raw_cummax_max:.6f}s"
        ),
        finding=(
            "The waveform-only helper reproduces the MJ1 charge-first geometry anchor. "
            "Discharge-first remains negative but is not the exact negative mirror, "
            "confirming phase sensitivity without imposing false symmetry."
        ),
        evidence_level="high_helper_anchor",
        files=[str(paths["mj1_anchor_summary"].relative_to(REPO))],
    )

# Step 2 — geometry helper historical validation
geom_val = data["geom_helper_validation"]
if geom_val is not None and len(geom_val):
    global_max = float(geom_val["global_max_abs_err_s"].iloc[0])
    global_median = float(geom_val["global_median_abs_err_s"].iloc[0])
    global_status = str(geom_val["global_status"].iloc[0])
    add_evidence(
        evidence_rows,
        block="19A.2 geometry helper validation against Day18B/v4",
        status=global_status,
        key_metrics=(
            f"n_rows={int(geom_val['n_rows'].iloc[0])}; "
            f"n_groups={int(geom_val['n_groups'].iloc[0])}; "
            f"global_max_abs_err_s={global_max:.6f}; "
            f"global_median_abs_err_s={global_median:.6f}"
        ),
        finding=(
            "Using actual rebased frequency metadata, the Cell 4A helper reproduces "
            "stored Day18B/Day18 v4 geometry columns. 4D v1 failure was a frequency-basis "
            "mismatch, not a helper failure."
        ),
        evidence_level="high_for_helper_equivalence",
        files=[str(paths["geom_helper_validation"].relative_to(REPO))],
    )

# Step 2 — trajectory helper selftest
selftest = data["trajectory_helper_selftest"]
if selftest is not None:
    add_evidence(
        evidence_rows,
        block="19A.2 trajectory helper de-dup self-test",
        status="ok",
        key_metrics=(
            f"n_records={len(selftest)}; "
            f"removed_samples={selftest['n_removed'].sum()}; "
            f"max_q_mismatch_mAh={selftest['q_match_max_abs_mAh'].max():.6g}"
        ),
        finding=(
            "Trajectory helper repairs all known duplicate timestamp cases while preserving "
            "strict-net Q consistency."
        ),
        evidence_level="high_helper_selftest",
        files=[str(paths["trajectory_helper_selftest"].relative_to(REPO))],
    )

# Step 3 — legacy residual / lineage
legacy_tax = data["legacy_taxonomy"]
legacy_lineage = data["legacy_lineage"]
if legacy_tax is not None:
    add_evidence(
        evidence_rows,
        block="19A.3 Day8/Day9 residual topology",
        status="ok",
        key_metrics=(
            f"n_pairs={len(legacy_tax)}; "
            f"stable_verdicts={vc_str(legacy_tax['stable_region_verdict'])}; "
            f"taxonomy={vc_str(legacy_tax['taxonomy'])}"
        ),
        finding=(
            "Day8/Day9 legacy DC-vs-DCAC pairs are all discharge-first. Most pairs are "
            "stable-null or stable-weak; max-based inspect labels are dominated by isolated "
            "spikes and boundary artifacts."
        ),
        evidence_level="high_raw_trajectory_retro_audit",
        files=[str(paths["legacy_taxonomy"].relative_to(REPO))],
    )

if legacy_lineage is not None:
    add_evidence(
        evidence_rows,
        block="19A.3 Day8/Day9 lineage verdict",
        status="ok",
        key_metrics=(
            f"n_lineages={len(legacy_lineage)}; "
            f"lineage_verdicts={vc_str(legacy_lineage['lineage_verdict'])}; "
            f"evidence_levels={vc_str(legacy_lineage['evidence_level'])}"
        ),
        finding=(
            "day8_x2 and day9_x5beta support geometry-dominated reclassification. "
            "X6alpha/X6beta distributed candidates occur in exploratory combined branches "
            "and are not clean mechanism evidence."
        ),
        evidence_level="high_for_legacy_reclassification",
        files=[str(paths["legacy_lineage"].relative_to(REPO))],
    )

# Step 4 — Day16 / nb20
day16_win = data["day16_window"]
day16_resid = data["day16_resid_summary"]

if day16_win is not None:
    add_evidence(
        evidence_rows,
        block="19A.4 Day16/nb20 Q-window gate",
        status="ok",
        key_metrics=(
            f"n_groups={len(day16_win)}; "
            f"refined_status={vc_str(day16_win['window_compatibility_refined'])}; "
            f"capacity_warnings={int(day16_win['capacity_scale_warning'].sum())}"
        ),
        finding=(
            "All Day16 groups pass the Q-window gate. Eleven groups are included with "
            "capacity-scale warning but not excluded."
        ),
        evidence_level="supporting_window_gate",
        files=[str(paths["day16_window"].relative_to(REPO))],
    )

if day16_resid is not None:
    add_evidence(
        evidence_rows,
        block="19A.4 Day16/nb20 stored-first-passage residual audit",
        status="ok",
        key_metrics=(
            f"n_groups={len(day16_resid)}; "
            f"stable_verdicts={vc_str(day16_resid['stable_region_verdict'])}; "
            f"taxonomy={vc_str(day16_resid['taxonomy'])}"
        ),
        finding=(
            "Day16 stored-first-passage audit is broadly consistent with Day8/Day9: "
            "most groups are stable-null, stable-weak, intermediate/spiky, or boundary-like; "
            "only a few retain stable-distributed-nonzero labels. Evidence level is lower "
            "because raw trajectories are unavailable."
        ),
        evidence_level="supporting_stored_first_passage_audit",
        files=[str(paths["day16_resid_summary"].relative_to(REPO))],
    )

# Step 5 — ablation phase audit
abl = data["ablation_phase_summary"]
nb18 = data["nb18_phase_pair"]

if abl is not None:
    add_evidence(
        evidence_rows,
        block="19A.5 DCAC-vs-DCAC ablation phase audit",
        status="ok",
        key_metrics=(
            f"n_notebooks={len(abl)}; "
            f"evidence_status={vc_str(abl['evidence_status'])}"
        ),
        finding=(
            "nb14/15/16/19 retain internal validity as same-phase discharge-first ablation "
            "evidence, but absolute Δt is not comparable to the charge-first mainline. "
            "nb18 is mixed-lineage phase-pair evidence."
        ),
        evidence_level="high_for_ablation_evidence_reclassification",
        files=[str(paths["ablation_phase_summary"].relative_to(REPO))],
    )

if nb18 is not None:
    v2_alias = bool(nb18["v2_prod_alias_byte_identical"].all()) if "v2_prod_alias_byte_identical" in nb18 else False
    add_evidence(
        evidence_rows,
        block="19A.5 nb18 paired phase evidence",
        status="ok" if v2_alias else "review",
        key_metrics=(
            f"n_files={len(nb18)}; "
            f"v2_alias_byte_identical={v2_alias}; "
            f"phases={vc_str(nb18['actual_phase_after_day18B_alignment'])}"
        ),
        finding=(
            "nb18 v1 historically labelled wrong_phase is reclassified as charge-first / "
            "experiment-faithful. nb18 v2 is discharge-first / nb16-aligned production, "
            "with byte-identical aligned alias. The v1/v2 pair is JES2 §4 phase-sensitivity evidence."
        ),
        evidence_level="high_for_phase_sensitivity_evidence",
        files=[str(paths["nb18_phase_pair"].relative_to(REPO))],
    )

evidence_register = pd.DataFrame(evidence_rows)

out_register = DATA / "day19A_step6_evidence_register.csv"
evidence_register.to_csv(out_register, index=False)

print(f"Wrote: {out_register} ({len(evidence_register)} rows)")
display(evidence_register)


# ---------------------------------------------------------------------
# Final verdict summary
# ---------------------------------------------------------------------

verdict_rows = [
    {
        "verdict_id": "V1",
        "claim": "Historical nb04–nb20 DCAC implementation was discharge-first relative to the MJ1 charge-first ARB intent.",
        "status": "supported",
        "basis": (
            "Day8/Day9 198/198 DC-vs-DCAC pairs inferred discharge-first; "
            "nb14/15/16/19 ablation notebooks discharge-first; nb18 mixed lineage "
            "with final v2 discharge-first production."
        ),
        "scope": "Historical PyBaMM implementation lineage; not a claim about experimental MJ1 data.",
        "caveat": "Use implementation–intent phase mismatch framing, not 'alternative designed branch'.",
        "action_for_docs": "Primary Day19A finding.",
    },
    {
        "verdict_id": "V2",
        "claim": "CSV stored Δt sign inversion is not the explanation for Day18B discrepancy.",
        "status": "supported",
        "basis": "7/7 time-pair-auditable CSV files use Δt=t_ref−t_protocol.",
        "scope": "Stored curve CSVs audited in Day19A.",
        "caveat": "Sign correctness does not imply phase correctness.",
        "action_for_docs": "Secondary finding; downstream of phase mismatch.",
    },
    {
        "verdict_id": "V3",
        "claim": "Raw Δt(Q) before geometry correction is phase-coupled and not a non-geometric state-layer observable by itself.",
        "status": "supported",
        "basis": (
            "MJ1 anchor reproduces charge-first positive geometry; Day18B smoke shows "
            "dt_model≈dt_geom and dt_resid near zero; nb18 v1/v2 preserved paired phase evidence."
        ),
        "scope": "Prescribed-current CC / first-passage geometry framework.",
        "caveat": "Full-protocol voltage-boundary/CV contributions require separate Day20+ segmented audit.",
        "action_for_docs": "JES2 §4 candidate methodological claim.",
    },
    {
        "verdict_id": "V4",
        "claim": "Day18B prescribed-current CC simulations establish a near-zero non-geometric residual baseline.",
        "status": "supported",
        "basis": "max|dt_resid|=0.044075 s, p95=0.009089 s, median=0.001425 s across 1032 points.",
        "scope": "Day18B prescribed-current CC smoke cases, 3 parameter sets, 18 pairs.",
        "caveat": "This does not negate full-protocol MJ1 measured gain; it only isolates CC prescribed-current residual.",
        "action_for_docs": "Use as null-baseline evidence.",
    },
    {
        "verdict_id": "V5",
        "claim": "Day8/Day9 legacy DC-vs-DCAC branch is mostly geometry-dominated or weak after topology audit.",
        "status": "supported_with_exceptions",
        "basis": (
            "198 discharge-first pairs decomposed; 155 stable-null or stable-weak; "
            "19 stable-distributed-nonzero; X6 distributed candidates concentrated in exploratory branches."
        ),
        "scope": "Legacy Day8/Day9 JSON raw-trajectory retro audit.",
        "caveat": "19 distributed candidates require lineage-specific review and are not clean mechanism evidence by default.",
        "action_for_docs": "Use for reclassification, not mechanism proof.",
    },
    {
        "verdict_id": "V6",
        "claim": "Day16/nb20 supports the same reclassification pattern at lower evidential level.",
        "status": "supporting",
        "basis": (
            "36 groups decomposed from stored first-passage curves; 24 stable-null/weak, "
            "9 intermediate/spiky, 3 stable-distributed-nonzero."
        ),
        "scope": "Day16 stored-first-passage audit; no raw current trajectories.",
        "caveat": "Evidence level lower than Day8/Day9; 11 groups carry capacity-scale warning.",
        "action_for_docs": "Use as supporting audit, not primary proof.",
    },
    {
        "verdict_id": "V7",
        "claim": "DCAC-vs-DCAC ablation outputs retain internal validity only within same-phase lineages.",
        "status": "supported",
        "basis": "nb14/15/16/19 same-phase discharge-first; nb18 mixed v1/v2 phase-pair evidence.",
        "scope": "Ablation notebooks, not DC-vs-DCAC absolute mainline.",
        "caveat": "Cross-notebook absolute Δt comparison invalid without matching phase/output lineage.",
        "action_for_docs": "Use for evidence-level classification.",
    },
    {
        "verdict_id": "V8",
        "claim": "Day19A reclassifies historical simulations rather than discarding them.",
        "status": "supported",
        "basis": "Legacy outputs remain usable as phase-control, geometry-corrected residual, and same-waveform ablation evidence.",
        "scope": "Repository audit trail and JES2 methodological framing.",
        "caveat": "Do not frame discharge-first as intentionally designed alternative branch unless supported by prior design documents.",
        "action_for_docs": "Commit message / README / ROADMAP wording.",
    },
]

final_verdict = pd.DataFrame(verdict_rows)

out_verdict = DATA / "day19A_step6_final_verdict_summary.csv"
final_verdict.to_csv(out_verdict, index=False)

print(f"\nWrote: {out_verdict} ({len(final_verdict)} rows)")
display(final_verdict)

print("\n" + "=" * 72)
print("Cell 8A PASSED — Day19A final verdict aggregation complete")
print("=" * 72)

All expected Day19A output files found.
Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step6_evidence_register.csv (14 rows)


,block,status,key_metrics,finding,evidence_level,files
0,19A.1 result/curve inventory,ok,n_files=45; families=legacy_day8_9_10_result_s...,Historical outputs were classified by file fam...,inventory_support,data/day19A_step1_result_curve_inventory.csv
1,19A.1 CSV sign self-audit,ok,n_ok=7/7; inferred_sign=official_tRef_minus_tP...,All time-pair-auditable CSV files store Δt wit...,high_for_excluding_sign_bug,data/day19A_step1_csv_timepair_sign_self_audit...
2,19A.1 legacy JSON strict-net validation,ok,usable_records=247/279; q_match_mode=strict_ne...,"Legacy Day8/Day9 JSON payloads preserve t_chg,...",high_raw_trajectory_ground_truth,data/day19A_step1_legacy_json_trajectory_valid...
3,19A.1 Day18B residual magnitude null baseline,near_zero_null_baseline,max_abs_identity_err_s=2.274e-13; dt_resid_max...,Day18B prescribed-current CC residual is a nea...,high_for_geometry_null_baseline,data/day19A_step1_day18B_smoke_identity_resid_...
4,19A.1 legacy time-axis anomaly audit,ok,n_anomalies=5; classes=duplicate_timestamps_on...,All known legacy time-axis anomalies are dupli...,high_for_trajectory_cleaning,data/day19A_step1_t_axis_anomaly_diagnostic.csv
5,19A.2 MJ1 waveform-geometry anchor,ok,charge_first_raw_mean=337.503105s; discharge_f...,The waveform-only helper reproduces the MJ1 ch...,high_helper_anchor,data/day19A_step2_MJ1_0p3_0p7_10tau_geom_ancho...
6,19A.2 geometry helper validation against Day18...,ok,n_rows=1272; n_groups=21; global_max_abs_err_s...,"Using actual rebased frequency metadata, the C...",high_for_helper_equivalence,data/day19A_step2_geometry_helper_validation_s...
7,19A.2 trajectory helper de-dup self-test,ok,n_records=5; removed_samples=5; max_q_mismatch...,Trajectory helper repairs all known duplicate ...,high_helper_selftest,data/day19A_step2_trajectory_helper_dedup_self...
8,19A.3 Day8/Day9 residual topology,ok,n_pairs=198; stable_verdicts=stable_weak=102; ...,Day8/Day9 legacy DC-vs-DCAC pairs are all disc...,high_raw_trajectory_retro_audit,data/day19A_step3_legacy_residual_taxonomy.csv
9,19A.3 Day8/Day9 lineage verdict,ok,n_lineages=7; lineage_verdicts=mixed_with_dist...,day8_x2 and day9_x5beta support geometry-domin...,high_for_legacy_reclassification,data/day19A_step3_legacy_lineage_verdict_summa...



Wrote: /Users/louislu/pybamm-dcac-superimposed/data/day19A_step6_final_verdict_summary.csv (8 rows)


,verdict_id,claim,status,basis,scope,caveat,action_for_docs
0,V1,Historical nb04–nb20 DCAC implementation was d...,supported,Day8/Day9 198/198 DC-vs-DCAC pairs inferred di...,Historical PyBaMM implementation lineage; not ...,Use implementation–intent phase mismatch frami...,Primary Day19A finding.
1,V2,CSV stored Δt sign inversion is not the explan...,supported,7/7 time-pair-auditable CSV files use Δt=t_ref...,Stored curve CSVs audited in Day19A.,Sign correctness does not imply phase correctn...,Secondary finding; downstream of phase mismatch.
2,V3,Raw Δt(Q) before geometry correction is phase-...,supported,MJ1 anchor reproduces charge-first positive ge...,Prescribed-current CC / first-passage geometry...,Full-protocol voltage-boundary/CV contribution...,JES2 §4 candidate methodological claim.
3,V4,Day18B prescribed-current CC simulations estab...,supported,"max|dt_resid|=0.044075 s, p95=0.009089 s, medi...","Day18B prescribed-current CC smoke cases, 3 pa...",This does not negate full-protocol MJ1 measure...,Use as null-baseline evidence.
4,V5,Day8/Day9 legacy DC-vs-DCAC branch is mostly g...,supported_with_exceptions,198 discharge-first pairs decomposed; 155 stab...,Legacy Day8/Day9 JSON raw-trajectory retro audit.,19 distributed candidates require lineage-spec...,"Use for reclassification, not mechanism proof."
5,V6,Day16/nb20 supports the same reclassification ...,supporting,36 groups decomposed from stored first-passage...,Day16 stored-first-passage audit; no raw curre...,Evidence level lower than Day8/Day9; 11 groups...,"Use as supporting audit, not primary proof."
6,V7,DCAC-vs-DCAC ablation outputs retain internal ...,supported,nb14/15/16/19 same-phase discharge-first; nb18...,"Ablation notebooks, not DC-vs-DCAC absolute ma...",Cross-notebook absolute Δt comparison invalid ...,Use for evidence-level classification.
7,V8,Day19A reclassifies historical simulations rat...,supported,"Legacy outputs remain usable as phase-control,...",Repository audit trail and JES2 methodological...,Do not frame discharge-first as intentionally ...,Commit message / README / ROADMAP wording.



Cell 8A PASSED — Day19A final verdict aggregation complete
